# LightGlue + ALIKED Matching Pipeline — DIMER E2E matching fine-tuning tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/lightglue-matching-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/lightglue-matching-pipeline/blob/main/tutorials/lightglue_matching_colab.ipynb) [![Upstream](https://img.shields.io/badge/Upstream-cvg%2FLightGlue-181717?style=flat&logo=github&logoColor=white)](https://github.com/cvg/LightGlue) [![Extractor](https://img.shields.io/badge/Extractor-Shiaoming%2FALIKED-181717?style=flat&logo=github&logoColor=white)](https://github.com/Shiaoming/ALIKED) [![arXiv](https://img.shields.io/badge/arXiv-2306.13643-b31b1b.svg)](https://arxiv.org/abs/2306.13643)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** sparse local-feature matching (ALIKED keypoints and descriptors matched by the LightGlue transformer, one confidence per correspondence), homography-supervised evaluation and bounded supervised fine-tuning of the matcher's last layers and assignment head on a labelled pair dataset, using the pinned release-tag checkpoints and a vendored network

**This notebook is standalone.** It carries the repository's package (7 modules under `src/lightglue_pipeline/`, at revision `88335608c75d`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the LightGlue GitHub release (`cvg/LightGlue`, the matcher) and the ALIKED repository at a pinned commit (`Shiaoming/ALIKED`, the extractor) at the immutable release tag `v0.1_arxiv` (~50 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, fetches the two pinned checkpoint pickles (the 48 MB `aliked_lightglue.pth` from the immutable `v0.1_arxiv` release and the 3 MB `aliked-n16.pth` from a pinned ALIKED commit), digest-verifies them, audits their pickle streams statically against the fleet's four allowed globals, converts each once with torch's weights-only unpickler into a safetensors file that is itself digest-verified, and loads those strictly into the carried network; fetches the 360 pinned iNaturalist photographs from the project's open-data bucket (about 39 MB, each refused on any byte-size or SHA-256 mismatch), cuts them per species into 216 / 48 / 96 training, validation and test photographs and turns each into a homography pair with an exact reference (two difficulty tiers, the hard one rotated up to ±150°); matches a drawn-shape pair through the inference contract with an input manifest and a rejection probe; measures the frozen matcher's precision at 3 px, inliers per pair and homography accuracy over the 96 test pairs beside the identity-guess, patch-nearest-neighbour and descriptor-nearest-neighbour baselines; runs a bounded fine-tuning of the matcher's last two layers and assignment head with the LightGlue assignment loss on cached ALIKED features and validation-precision epoch selection; scores the held-out pairs again per tier; re-matches the drawn pair with the adapted model; exports the adapter as safetensors with a manifest; and reloads that artifact into a fresh pipeline to verify match parity. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5). On CPU the whole path took about @P:CPU_TOTAL_MIN@ minutes on the build workstation after the downloads (expect longer on a 2-vCPU hosted runtime); a CUDA runtime is used automatically when present and finishes in minutes.

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4 and re-run from that cell to upload one zip of JPEG / PNG photographs — at least eight, any subject, ideally textured — which are split by image, turned into homography pairs with the same seeded warps, and passed through the same validation, baselines, fine-tuning, held-out evaluation, artifact export and reload-parity cells as the iNaturalist sample. The expected layout and the ceilings are stated in the Prerequisites and in Section 4, and uploaded files stay inside this runtime. BYOD is optional and never part of the default path. Real pairs of two different photographs of one scene need a reference homography or depth to be scored; the contract accepts only synthesised pairs with an exact `H`.

LightGlue (Lindenberger, Sarlin and Pollefeys, ICCV 2023) is a sparse matcher: given two sets of keypoints with descriptors it runs nine transformer layers of self- and cross-attention with rotary positional encoding, then a unified assignment head (matchability per keypoint and a dual-softmax over descriptor similarities) that returns, for each mutual best match above a threshold, one confidence. The keypoints and descriptors come from ALIKED-N(16) (Zhao et al., 2023), a deformable-convolution detector-descriptor. The checkpoints are two GitHub files, not a Hugging Face repository: `aliked_lightglue.pth` is an asset of the immutable `v0.1_arxiv` release of `cvg/LightGlue` (**Apache-2.0**; 11,884,625 parameters, 253 tensors) and `aliked-n16.pth` a file of `Shiaoming/ALIKED` at a pinned commit (**BSD-3-Clause**; 677,356 parameters). Both are plain PyTorch pickles: this notebook audits them statically, converts each once with the weights-only unpickler to a safetensors file and loads only those. The matcher's adaptive early exit and point pruning are switched off, so every pair runs all nine layers on every keypoint. The confidences are assignment scores, **not calibrated probabilities** that a match is right, and the matcher never abstains — any two images produce whatever passes the threshold, overlapping or not. This repository carries both networks as plain PyTorch (`modeling.py`, vendored from the upstream commits) so no third-party matching framework is installed.

What this notebook adds to inference is **adaptation with labelled pairs** — pairs whose correct answer is known exactly. The images are real: 360 CC0-licensed, research-grade iNaturalist photographs of six bird species (**CC0 1.0**; the fleet's SigLIP sample, 60 per species, one per observer), pinned by photo id, byte size and SHA-256 and fetched from the project's open-data bucket at run time. Each photograph becomes one pair with a seeded homography warp and seeded photometric changes, so every returned match has a **reprojection error** against the reference `H`. Two tiers alternate: `easy` (small perspective, ±10°, mild photometry) and `hard` (large perspective, **rotation up to ±150°**, scale 0.6–1.4, strong photometry). The rotation is the point: ALIKED descriptors and LightGlue's positional encoding are not rotation-invariant, and the build record measured the frozen matcher at @P:FROZEN_P3@ precision at 3 px over the 96 test pairs — @P:FROZEN_EASY_P3@ on the easy tier, **@P:FROZEN_HARD_P3@ on the rotated tier** — so the honest question is whether a bounded fine-tuning of the matcher's last layers on 216 pairs (half of them rotated) moves **precision at 3 px**, the **inlier count** and the **homography accuracy** on an image-disjoint test split, per tier, against three **baselines** (the **identity guess**, a **patch nearest neighbour**, and the same ALIKED keypoints matched by **descriptor nearest neighbour** without the learned matcher). Nothing here is a quality claim about your images: it is one seeded split of one sample under synthetic warps.

**Snapshot note:** the manifest in Section 3 lists the two source pickles with their sizes, digests, source URLs and static-audit digests, and the safetensors file each converts to with its own size and digest. Section 3 stages the sources, verifies them, converts them once and verifies the conversions before the networks are constructed; a re-run that finds the safetensors files present never opens a pickle again.

**Learning objectives:** install the pinned runtime; read what the carried package guarantees; stage, digest-verify, audit and convert two immutable upstream checkpoint pickles into a vendored network; build a labelled pair set with exact references from digest-pinned photographs, validate it and split it by photograph without leakage; match a drawn pair through the public API and read match confidences correctly (thresholded, uncalibrated, no abstention); measure the frozen matcher's precision, inlier count and homography accuracy beside three baselines and read the per-tier breakdown; run a bounded fine-tuning with the LightGlue assignment loss, explicit hyperparameters and validation-based epoch selection; evaluate on an image-disjoint test split; re-match a drawing from a different image family with the adapted model; and export a safetensors adapter that reloads against the pinned base with verified parity.

**This notebook does not demonstrate:** object detection, semantic segmentation, OCR, caption generation, calibrated match probabilities or universal thresholds, geometric verification inside the pipeline (the evaluation's RANSAC is a metric, not a filter), pose or depth estimation, dense or detector-free matching, other extractors (SuperPoint, DISK, SIFT — LightGlue has checkpoints for each; only the ALIKED one is carried), the adaptive early exit and point pruning of the upstream demo, fine-tuning of the extractor, the input projection, the positional encoding or the earlier matcher layers, training on images that are not the pinned sample or your own uploads, evaluation on HPatches, MegaDepth or any benchmark proper (only one seeded 360-pair sample is scored here), and any claim that six bird species stand in for your scenes. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU (float32) and uses CUDA automatically when available. CPU is workable: sparse matching at 640 px costs about a second per pair on the build workstation, the three baselines a few minutes, and the fine-tuning about @P:CPU_ADAPT_MIN@ minutes for the default recipe; a T4 finishes the whole path in minutes.
- **Knowledge:** basic Python and PIL; what a homography is and why a warped copy of an image has an exact correspondence for every pixel; what precision, reprojection error and RANSAC measure and why none is a human judgement; why 96 pairs from one seeded draw give no dispersion estimate; why a self-drawn scene is a plumbing check while an image-disjoint split of one sample is a measurement of that sample only; why a matcher that never abstains needs its baselines read first.
- **Data contract:** records are `{id, image0, image1, homography}` — two PIL images (or files decodable by Pillow) with sides in [64, 1024] px (the sample is built at 640 px on the long side, sides multiples of 8) and a finite, non-singular 3 × 3 `H` mapping image0 coordinates to image1 coordinates; the sample records also carry `tier` and `seed`. Between 4 and 5,000 records per split; BYOD is one zip or directory of JPEG / PNG photographs (at least eight) which the contract turns into pairs itself.
- **Validation is structural, not semantic:** every image is opened and decoded and every `H` checked for shape and rank, but nothing checks that `image1` really is `image0` under `H` — a wrong reference is scored without complaint and the numbers are then meaningless. The tutorial's references are exact by construction.
- **Privacy:** Do not upload confidential or restricted data to a hosted runtime unless you are authorized to process it there. The default path uploads nothing.
- **External access (data):** besides the two checkpoint files, the default path fetches 360 JPEG/PNG files from `https://inaturalist-open-data.s3.amazonaws.com/photos/<id>/medium.<ext>` (about 39 MB in total), each pinned by byte size and SHA-256 in the carried `samples.py`; a cached copy under `weights/inat-birds/` is re-hashed and used when present.
- **External access:** the LightGlue GitHub release (`cvg/LightGlue`, the matcher) and the ALIKED repository at a pinned commit (`Shiaoming/ALIKED`, the extractor) only, to fetch the pinned `cvg/LightGlue` snapshot (~50 MB in total) at revision `v0.1_arxiv…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same pins as the repository's pyproject.toml at the generating revision; any `--index-url`/`--find-links` lines are passed to pip as written) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `numpy` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'numpy==2.5.3',
    'pillow==11.3.0',
    'safetensors==0.8.0',
    'torch==2.14.0',
    'torchvision==0.29.0',
]
NOTEBOOK_SOURCE = {
    'repository': 'lightglue-matching-pipeline',
    'repository_revision': '88335608c75da424b30d52e2f314cd02822db935',
    'embedded_module': 'src/lightglue_pipeline/config.py',
    'embedded_modules': ['src/lightglue_pipeline/config.py', 'src/lightglue_pipeline/metrics.py', 'src/lightglue_pipeline/modeling.py', 'src/lightglue_pipeline/model.py', 'src/lightglue_pipeline/provenance.py', 'src/lightglue_pipeline/samples.py', 'src/lightglue_pipeline/pipeline.py'],
    'module_sha256': 'c51d9a7e0461d61e092913a6763ab58208bc3f423a8808f47f0be093503c2760',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, numpy
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'numpy': numpy.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/lightglue_pipeline/` @ `88335608c75d`)

The next 7 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (2 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/7:** `src/lightglue_pipeline/config.py`

In [ ]:
from __future__ import annotations

# LightGlue (Lindenberger, Sarlin and Pollefeys, ICCV 2023) with ALIKED local features. LightGlue
# publishes no Hugging Face repository: the matcher checkpoint is an asset of the immutable GitHub
# release tag below, and the ALIKED extractor checkpoint is a file of the ALIKED repository at a
# pinned commit. Both are plain PyTorch state-dict pickles, audited statically and converted once to
# safetensors (the only files the network is ever loaded from).
MODEL_ID = "cvg/LightGlue"
# GitHub release tag (2023-06-26); its assets are immutable.
MODEL_REVISION = "v0.1_arxiv"
MODEL_REVISION_KIND = "github-release-tag"
# LightGlue code and weights.
MODEL_LICENSE = "Apache-2.0"
EXTRACTOR_LICENSE = "BSD-3-Clause"  # ALIKED code (the port carried in modeling.py) and weights

# --- the matcher: LightGlue trained on ALIKED features -------------------------------------------
MATCHER_SOURCE_FILENAME = "aliked_lightglue.pth"
MATCHER_SOURCE_URL = (
    f"https://github.com/{MODEL_ID}/releases/download/{MODEL_REVISION}/{MATCHER_SOURCE_FILENAME}"
)
MATCHER_SOURCE_SHA256 = "d975e965b105311a6143194852297dff4f02aea5cc2e10cecfed966ca0e22503"
MATCHER_SOURCE_SIZE_BYTES = 47_632_827
MATCHER_PICKLE_AUDIT_SHA256 = "e7b998d087a5dcadd37713daf30b63cc571160c3180ebc138500ab662197e932"
MATCHER_FILENAME = "aliked_lightglue.safetensors"  # the deterministic conversion of the source
MATCHER_SHA256 = "9c630a386c74c534428370ce46253e1d0968655db180f97074cb6ad797bd2bc6"
MATCHER_SIZE_BYTES = 47_564_948
MATCHER_STATE_TENSORS = 253  # all float32 parameters; the confidence-threshold buffer is computed
MATCHER_PARAMETER_COUNT = 11_884_625

# --- the extractor: ALIKED-N(16) ----------------------------------------------------------------
EXTRACTOR_REPOSITORY = "Shiaoming/ALIKED"
EXTRACTOR_COMMIT = "683d7c65197395c0b3f01ebe76e1084a27e73a65"
EXTRACTOR_SOURCE_FILENAME = "aliked-n16.pth"
EXTRACTOR_SOURCE_URL = (
    f"https://raw.githubusercontent.com/{EXTRACTOR_REPOSITORY}/{EXTRACTOR_COMMIT}"
    f"/models/{EXTRACTOR_SOURCE_FILENAME}"
)
EXTRACTOR_SOURCE_SHA256 = "5be8704840ed662d9d8c561bf7279c222092674e7eb05fd0feab94899e9d82f2"
EXTRACTOR_SOURCE_SIZE_BYTES = 2_738_091
EXTRACTOR_PICKLE_AUDIT_SHA256 = "5b9f0ba08490293d6c17b9cef219991e1a6edda31609429679f8dca1af5a7b10"
EXTRACTOR_FILENAME = "aliked-n16.safetensors"
EXTRACTOR_SHA256 = "3c8ca40c0c985cd4d641e96e4b408b14d067b5b3521ac17b36590447d49d115a"
EXTRACTOR_SIZE_BYTES = 2_719_928
EXTRACTOR_STATE_TENSORS = 76  # 68 float32 parameter / running-statistic tensors + 8 int64 counters
EXTRACTOR_PARAMETER_COUNT = 677_356

# The globals a checkpoint pickle may import (the fleet's four); anything else fails the audit.
CKPT_ALLOWED_GLOBALS = frozenset(
    {
        "collections.OrderedDict",
        "torch.FloatStorage",
        "torch.LongStorage",
        "torch._utils._rebuild_tensor_v2",
    }
)

# The served (primary) weight file for the fleet's single-file conventions is the matcher.
MODEL_FILENAME = MATCHER_FILENAME
MODEL_SHA256 = MATCHER_SHA256
MODEL_SIZE_BYTES = MATCHER_SIZE_BYTES
SOURCE_FILENAMES = (MATCHER_SOURCE_FILENAME, EXTRACTOR_SOURCE_FILENAME)
CONVERTED_FILENAMES = (MATCHER_FILENAME, EXTRACTOR_FILENAME)

DEFAULT_MODEL_KEY = "lightglue-aliked"
UNSAFE_WEIGHT_EXTENSIONS = (
    ".bin",
    ".pt",
    ".pth",
    ".ckpt",
    ".pkl",
    ".pickle",
    ".h5",
    ".msgpack",
)
ALLOWED_CHECKPOINT_FILES = CONVERTED_FILENAMES

# Inference contract.
MAX_KEYPOINTS = 2048  # ALIKED keypoints per image (upstream demo default)
DETECTION_THRESHOLD = 0.2  # ALIKED keypoint score threshold (upstream default)
NMS_RADIUS = 2  # ALIKED non-maximum suppression radius, px (upstream default)
FILTER_THRESHOLD = 0.1  # LightGlue match threshold on the assignment score (upstream default)
DEPTH_CONFIDENCE = -1.0  # upstream's adaptive early exit (0.95) is off: all nine layers always run
WIDTH_CONFIDENCE = -1.0  # upstream's adaptive point pruning (0.99) is off: no keypoint is dropped
DIVISIBLE_BY = 8  # sample pairs are built with sides that are multiples of this (ALIKED pads to 32)
MIN_SIDE = 64
MAX_SIDE = 1024

**Module 2/7:** `src/lightglue_pipeline/metrics.py` (carried verbatim; see the note above)

In [ ]:
"""Homography-supervised matching metrics and two non-neural baselines, in numpy.

A record pairs an image with a warped copy of itself under a known 3 × 3 homography `H` (image0 → image1),
so every match has an exact reference: the reprojection error of `(x0, y0)` mapped by `H` against `(x1, y1)`.
For a set of records the pipeline reports:

- **precision at 3 px** (the fraction of returned matches with reprojection error under 3 px — the
  matching-precision reading), also at 1 px and 5 px;
- **matches per pair** and **inliers per pair** at 3 px (how much a downstream solver has to work with);
- **median reprojection error** of the inliers (sub-pixel accuracy);
- **homography accuracy at 3 px / 5 px**: the fraction of pairs whose homography, estimated from the
  matches by a normalised DLT inside a plain RANSAC loop, moves the four image corners by less than the
  threshold on average against the reference `H` (the usual HPatches-style reading; pairs with fewer than
  four inliers count as failures).

Three baselines a matcher must beat: the **identity guess** (every grid point maps to itself — correct only
where the warp is small), a **patch nearest neighbour** (for each grid point of image0 the best
normalised-cross-correlation 15 × 15 patch of image1 within a search window — a matcher that knows the
images through raw intensities) and, in the pipeline, the **descriptor mutual nearest neighbour** (the same
ALIKED keypoints and descriptors the model matches, paired by mutual nearest neighbour in descriptor space
with `mutual_nn_matches` below — what LightGlue is replacing). All return the same match structure the model
does and are scored by the same code.
"""
# ruff: noqa: E501  -- fleet metrics module written at the 110-column fleet width; this repo lints at 100

from __future__ import annotations

import math
from collections.abc import Mapping, Sequence
from typing import Any

import numpy as np
from PIL import Image

METRIC_DEFINITIONS = {
    "precision_3px": "fraction of returned matches whose reprojection error under the reference homography is below 3 px, averaged over pairs (a pair with no matches scores 0); in 0..1",
    "precision_1px": "the same at 1 px",
    "precision_5px": "the same at 5 px",
    "matches_per_pair": "mean number of returned matches per pair",
    "inliers_per_pair": "mean number of returned matches under 3 px per pair",
    "median_error_px": "median reprojection error of the inliers under 3 px, pooled over pairs; px",
    "homography_acc_3px": "fraction of pairs whose RANSAC-DLT homography from the matches moves the four corners by less than 3 px on average against the reference; in 0..1",
    "homography_acc_5px": "the same at 5 px",
}
THRESHOLDS = (1.0, 3.0, 5.0)
INLIER_PX = 3.0


def warp_points(points: np.ndarray, homography: np.ndarray) -> np.ndarray:
    """Apply a 3 × 3 homography to (N, 2) pixel coordinates."""
    pts = np.asarray(points, dtype=np.float64)
    if pts.ndim != 2 or pts.shape[1] != 2:
        raise ValueError("points must be an (N, 2) array")
    hom = np.concatenate([pts, np.ones((len(pts), 1))], axis=1) @ np.asarray(homography, dtype=np.float64).T
    with np.errstate(divide="ignore", invalid="ignore"):  # points at infinity under a degenerate candidate
        return hom[:, :2] / hom[:, 2:3]


def reprojection_errors(kpts0: np.ndarray, kpts1: np.ndarray, homography: np.ndarray) -> np.ndarray:
    """Per-match distance between `H · kpts0` and `kpts1`, in px."""
    kpts0 = np.asarray(kpts0, dtype=np.float64).reshape(-1, 2)
    kpts1 = np.asarray(kpts1, dtype=np.float64).reshape(-1, 2)
    if len(kpts0) != len(kpts1):
        raise ValueError("kpts0 and kpts1 must have the same length")
    if len(kpts0) == 0:
        return np.zeros((0,), dtype=np.float64)
    errors = np.linalg.norm(warp_points(kpts0, homography) - kpts1, axis=1)
    return np.where(np.isfinite(errors), errors, np.inf)


def _normalise(points: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    mean = points.mean(axis=0)
    scale = math.sqrt(2.0) / max(float(np.sqrt(((points - mean) ** 2).sum(axis=1)).mean()), 1e-9)
    transform = np.array([[scale, 0.0, -scale * mean[0]], [0.0, scale, -scale * mean[1]], [0.0, 0.0, 1.0]])
    hom = np.concatenate([points, np.ones((len(points), 1))], axis=1) @ transform.T
    return hom[:, :2], transform


def dlt_homography(kpts0: np.ndarray, kpts1: np.ndarray) -> np.ndarray | None:
    """Normalised direct linear transform from at least four correspondences; None when degenerate."""
    kpts0 = np.asarray(kpts0, dtype=np.float64)
    kpts1 = np.asarray(kpts1, dtype=np.float64)
    if len(kpts0) < 4:
        return None
    p0, t0 = _normalise(kpts0)
    p1, t1 = _normalise(kpts1)
    rows = []
    for (x, y), (u, v) in zip(p0, p1, strict=True):
        rows.append([-x, -y, -1.0, 0.0, 0.0, 0.0, u * x, u * y, u])
        rows.append([0.0, 0.0, 0.0, -x, -y, -1.0, v * x, v * y, v])
    a = np.asarray(rows)
    try:
        _u, sigma, vt = np.linalg.svd(a)
    except np.linalg.LinAlgError:
        return None
    if sigma[-2] < 1e-12:  # rank-deficient: collinear points
        return None
    h_norm = vt[-1].reshape(3, 3)
    homography = np.linalg.inv(t1) @ h_norm @ t0
    if abs(homography[2, 2]) < 1e-12:
        return None
    return homography / homography[2, 2]


def ransac_homography(
    kpts0: np.ndarray,
    kpts1: np.ndarray,
    *,
    threshold: float = INLIER_PX,
    iterations: int = 500,
    seed: int = 0,
) -> tuple[np.ndarray | None, np.ndarray]:
    """A plain RANSAC over four-point DLT samples, refit on the consensus set. Returns (H or None, inlier mask)."""
    kpts0 = np.asarray(kpts0, dtype=np.float64).reshape(-1, 2)
    kpts1 = np.asarray(kpts1, dtype=np.float64).reshape(-1, 2)
    n = len(kpts0)
    if n < 4:
        return None, np.zeros((n,), dtype=bool)
    rng = np.random.default_rng(seed)
    best_mask = np.zeros((n,), dtype=bool)
    for _ in range(iterations):
        sample = rng.choice(n, size=4, replace=False)
        candidate = dlt_homography(kpts0[sample], kpts1[sample])
        if candidate is None:
            continue
        mask = reprojection_errors(kpts0, kpts1, candidate) < threshold
        if mask.sum() > best_mask.sum():
            best_mask = mask
            if best_mask.sum() == n:
                break
    if best_mask.sum() < 4:
        return None, best_mask
    refit = dlt_homography(kpts0[best_mask], kpts1[best_mask])
    if refit is None:
        return None, best_mask
    return refit, reprojection_errors(kpts0, kpts1, refit) < threshold


def corner_error(estimated: np.ndarray, reference: np.ndarray, size: tuple[int, int]) -> float:
    """Mean displacement of the four image corners between two homographies, in px."""
    width, height = size
    corners = np.array([[0.0, 0.0], [width - 1.0, 0.0], [width - 1.0, height - 1.0], [0.0, height - 1.0]])
    return float(np.linalg.norm(warp_points(corners, estimated) - warp_points(corners, reference), axis=1).mean())


def pair_metrics(match: Mapping[str, Any], homography: np.ndarray, size: tuple[int, int]) -> dict[str, Any]:
    """Per-pair scores for one match result `{kpts0, kpts1}` against the reference homography."""
    kpts0 = np.asarray(match["kpts0"], dtype=np.float64).reshape(-1, 2)
    kpts1 = np.asarray(match["kpts1"], dtype=np.float64).reshape(-1, 2)
    errors = reprojection_errors(kpts0, kpts1, homography)
    out: dict[str, Any] = {"n_matches": int(len(errors))}
    for t in THRESHOLDS:
        out[f"precision_{int(t)}px"] = float((errors < t).mean()) if len(errors) else 0.0
    inliers = errors[errors < INLIER_PX]
    out["n_inliers"] = int(len(inliers))
    out["inlier_errors"] = inliers.tolist()
    estimated, _mask = ransac_homography(kpts0, kpts1)
    out["corner_error_px"] = corner_error(estimated, homography, size) if estimated is not None else math.inf
    return out


def matching_metrics(per_pair: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """Aggregate `pair_metrics` rows over a set of pairs."""
    if not per_pair:
        raise ValueError("at least one pair is required")
    pooled = [e for row in per_pair for e in row["inlier_errors"]]
    out: dict[str, Any] = {
        "n": len(per_pair),
        "matches_per_pair": float(np.mean([row["n_matches"] for row in per_pair])),
        "inliers_per_pair": float(np.mean([row["n_inliers"] for row in per_pair])),
        "median_error_px": float(np.median(pooled)) if pooled else math.inf,
        "homography_acc_3px": float(np.mean([row["corner_error_px"] < 3.0 for row in per_pair])),
        "homography_acc_5px": float(np.mean([row["corner_error_px"] < 5.0 for row in per_pair])),
        "definitions": METRIC_DEFINITIONS,
    }
    for t in THRESHOLDS:
        key = f"precision_{int(t)}px"
        out[key] = float(np.mean([row[key] for row in per_pair]))
    return out


# --------------------------------------------------------------------------------------------------
# non-neural baselines
# --------------------------------------------------------------------------------------------------


def grid_points(size: tuple[int, int], step: int = 32, margin: int = 16) -> np.ndarray:
    width, height = size
    xs = np.arange(margin, width - margin, step, dtype=np.float64)
    ys = np.arange(margin, height - margin, step, dtype=np.float64)
    gx, gy = np.meshgrid(xs, ys)
    return np.stack([gx.ravel(), gy.ravel()], axis=1)


def identity_baseline(image0: Image.Image, image1: Image.Image, *, step: int = 32) -> dict[str, Any]:
    """Every grid point of image0 is matched to the same coordinates in image1 (no motion assumed)."""
    pts = grid_points(image0.size, step=step)
    return {"kpts0": pts, "kpts1": pts.copy(), "confidence": np.ones(len(pts)), "baseline": "identity guess"}


def _gray(image: Image.Image) -> np.ndarray:
    return np.asarray(image.convert("L"), dtype=np.float64)


def _ncc(patch: np.ndarray, window: np.ndarray) -> np.ndarray:
    """Normalised cross-correlation of a (p, p) patch over every (p, p) position of a (h, w) window."""
    p = patch.shape[0]
    h, w = window.shape
    if h < p or w < p:
        return np.zeros((0, 0))
    strides = np.lib.stride_tricks.sliding_window_view(window, (p, p))  # (h-p+1, w-p+1, p, p)
    tiles = strides.reshape(strides.shape[0], strides.shape[1], -1)
    tiles = tiles - tiles.mean(axis=2, keepdims=True)
    flat = (patch - patch.mean()).ravel()
    denom = np.sqrt((tiles**2).sum(axis=2) * (flat**2).sum()) + 1e-9
    return (tiles @ flat) / denom


def patch_neighbour_baseline(
    image0: Image.Image,
    image1: Image.Image,
    *,
    step: int = 32,
    patch: int = 15,
    search: int = 48,
) -> dict[str, Any]:
    """For each grid point of image0, the position in image1 (within ±`search` px) whose `patch` × `patch`
    neighbourhood has the highest normalised cross-correlation with the point's own patch."""
    g0, g1 = _gray(image0), _gray(image1)
    half = patch // 2
    pts0 = grid_points(image0.size, step=step, margin=max(16, half + 1))
    kpts0, kpts1, conf = [], [], []
    for x, y in pts0:
        xi, yi = int(round(x)), int(round(y))
        tile = g0[yi - half : yi + half + 1, xi - half : xi + half + 1]
        y0, y1 = max(0, yi - search - half), min(g1.shape[0], yi + search + half + 1)
        x0, x1 = max(0, xi - search - half), min(g1.shape[1], xi + search + half + 1)
        scores = _ncc(tile, g1[y0:y1, x0:x1])
        if scores.size == 0:
            continue
        best = np.unravel_index(int(np.argmax(scores)), scores.shape)
        kpts0.append([x, y])
        kpts1.append([x0 + best[1] + half, y0 + best[0] + half])
        conf.append(float(scores[best]))
    return {
        "kpts0": np.asarray(kpts0, dtype=np.float64).reshape(-1, 2),
        "kpts1": np.asarray(kpts1, dtype=np.float64).reshape(-1, 2),
        "confidence": np.asarray(conf, dtype=np.float64),
        "baseline": f"patch nearest neighbour ({patch}x{patch} NCC, ±{search} px search)",
    }


def mutual_nn_matches(
    desc0: np.ndarray, desc1: np.ndarray, *, min_similarity: float = 0.0
) -> tuple[np.ndarray, np.ndarray]:
    """Mutual nearest neighbours of two L2-normalised descriptor sets by cosine similarity: index pairs
    (M, 2) and their similarities (M,). The classical detector-and-describe matcher without any learned
    context — the reference for what a learned matcher adds on the same keypoints."""
    d0 = np.asarray(desc0, dtype=np.float64)
    d1 = np.asarray(desc1, dtype=np.float64)
    if d0.ndim != 2 or d1.ndim != 2 or d0.shape[1] != d1.shape[1]:
        raise ValueError("descriptors must be (N, D) arrays with a common D")
    if len(d0) == 0 or len(d1) == 0:
        return np.zeros((0, 2), dtype=np.int64), np.zeros((0,), dtype=np.float64)
    sim = d0 @ d1.T
    nn01 = sim.argmax(axis=1)
    nn10 = sim.argmax(axis=0)
    i = np.arange(len(d0))
    mutual = nn10[nn01] == i
    scores = sim[i, nn01]
    keep = mutual & (scores >= min_similarity)
    pairs = np.stack([i[keep], nn01[keep]], axis=1)
    return pairs.astype(np.int64), scores[keep]

**Module 3/7:** `src/lightglue_pipeline/modeling.py` (carried verbatim; see the note above)

In [ ]:
"""ALIKED (Zhao et al., IEEE TIM 2023) local-feature extractor and LightGlue (Lindenberger, Sarlin and Pollefeys,
ICCV 2023) matcher, vendored from https://github.com/cvg/LightGlue at commit
eb42fee2d71449efb0aa5c10549752b5d75384d8 (Apache-2.0; ``lightglue/aliked.py`` is the authors' BSD-3-Clause port of
https://github.com/Shiaoming/ALIKED and keeps its licence header below): ``lightglue/utils.py`` (the ``Extractor``
base only), ``lightglue/aliked.py`` and ``lightglue/lightglue.py`` concatenated in dependency order with the
package-relative imports removed. Nothing is fetched or unpickled at construction: ``model.load_components``
loads the audited, converted safetensors state dicts strictly.

Edits against upstream, each marked ``# vendored:`` in place and listed in ``docs/WEIGHTS.md``: the kornia
``grayscale_to_rgb`` call is a channel ``expand``; ``torchvision.models.resnet.conv1x1`` / ``conv3x3`` are the two
one-line helpers below (``torchvision.ops.deform_conv2d`` is the one torchvision call kept); the optional
``flash_attn`` import, the download-at-construction code of both classes, ``ALIKED.describe`` and the cv2 /
kornia image utilities are not carried; and LightGlue's ``confidence_thresholds`` buffer is registered
``persistent=False`` (it is computed from the configuration and absent from the checkpoint) so the checkpoint
loads with ``strict=True`` instead of upstream's ``strict=False``.
"""
# ruff: noqa: E501, N801, N802, N803, N806, E741, B905, B006, B008, F841, E712, UP004, UP006, UP008, UP031, UP032, UP035, UP045  -- vendored code kept as upstream wrote it, for auditability

from __future__ import annotations

import warnings
from types import SimpleNamespace
from typing import Callable, List, Optional, Tuple

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from torch.nn.modules.utils import _pair
from torchvision.ops import deform_conv2d

UPSTREAM_REPOSITORY = "https://github.com/cvg/LightGlue"
UPSTREAM_COMMIT = "eb42fee2d71449efb0aa5c10549752b5d75384d8"
ALIKED_REPOSITORY = "https://github.com/Shiaoming/ALIKED"


def conv3x3(in_planes: int, out_planes: int, stride: int = 1, groups: int = 1, dilation: int = 1) -> nn.Conv2d:
    """``torchvision.models.resnet.conv3x3``, carried so the vendored code does not import torchvision.models."""
    return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride, padding=dilation, groups=groups, bias=False, dilation=dilation)


def conv1x1(in_planes: int, out_planes: int, stride: int = 1) -> nn.Conv2d:
    """``torchvision.models.resnet.conv1x1``."""
    return nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False)


# --------------------------------------------------------------------------------------------------
# lightglue/utils.py (Extractor base only)
# --------------------------------------------------------------------------------------------------


class Extractor(torch.nn.Module):
    # upstream `lightglue/utils.py`: only the configuration base is carried; `extract` (kornia resize) is not
    def __init__(self, **conf):
        super().__init__()
        self.conf = SimpleNamespace(**{**self.default_conf, **conf})


# --------------------------------------------------------------------------------------------------
# lightglue/aliked.py
# --------------------------------------------------------------------------------------------------

# BSD 3-Clause License

# Copyright (c) 2022, Zhao Xiaoming
# All rights reserved.

# Redistribution and use in source and binary forms, with or without
# modification, are permitted provided that the following conditions are met:

# 1. Redistributions of source code must retain the above copyright notice, this
#    list of conditions and the following disclaimer.

# 2. Redistributions in binary form must reproduce the above copyright notice,
#    this list of conditions and the following disclaimer in the documentation
#    and/or other materials provided with the distribution.

# 3. Neither the name of the copyright holder nor the names of its
#    contributors may be used to endorse or promote products derived from
#    this software without specific prior written permission.

# THIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS "AS IS"
# AND ANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE
# IMPLIED WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE ARE
# DISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT HOLDER OR CONTRIBUTORS BE LIABLE
# FOR ANY DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR CONSEQUENTIAL
# DAMAGES (INCLUDING, BUT NOT LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR
# SERVICES; LOSS OF USE, DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER
# CAUSED AND ON ANY THEORY OF LIABILITY, WHETHER IN CONTRACT, STRICT LIABILITY,
# OR TORT (INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY OUT OF THE USE
# OF THIS SOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGE.

# Authors:
# Xiaoming Zhao, Xingming Wu, Weihai Chen, Peter C.Y. Chen, Qingsong Xu, and Zhengguo Li
# Code from https://github.com/Shiaoming/ALIKED



def get_patches(
    tensor: torch.Tensor, required_corners: torch.Tensor, ps: int
) -> torch.Tensor:
    c, h, w = tensor.shape
    corner = (required_corners - ps / 2 + 1).long()
    corner[:, 0] = corner[:, 0].clamp(min=0, max=w - 1 - ps)
    corner[:, 1] = corner[:, 1].clamp(min=0, max=h - 1 - ps)
    offset = torch.arange(0, ps)

    kw = {"indexing": "ij"} if torch.__version__ >= "1.10" else {}
    x, y = torch.meshgrid(offset, offset, **kw)
    patches = torch.stack((x, y)).permute(2, 1, 0).unsqueeze(2)
    patches = patches.to(corner) + corner[None, None]
    pts = patches.reshape(-1, 2)
    sampled = tensor.permute(1, 2, 0)[tuple(pts.T)[::-1]]
    sampled = sampled.reshape(ps, ps, -1, c)
    assert sampled.shape[:3] == patches.shape[:3]
    return sampled.permute(2, 3, 0, 1)


def simple_nms(scores: torch.Tensor, nms_radius: int):
    """Fast Non-maximum suppression to remove nearby points"""

    zeros = torch.zeros_like(scores)
    max_mask = scores == torch.nn.functional.max_pool2d(
        scores, kernel_size=nms_radius * 2 + 1, stride=1, padding=nms_radius
    )

    for _ in range(2):
        supp_mask = (
            torch.nn.functional.max_pool2d(
                max_mask.float(),
                kernel_size=nms_radius * 2 + 1,
                stride=1,
                padding=nms_radius,
            )
            > 0
        )
        supp_scores = torch.where(supp_mask, zeros, scores)
        new_max_mask = supp_scores == torch.nn.functional.max_pool2d(
            supp_scores, kernel_size=nms_radius * 2 + 1, stride=1, padding=nms_radius
        )
        max_mask = max_mask | (new_max_mask & (~supp_mask))
    return torch.where(max_mask, scores, zeros)


class DKD(nn.Module):
    def __init__(
        self,
        radius: int = 2,
        top_k: int = 0,
        scores_th: float = 0.2,
        n_limit: int = 20000,
    ):
        """
        Args:
            radius: soft detection radius, kernel size is (2 * radius + 1)
            top_k: top_k > 0: return top k keypoints
            scores_th: top_k <= 0 threshold mode:
                scores_th > 0: return keypoints with scores>scores_th
                else: return keypoints with scores > scores.mean()
            n_limit: max number of keypoint in threshold mode
        """
        super().__init__()
        self.radius = radius
        self.top_k = top_k
        self.scores_th = scores_th
        self.n_limit = n_limit
        self.kernel_size = 2 * self.radius + 1
        self.temperature = 0.1  # tuned temperature
        self.unfold = nn.Unfold(kernel_size=self.kernel_size, padding=self.radius)
        # local xy grid
        x = torch.linspace(-self.radius, self.radius, self.kernel_size)
        # (kernel_size*kernel_size) x 2 : (w,h)
        kw = {"indexing": "ij"} if torch.__version__ >= "1.10" else {}
        self.hw_grid = (
            torch.stack(torch.meshgrid([x, x], **kw)).view(2, -1).t()[:, [1, 0]]
        )

    def forward(
        self,
        scores_map: torch.Tensor,
        sub_pixel: bool = True,
        image_size: Optional[torch.Tensor] = None,
    ):
        """
        :param scores_map: Bx1xHxW
        :param descriptor_map: BxCxHxW
        :param sub_pixel: whether to use sub-pixel keypoint detection
        :return: kpts: list[Nx2,...]; kptscores: list[N,....] normalised position: -1~1
        """
        b, c, h, w = scores_map.shape
        scores_nograd = scores_map.detach()
        nms_scores = simple_nms(scores_nograd, self.radius)

        # remove border
        nms_scores[:, :, : self.radius, :] = 0
        nms_scores[:, :, :, : self.radius] = 0
        if image_size is not None:
            for i in range(scores_map.shape[0]):
                w, h = image_size[i].long()
                nms_scores[i, :, h.item() - self.radius :, :] = 0
                nms_scores[i, :, :, w.item() - self.radius :] = 0
        else:
            nms_scores[:, :, -self.radius :, :] = 0
            nms_scores[:, :, :, -self.radius :] = 0

        # detect keypoints without grad
        if self.top_k > 0:
            topk = torch.topk(nms_scores.view(b, -1), self.top_k)
            indices_keypoints = [topk.indices[i] for i in range(b)]  # B x top_k
        else:
            if self.scores_th > 0:
                masks = nms_scores > self.scores_th
                if masks.sum() == 0:
                    th = scores_nograd.reshape(b, -1).mean(dim=1)  # th = self.scores_th
                    masks = nms_scores > th.reshape(b, 1, 1, 1)
            else:
                th = scores_nograd.reshape(b, -1).mean(dim=1)  # th = self.scores_th
                masks = nms_scores > th.reshape(b, 1, 1, 1)
            masks = masks.reshape(b, -1)

            indices_keypoints = []  # list, B x (any size)
            scores_view = scores_nograd.reshape(b, -1)
            for mask, scores in zip(masks, scores_view):
                indices = mask.nonzero()[:, 0]
                if len(indices) > self.n_limit:
                    kpts_sc = scores[indices]
                    sort_idx = kpts_sc.sort(descending=True)[1]
                    sel_idx = sort_idx[: self.n_limit]
                    indices = indices[sel_idx]
                indices_keypoints.append(indices)

        wh = torch.tensor([w - 1, h - 1], device=scores_nograd.device)

        keypoints = []
        scoredispersitys = []
        kptscores = []
        if sub_pixel:
            # detect soft keypoints with grad backpropagation
            patches = self.unfold(scores_map)  # B x (kernel**2) x (H*W)
            self.hw_grid = self.hw_grid.to(scores_map)  # to device
            for b_idx in range(b):
                patch = patches[b_idx].t()  # (H*W) x (kernel**2)
                indices_kpt = indices_keypoints[
                    b_idx
                ]  # one dimension vector, say its size is M
                patch_scores = patch[indices_kpt]  # M x (kernel**2)
                keypoints_xy_nms = torch.stack(
                    [indices_kpt % w, torch.div(indices_kpt, w, rounding_mode="trunc")],
                    dim=1,
                )  # Mx2

                # max is detached to prevent undesired backprop loops in the graph
                max_v = patch_scores.max(dim=1).values.detach()[:, None]
                x_exp = (
                    (patch_scores - max_v) / self.temperature
                ).exp()  # M * (kernel**2), in [0, 1]

                # \frac{ \sum{(i,j) \times \exp(x/T)} }{ \sum{\exp(x/T)} }
                xy_residual = (
                    x_exp @ self.hw_grid / x_exp.sum(dim=1)[:, None]
                )  # Soft-argmax, Mx2

                hw_grid_dist2 = (
                    torch.norm(
                        (self.hw_grid[None, :, :] - xy_residual[:, None, :])
                        / self.radius,
                        dim=-1,
                    )
                    ** 2
                )
                scoredispersity = (x_exp * hw_grid_dist2).sum(dim=1) / x_exp.sum(dim=1)

                # compute result keypoints
                keypoints_xy = keypoints_xy_nms + xy_residual
                keypoints_xy = keypoints_xy / wh * 2 - 1  # (w,h) -> (-1~1,-1~1)

                kptscore = torch.nn.functional.grid_sample(
                    scores_map[b_idx].unsqueeze(0),
                    keypoints_xy.view(1, 1, -1, 2),
                    mode="bilinear",
                    align_corners=True,
                )[
                    0, 0, 0, :
                ]  # CxN

                keypoints.append(keypoints_xy)
                scoredispersitys.append(scoredispersity)
                kptscores.append(kptscore)
        else:
            for b_idx in range(b):
                indices_kpt = indices_keypoints[
                    b_idx
                ]  # one dimension vector, say its size is M
                # To avoid warning: UserWarning: __floordiv__ is deprecated
                keypoints_xy_nms = torch.stack(
                    [indices_kpt % w, torch.div(indices_kpt, w, rounding_mode="trunc")],
                    dim=1,
                )  # Mx2
                keypoints_xy = keypoints_xy_nms / wh * 2 - 1  # (w,h) -> (-1~1,-1~1)
                kptscore = torch.nn.functional.grid_sample(
                    scores_map[b_idx].unsqueeze(0),
                    keypoints_xy.view(1, 1, -1, 2),
                    mode="bilinear",
                    align_corners=True,
                )[
                    0, 0, 0, :
                ]  # CxN
                keypoints.append(keypoints_xy)
                scoredispersitys.append(kptscore)  # for jit.script compatability
                kptscores.append(kptscore)

        return keypoints, kptscores, scoredispersitys


class InputPadder(object):
    """Pads images such that dimensions are divisible by 8"""

    def __init__(self, h: int, w: int, divis_by: int = 8):
        self.ht = h
        self.wd = w
        pad_ht = (((self.ht // divis_by) + 1) * divis_by - self.ht) % divis_by
        pad_wd = (((self.wd // divis_by) + 1) * divis_by - self.wd) % divis_by
        self._pad = [
            pad_wd // 2,
            pad_wd - pad_wd // 2,
            pad_ht // 2,
            pad_ht - pad_ht // 2,
        ]

    def pad(self, x: torch.Tensor):
        assert x.ndim == 4
        return F.pad(x, self._pad, mode="replicate")

    def unpad(self, x: torch.Tensor):
        assert x.ndim == 4
        ht = x.shape[-2]
        wd = x.shape[-1]
        c = [self._pad[2], ht - self._pad[3], self._pad[0], wd - self._pad[1]]
        return x[..., c[0] : c[1], c[2] : c[3]]


class DeformableConv2d(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size=3,
        stride=1,
        padding=1,
        bias=False,
        mask=False,
    ):
        super(DeformableConv2d, self).__init__()

        self.padding = padding
        self.mask = mask

        self.channel_num = (
            3 * kernel_size * kernel_size if mask else 2 * kernel_size * kernel_size
        )
        self.offset_conv = nn.Conv2d(
            in_channels,
            self.channel_num,
            kernel_size=kernel_size,
            stride=stride,
            padding=self.padding,
            bias=True,
        )

        self.regular_conv = nn.Conv2d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=self.padding,
            bias=bias,
        )

    def forward(self, x):
        h, w = x.shape[2:]
        max_offset = max(h, w) / 4.0

        out = self.offset_conv(x)
        if self.mask:
            o1, o2, mask = torch.chunk(out, 3, dim=1)
            offset = torch.cat((o1, o2), dim=1)
            mask = torch.sigmoid(mask)
        else:
            offset = out
            mask = None
        offset = offset.clamp(-max_offset, max_offset)
        x = deform_conv2d(
            input=x,
            offset=offset,
            weight=self.regular_conv.weight,
            bias=self.regular_conv.bias,
            padding=self.padding,
            mask=mask,
        )
        return x


def get_conv(
    inplanes,
    planes,
    kernel_size=3,
    stride=1,
    padding=1,
    bias=False,
    conv_type="conv",
    mask=False,
):
    if conv_type == "conv":
        conv = nn.Conv2d(
            inplanes,
            planes,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            bias=bias,
        )
    elif conv_type == "dcn":
        conv = DeformableConv2d(
            inplanes,
            planes,
            kernel_size=kernel_size,
            stride=stride,
            padding=_pair(padding),
            bias=bias,
            mask=mask,
        )
    else:
        raise TypeError
    return conv


class ConvBlock(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        gate: Optional[Callable[..., nn.Module]] = None,
        norm_layer: Optional[Callable[..., nn.Module]] = None,
        conv_type: str = "conv",
        mask: bool = False,
    ):
        super().__init__()
        if gate is None:
            self.gate = nn.ReLU(inplace=True)
        else:
            self.gate = gate
        if norm_layer is None:
            norm_layer = nn.BatchNorm2d
        self.conv1 = get_conv(
            in_channels, out_channels, kernel_size=3, conv_type=conv_type, mask=mask
        )
        self.bn1 = norm_layer(out_channels)
        self.conv2 = get_conv(
            out_channels, out_channels, kernel_size=3, conv_type=conv_type, mask=mask
        )
        self.bn2 = norm_layer(out_channels)

    def forward(self, x):
        x = self.gate(self.bn1(self.conv1(x)))  # B x in_channels x H x W
        x = self.gate(self.bn2(self.conv2(x)))  # B x out_channels x H x W
        return x


# modified based on torchvision\models\resnet.py#27->BasicBlock
class ResBlock(nn.Module):
    expansion: int = 1

    def __init__(
        self,
        inplanes: int,
        planes: int,
        stride: int = 1,
        downsample: Optional[nn.Module] = None,
        groups: int = 1,
        base_width: int = 64,
        dilation: int = 1,
        gate: Optional[Callable[..., nn.Module]] = None,
        norm_layer: Optional[Callable[..., nn.Module]] = None,
        conv_type: str = "conv",
        mask: bool = False,
    ) -> None:
        super(ResBlock, self).__init__()
        if gate is None:
            self.gate = nn.ReLU(inplace=True)
        else:
            self.gate = gate
        if norm_layer is None:
            norm_layer = nn.BatchNorm2d
        if groups != 1 or base_width != 64:
            raise ValueError("ResBlock only supports groups=1 and base_width=64")
        if dilation > 1:
            raise NotImplementedError("Dilation > 1 not supported in ResBlock")
        # Both self.conv1 and self.downsample layers
        # downsample the input when stride != 1
        self.conv1 = get_conv(
            inplanes, planes, kernel_size=3, conv_type=conv_type, mask=mask
        )
        self.bn1 = norm_layer(planes)
        self.conv2 = get_conv(
            planes, planes, kernel_size=3, conv_type=conv_type, mask=mask
        )
        self.bn2 = norm_layer(planes)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.gate(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.gate(out)

        return out


class SDDH(nn.Module):
    def __init__(
        self,
        dims: int,
        kernel_size: int = 3,
        n_pos: int = 8,
        gate=nn.ReLU(),
        conv2D=False,
        mask=False,
    ):
        super(SDDH, self).__init__()
        self.kernel_size = kernel_size
        self.n_pos = n_pos
        self.conv2D = conv2D
        self.mask = mask

        self.get_patches_func = get_patches

        # estimate offsets
        self.channel_num = 3 * n_pos if mask else 2 * n_pos
        self.offset_conv = nn.Sequential(
            nn.Conv2d(
                dims,
                self.channel_num,
                kernel_size=kernel_size,
                stride=1,
                padding=0,
                bias=True,
            ),
            gate,
            nn.Conv2d(
                self.channel_num,
                self.channel_num,
                kernel_size=1,
                stride=1,
                padding=0,
                bias=True,
            ),
        )

        # sampled feature conv
        self.sf_conv = nn.Conv2d(
            dims, dims, kernel_size=1, stride=1, padding=0, bias=False
        )

        # convM
        if not conv2D:
            # deformable desc weights
            agg_weights = torch.nn.Parameter(torch.rand(n_pos, dims, dims))
            self.register_parameter("agg_weights", agg_weights)
        else:
            self.convM = nn.Conv2d(
                dims * n_pos, dims, kernel_size=1, stride=1, padding=0, bias=False
            )

    def forward(self, x, keypoints):
        # x: [B,C,H,W]
        # keypoints: list, [[N_kpts,2], ...] (w,h)
        b, c, h, w = x.shape
        wh = torch.tensor([[w - 1, h - 1]], device=x.device)
        max_offset = max(h, w) / 4.0

        offsets = []
        descriptors = []
        # get offsets for each keypoint
        for ib in range(b):
            xi, kptsi = x[ib], keypoints[ib]
            kptsi_wh = (kptsi / 2 + 0.5) * wh
            N_kpts = len(kptsi)

            if self.kernel_size > 1:
                patch = self.get_patches_func(
                    xi, kptsi_wh.long(), self.kernel_size
                )  # [N_kpts, C, K, K]
            else:
                kptsi_wh_long = kptsi_wh.long()
                patch = (
                    xi[:, kptsi_wh_long[:, 1], kptsi_wh_long[:, 0]]
                    .permute(1, 0)
                    .reshape(N_kpts, c, 1, 1)
                )

            offset = self.offset_conv(patch).clamp(
                -max_offset, max_offset
            )  # [N_kpts, 2*n_pos, 1, 1]
            if self.mask:
                offset = (
                    offset[:, :, 0, 0].view(N_kpts, 3, self.n_pos).permute(0, 2, 1)
                )  # [N_kpts, n_pos, 3]
                offset = offset[:, :, :-1]  # [N_kpts, n_pos, 2]
                mask_weight = torch.sigmoid(offset[:, :, -1])  # [N_kpts, n_pos]
            else:
                offset = (
                    offset[:, :, 0, 0].view(N_kpts, 2, self.n_pos).permute(0, 2, 1)
                )  # [N_kpts, n_pos, 2]
            offsets.append(offset)  # for visualization

            # get sample positions
            pos = kptsi_wh.unsqueeze(1) + offset  # [N_kpts, n_pos, 2]
            pos = 2.0 * pos / wh[None] - 1
            pos = pos.reshape(1, N_kpts * self.n_pos, 1, 2)

            # sample features
            features = F.grid_sample(
                xi.unsqueeze(0), pos, mode="bilinear", align_corners=True
            )  # [1,C,(N_kpts*n_pos),1]
            features = features.reshape(c, N_kpts, self.n_pos, 1).permute(
                1, 0, 2, 3
            )  # [N_kpts, C, n_pos, 1]
            if self.mask:
                features = torch.einsum("ncpo,np->ncpo", features, mask_weight)

            features = torch.selu_(self.sf_conv(features)).squeeze(
                -1
            )  # [N_kpts, C, n_pos]
            # convM
            if not self.conv2D:
                descs = torch.einsum(
                    "ncp,pcd->nd", features, self.agg_weights
                )  # [N_kpts, C]
            else:
                features = features.reshape(N_kpts, -1)[
                    :, :, None, None
                ]  # [N_kpts, C*n_pos, 1, 1]
                descs = self.convM(features).squeeze()  # [N_kpts, C]

            # normalize
            descs = F.normalize(descs, p=2.0, dim=1)
            descriptors.append(descs)

        return descriptors, offsets


class ALIKED(Extractor):
    default_conf = {
        "model_name": "aliked-n16",
        "max_num_keypoints": -1,
        "detection_threshold": 0.2,
        "nms_radius": 2,
    }

    checkpoint_url = "https://github.com/Shiaoming/ALIKED/raw/main/models/{}.pth"

    n_limit_max = 20000

    # c1, c2, c3, c4, dim, K, M
    cfgs = {
        "aliked-t16": [8, 16, 32, 64, 64, 3, 16],
        "aliked-n16": [16, 32, 64, 128, 128, 3, 16],
        "aliked-n16rot": [16, 32, 64, 128, 128, 3, 16],
        "aliked-n32": [16, 32, 64, 128, 128, 3, 32],
    }
    preprocess_conf = {
        "resize": 1024,
    }

    required_data_keys = ["image"]

    def __init__(self, **conf):
        super().__init__(**conf)  # Update with default configuration.
        conf = self.conf
        c1, c2, c3, c4, dim, K, M = self.cfgs[conf.model_name]
        conv_types = ["conv", "conv", "dcn", "dcn"]
        conv2D = False
        mask = False

        # build model
        self.pool2 = nn.AvgPool2d(kernel_size=2, stride=2)
        self.pool4 = nn.AvgPool2d(kernel_size=4, stride=4)
        self.norm = nn.BatchNorm2d
        self.gate = nn.SELU(inplace=True)
        self.block1 = ConvBlock(3, c1, self.gate, self.norm, conv_type=conv_types[0])
        self.block2 = self.get_resblock(c1, c2, conv_types[1], mask)
        self.block3 = self.get_resblock(c2, c3, conv_types[2], mask)
        self.block4 = self.get_resblock(c3, c4, conv_types[3], mask)

        self.conv1 = conv1x1(c1, dim // 4)
        self.conv2 = conv1x1(c2, dim // 4)
        self.conv3 = conv1x1(c3, dim // 4)
        self.conv4 = conv1x1(dim, dim // 4)
        self.upsample2 = nn.Upsample(
            scale_factor=2, mode="bilinear", align_corners=True
        )
        self.upsample4 = nn.Upsample(
            scale_factor=4, mode="bilinear", align_corners=True
        )
        self.upsample8 = nn.Upsample(
            scale_factor=8, mode="bilinear", align_corners=True
        )
        self.upsample32 = nn.Upsample(
            scale_factor=32, mode="bilinear", align_corners=True
        )
        self.score_head = nn.Sequential(
            conv1x1(dim, 8),
            self.gate,
            conv3x3(8, 4),
            self.gate,
            conv3x3(4, 4),
            self.gate,
            conv3x3(4, 1),
        )
        self.desc_head = SDDH(dim, K, M, gate=self.gate, conv2D=conv2D, mask=mask)
        self.dkd = DKD(
            radius=conf.nms_radius,
            top_k=-1 if conf.detection_threshold > 0 else conf.max_num_keypoints,
            scores_th=conf.detection_threshold,
            n_limit=(
                conf.max_num_keypoints
                if conf.max_num_keypoints > 0
                else self.n_limit_max
            ),
        )

        # vendored: no download at construction; `model.load_components` loads the audited, converted
        # `aliked-n16.safetensors` strictly

    def get_resblock(self, c_in, c_out, conv_type, mask):
        return ResBlock(
            c_in,
            c_out,
            1,
            nn.Conv2d(c_in, c_out, 1),
            gate=self.gate,
            norm_layer=self.norm,
            conv_type=conv_type,
            mask=mask,
        )

    def extract_dense_map(self, image):
        # Pads images such that dimensions are divisible by
        div_by = 2**5
        padder = InputPadder(image.shape[-2], image.shape[-1], div_by)
        image = padder.pad(image)

        # ================================== feature encoder
        x1 = self.block1(image)  # B x c1 x H x W
        x2 = self.pool2(x1)
        x2 = self.block2(x2)  # B x c2 x H/2 x W/2
        x3 = self.pool4(x2)
        x3 = self.block3(x3)  # B x c3 x H/8 x W/8
        x4 = self.pool4(x3)
        x4 = self.block4(x4)  # B x dim x H/32 x W/32
        # ================================== feature aggregation
        x1 = self.gate(self.conv1(x1))  # B x dim//4 x H x W
        x2 = self.gate(self.conv2(x2))  # B x dim//4 x H//2 x W//2
        x3 = self.gate(self.conv3(x3))  # B x dim//4 x H//8 x W//8
        x4 = self.gate(self.conv4(x4))  # B x dim//4 x H//32 x W//32
        x2_up = self.upsample2(x2)  # B x dim//4 x H x W
        x3_up = self.upsample8(x3)  # B x dim//4 x H x W
        x4_up = self.upsample32(x4)  # B x dim//4 x H x W
        x1234 = torch.cat([x1, x2_up, x3_up, x4_up], dim=1)
        # ================================== score head
        score_map = torch.sigmoid(self.score_head(x1234))
        feature_map = torch.nn.functional.normalize(x1234, p=2, dim=1)

        # Unpads images
        feature_map = padder.unpad(feature_map)
        score_map = padder.unpad(score_map)

        return feature_map, score_map

    # vendored: `describe` (kornia-resized re-description of given keypoints) is not carried

    def forward(self, data: dict) -> dict:
        image = data["image"]
        if image.shape[1] == 1:
            image = image.expand(-1, 3, -1, -1)  # vendored: kornia.color.grayscale_to_rgb replaced
        feature_map, score_map = self.extract_dense_map(image)
        keypoints, kptscores, scoredispersitys = self.dkd(
            score_map, image_size=data.get("image_size")
        )
        descriptors, offsets = self.desc_head(feature_map, keypoints)

        _, _, h, w = image.shape
        wh = torch.tensor([w - 1, h - 1], device=image.device)
        # no padding required
        # we can set detection_threshold=-1 and conf.max_num_keypoints > 0
        return {
            "keypoints": wh * (torch.stack(keypoints) + 1) / 2.0,  # B x N x 2
            "descriptors": torch.stack(descriptors),  # B x N x D
            "keypoint_scores": torch.stack(kptscores),  # B x N
        }


# --------------------------------------------------------------------------------------------------
# lightglue/lightglue.py
# --------------------------------------------------------------------------------------------------

FlashCrossAttention = None  # vendored: the optional flash-attn import is not carried; torch SDPA is used

if FlashCrossAttention or hasattr(F, "scaled_dot_product_attention"):
    FLASH_AVAILABLE = True
else:
    FLASH_AVAILABLE = False

torch.backends.cudnn.deterministic = True


AMP_CUSTOM_FWD_F32 = (
    torch.amp.custom_fwd(cast_inputs=torch.float32, device_type="cuda")
    if hasattr(torch, "amp") and hasattr(torch.amp, "custom_fwd")
    else torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)
)


@AMP_CUSTOM_FWD_F32
def normalize_keypoints(
    kpts: torch.Tensor, size: Optional[torch.Tensor] = None
) -> torch.Tensor:
    if size is None:
        size = 1 + kpts.max(-2).values - kpts.min(-2).values
    elif not isinstance(size, torch.Tensor):
        size = torch.tensor(size, device=kpts.device, dtype=kpts.dtype)
    size = size.to(kpts)
    shift = size / 2
    scale = size.max(-1).values / 2
    kpts = (kpts - shift[..., None, :]) / scale[..., None, None]
    return kpts


def pad_to_length(x: torch.Tensor, length: int) -> Tuple[torch.Tensor]:
    if length <= x.shape[-2]:
        return x, torch.ones_like(x[..., :1], dtype=torch.bool)
    pad = torch.ones(
        *x.shape[:-2], length - x.shape[-2], x.shape[-1], device=x.device, dtype=x.dtype
    )
    y = torch.cat([x, pad], dim=-2)
    mask = torch.zeros(*y.shape[:-1], 1, dtype=torch.bool, device=x.device)
    mask[..., : x.shape[-2], :] = True
    return y, mask


def rotate_half(x: torch.Tensor) -> torch.Tensor:
    x = x.unflatten(-1, (-1, 2))
    x1, x2 = x.unbind(dim=-1)
    return torch.stack((-x2, x1), dim=-1).flatten(start_dim=-2)


def apply_cached_rotary_emb(freqs: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
    return (t * freqs[0]) + (rotate_half(t) * freqs[1])


class LearnableFourierPositionalEncoding(nn.Module):
    def __init__(self, M: int, dim: int, F_dim: int = None, gamma: float = 1.0) -> None:
        super().__init__()
        F_dim = F_dim if F_dim is not None else dim
        self.gamma = gamma
        self.Wr = nn.Linear(M, F_dim // 2, bias=False)
        nn.init.normal_(self.Wr.weight.data, mean=0, std=self.gamma**-2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """encode position vector"""
        projected = self.Wr(x)
        cosines, sines = torch.cos(projected), torch.sin(projected)
        emb = torch.stack([cosines, sines], 0).unsqueeze(-3)
        return emb.repeat_interleave(2, dim=-1)


class TokenConfidence(nn.Module):
    def __init__(self, dim: int) -> None:
        super().__init__()
        self.token = nn.Sequential(nn.Linear(dim, 1), nn.Sigmoid())

    def forward(self, desc0: torch.Tensor, desc1: torch.Tensor):
        """get confidence tokens"""
        return (
            self.token(desc0.detach()).squeeze(-1),
            self.token(desc1.detach()).squeeze(-1),
        )


class Attention(nn.Module):
    def __init__(self, allow_flash: bool) -> None:
        super().__init__()
        if allow_flash and not FLASH_AVAILABLE:
            warnings.warn(
                "FlashAttention is not available. For optimal speed, "
                "consider installing torch >= 2.0 or flash-attn.",
                stacklevel=2,
            )
        self.enable_flash = allow_flash and FLASH_AVAILABLE
        self.has_sdp = hasattr(F, "scaled_dot_product_attention")
        if allow_flash and FlashCrossAttention:
            self.flash_ = FlashCrossAttention()
        if self.has_sdp:
            torch.backends.cuda.enable_flash_sdp(allow_flash)

    def forward(self, q, k, v, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        if q.shape[-2] == 0 or k.shape[-2] == 0:
            return q.new_zeros((*q.shape[:-1], v.shape[-1]))
        if self.enable_flash and q.device.type == "cuda":
            # use torch 2.0 scaled_dot_product_attention with flash
            if self.has_sdp:
                args = [x.half().contiguous() for x in [q, k, v]]
                v = F.scaled_dot_product_attention(*args, attn_mask=mask).to(q.dtype)
                return v if mask is None else v.nan_to_num()
            else:
                assert mask is None
                q, k, v = [x.transpose(-2, -3).contiguous() for x in [q, k, v]]
                m = self.flash_(q.half(), torch.stack([k, v], 2).half())
                return m.transpose(-2, -3).to(q.dtype).clone()
        elif self.has_sdp:
            args = [x.contiguous() for x in [q, k, v]]
            v = F.scaled_dot_product_attention(*args, attn_mask=mask)
            return v if mask is None else v.nan_to_num()
        else:
            s = q.shape[-1] ** -0.5
            sim = torch.einsum("...id,...jd->...ij", q, k) * s
            if mask is not None:
                sim.masked_fill(~mask, -float("inf"))
            attn = F.softmax(sim, -1)
            return torch.einsum("...ij,...jd->...id", attn, v)


class SelfBlock(nn.Module):
    def __init__(
        self, embed_dim: int, num_heads: int, flash: bool = False, bias: bool = True
    ) -> None:
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        assert self.embed_dim % num_heads == 0
        self.head_dim = self.embed_dim // num_heads
        self.Wqkv = nn.Linear(embed_dim, 3 * embed_dim, bias=bias)
        self.inner_attn = Attention(flash)
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=bias)
        self.ffn = nn.Sequential(
            nn.Linear(2 * embed_dim, 2 * embed_dim),
            nn.LayerNorm(2 * embed_dim, elementwise_affine=True),
            nn.GELU(),
            nn.Linear(2 * embed_dim, embed_dim),
        )

    def forward(
        self,
        x: torch.Tensor,
        encoding: torch.Tensor,
        mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        qkv = self.Wqkv(x)
        qkv = qkv.unflatten(-1, (self.num_heads, -1, 3)).transpose(1, 2)
        q, k, v = qkv[..., 0], qkv[..., 1], qkv[..., 2]
        q = apply_cached_rotary_emb(encoding, q)
        k = apply_cached_rotary_emb(encoding, k)
        context = self.inner_attn(q, k, v, mask=mask)
        message = self.out_proj(context.transpose(1, 2).flatten(start_dim=-2))
        return x + self.ffn(torch.cat([x, message], -1))


class CrossBlock(nn.Module):
    def __init__(
        self, embed_dim: int, num_heads: int, flash: bool = False, bias: bool = True
    ) -> None:
        super().__init__()
        self.heads = num_heads
        dim_head = embed_dim // num_heads
        self.scale = dim_head**-0.5
        inner_dim = dim_head * num_heads
        self.to_qk = nn.Linear(embed_dim, inner_dim, bias=bias)
        self.to_v = nn.Linear(embed_dim, inner_dim, bias=bias)
        self.to_out = nn.Linear(inner_dim, embed_dim, bias=bias)
        self.ffn = nn.Sequential(
            nn.Linear(2 * embed_dim, 2 * embed_dim),
            nn.LayerNorm(2 * embed_dim, elementwise_affine=True),
            nn.GELU(),
            nn.Linear(2 * embed_dim, embed_dim),
        )
        if flash and FLASH_AVAILABLE:
            self.flash = Attention(True)
        else:
            self.flash = None

    def map_(self, func: Callable, x0: torch.Tensor, x1: torch.Tensor):
        return func(x0), func(x1)

    def forward(
        self, x0: torch.Tensor, x1: torch.Tensor, mask: Optional[torch.Tensor] = None
    ) -> List[torch.Tensor]:
        qk0, qk1 = self.map_(self.to_qk, x0, x1)
        v0, v1 = self.map_(self.to_v, x0, x1)
        qk0, qk1, v0, v1 = map(
            lambda t: t.unflatten(-1, (self.heads, -1)).transpose(1, 2),
            (qk0, qk1, v0, v1),
        )
        if self.flash is not None and qk0.device.type == "cuda":
            m0 = self.flash(qk0, qk1, v1, mask)
            m1 = self.flash(
                qk1, qk0, v0, mask.transpose(-1, -2) if mask is not None else None
            )
        else:
            qk0, qk1 = qk0 * self.scale**0.5, qk1 * self.scale**0.5
            sim = torch.einsum("bhid, bhjd -> bhij", qk0, qk1)
            if mask is not None:
                sim = sim.masked_fill(~mask, -float("inf"))
            attn01 = F.softmax(sim, dim=-1)
            attn10 = F.softmax(sim.transpose(-2, -1).contiguous(), dim=-1)
            m0 = torch.einsum("bhij, bhjd -> bhid", attn01, v1)
            m1 = torch.einsum("bhji, bhjd -> bhid", attn10.transpose(-2, -1), v0)
            if mask is not None:
                m0, m1 = m0.nan_to_num(), m1.nan_to_num()
        m0, m1 = self.map_(lambda t: t.transpose(1, 2).flatten(start_dim=-2), m0, m1)
        m0, m1 = self.map_(self.to_out, m0, m1)
        x0 = x0 + self.ffn(torch.cat([x0, m0], -1))
        x1 = x1 + self.ffn(torch.cat([x1, m1], -1))
        return x0, x1


class TransformerLayer(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__()
        self.self_attn = SelfBlock(*args, **kwargs)
        self.cross_attn = CrossBlock(*args, **kwargs)

    def forward(
        self,
        desc0,
        desc1,
        encoding0,
        encoding1,
        mask0: Optional[torch.Tensor] = None,
        mask1: Optional[torch.Tensor] = None,
    ):
        if mask0 is not None and mask1 is not None:
            return self.masked_forward(desc0, desc1, encoding0, encoding1, mask0, mask1)
        else:
            desc0 = self.self_attn(desc0, encoding0)
            desc1 = self.self_attn(desc1, encoding1)
            return self.cross_attn(desc0, desc1)

    # This part is compiled and allows padding inputs
    def masked_forward(self, desc0, desc1, encoding0, encoding1, mask0, mask1):
        mask = mask0 & mask1.transpose(-1, -2)
        mask0 = mask0 & mask0.transpose(-1, -2)
        mask1 = mask1 & mask1.transpose(-1, -2)
        desc0 = self.self_attn(desc0, encoding0, mask0)
        desc1 = self.self_attn(desc1, encoding1, mask1)
        return self.cross_attn(desc0, desc1, mask)


def sigmoid_log_double_softmax(
    sim: torch.Tensor, z0: torch.Tensor, z1: torch.Tensor
) -> torch.Tensor:
    """create the log assignment matrix from logits and similarity"""
    b, m, n = sim.shape
    certainties = F.logsigmoid(z0) + F.logsigmoid(z1).transpose(1, 2)
    scores0 = F.log_softmax(sim, 2)
    scores1 = F.log_softmax(sim.transpose(-1, -2).contiguous(), 2).transpose(-1, -2)
    scores = sim.new_full((b, m + 1, n + 1), 0)
    scores[:, :m, :n] = scores0 + scores1 + certainties
    scores[:, :-1, -1] = F.logsigmoid(-z0.squeeze(-1))
    scores[:, -1, :-1] = F.logsigmoid(-z1.squeeze(-1))
    return scores


class MatchAssignment(nn.Module):
    def __init__(self, dim: int) -> None:
        super().__init__()
        self.dim = dim
        self.matchability = nn.Linear(dim, 1, bias=True)
        self.final_proj = nn.Linear(dim, dim, bias=True)

    def forward(self, desc0: torch.Tensor, desc1: torch.Tensor):
        """build assignment matrix from descriptors"""
        mdesc0, mdesc1 = self.final_proj(desc0), self.final_proj(desc1)
        _, _, d = mdesc0.shape
        mdesc0, mdesc1 = mdesc0 / d**0.25, mdesc1 / d**0.25
        sim = torch.einsum("bmd,bnd->bmn", mdesc0, mdesc1)
        z0 = self.matchability(desc0)
        z1 = self.matchability(desc1)
        scores = sigmoid_log_double_softmax(sim, z0, z1)
        return scores, sim

    def get_matchability(self, desc: torch.Tensor):
        return torch.sigmoid(self.matchability(desc)).squeeze(-1)


def filter_matches(scores: torch.Tensor, th: float):
    """obtain matches from a log assignment matrix [Bx M+1 x N+1]"""
    max0, max1 = scores[:, :-1, :-1].max(2), scores[:, :-1, :-1].max(1)
    m0, m1 = max0.indices, max1.indices
    indices0 = torch.arange(m0.shape[1], device=m0.device)[None]
    indices1 = torch.arange(m1.shape[1], device=m1.device)[None]
    mutual0 = indices0 == m1.gather(1, m0)
    mutual1 = indices1 == m0.gather(1, m1)
    max0_exp = max0.values.exp()
    zero = max0_exp.new_tensor(0)
    mscores0 = torch.where(mutual0, max0_exp, zero)
    mscores1 = torch.where(mutual1, mscores0.gather(1, m1), zero)
    valid0 = mutual0 & (mscores0 > th)
    valid1 = mutual1 & valid0.gather(1, m1)
    m0 = torch.where(valid0, m0, -1)
    m1 = torch.where(valid1, m1, -1)
    return m0, m1, mscores0, mscores1


class LightGlue(nn.Module):
    default_conf = {
        "name": "lightglue",  # just for interfacing
        "input_dim": 256,  # input descriptor dimension (autoselected from weights)
        "descriptor_dim": 256,
        "add_scale_ori": False,
        "n_layers": 9,
        "num_heads": 4,
        "flash": True,  # enable FlashAttention if available.
        "mp": False,  # enable mixed precision
        "depth_confidence": 0.95,  # early stopping, disable with -1
        "width_confidence": 0.99,  # point pruning, disable with -1
        "filter_threshold": 0.1,  # match threshold
        "weights": None,
    }

    # Point pruning involves an overhead (gather).
    # Therefore, we only activate it if there are enough keypoints.
    pruning_keypoint_thresholds = {
        "cpu": -1,
        "mps": -1,
        "cuda": 1024,
        "flash": 1536,
    }

    required_data_keys = ["image0", "image1"]

    version = "v0.1_arxiv"  # the release whose `aliked_lightglue.pth` this repository pins

    features = {
        "superpoint": {
            "weights": "superpoint_lightglue",
            "input_dim": 256,
        },
        "disk": {
            "weights": "disk_lightglue",
            "input_dim": 128,
        },
        "aliked": {
            "weights": "aliked_lightglue",
            "input_dim": 128,
        },
        "raco-aliked": {
            "weights": "raco_aliked_lightglue",
            "input_dim": 128,
        },
        "sift": {
            "weights": "sift_lightglue",
            "input_dim": 128,
            "add_scale_ori": True,
        },
        "doghardnet": {
            "weights": "doghardnet_lightglue",
            "input_dim": 128,
            "add_scale_ori": True,
        },
    }

    def __init__(self, features="superpoint", **conf) -> None:
        super().__init__()
        self.conf = conf = SimpleNamespace(**{**self.default_conf, **conf})
        if features is not None:
            if features not in self.features:
                raise ValueError(
                    f"Unsupported features: {features} not in "
                    f"{{{','.join(self.features)}}}"
                )
            for k, v in self.features[features].items():
                setattr(conf, k, v)

        if conf.input_dim != conf.descriptor_dim:
            self.input_proj = nn.Linear(conf.input_dim, conf.descriptor_dim, bias=True)
        else:
            self.input_proj = nn.Identity()

        head_dim = conf.descriptor_dim // conf.num_heads
        self.posenc = LearnableFourierPositionalEncoding(
            2 + 2 * self.conf.add_scale_ori, head_dim, head_dim
        )

        h, n, d = conf.num_heads, conf.n_layers, conf.descriptor_dim

        self.transformers = nn.ModuleList(
            [TransformerLayer(d, h, conf.flash) for _ in range(n)]
        )

        self.log_assignment = nn.ModuleList([MatchAssignment(d) for _ in range(n)])
        self.token_confidence = nn.ModuleList(
            [TokenConfidence(d) for _ in range(n - 1)]
        )
        self.register_buffer(
            "confidence_thresholds",
            torch.Tensor(
                [self.confidence_threshold(i) for i in range(self.conf.n_layers)]
            ),
            persistent=False,  # vendored: computed from the configuration, absent from the checkpoint -> strict load
        )

        # vendored: no download and no torch.load at construction; `model.load_components` loads the
        # audited, converted `aliked_lightglue.safetensors` strictly (the release checkpoint already uses the
        # `transformers.{i}.self_attn` key layout, so upstream's rename pass is not needed)

        # static lengths LightGlue is compiled for (only used with torch.compile)
        self.static_lengths = None

    def compile(
        self, mode="reduce-overhead", static_lengths=[256, 512, 768, 1024, 1280, 1536]
    ):
        if self.conf.width_confidence != -1:
            warnings.warn(
                "Point pruning is partially disabled for compiled forward.",
                stacklevel=2,
            )

        torch._inductor.cudagraph_mark_step_begin()
        for i in range(self.conf.n_layers):
            self.transformers[i].masked_forward = torch.compile(
                self.transformers[i].masked_forward, mode=mode, fullgraph=True
            )

        self.static_lengths = static_lengths

    def forward(self, data: dict) -> dict:
        """
        Match keypoints and descriptors between two images

        Input (dict):
            image0: dict
                keypoints: [B x M x 2]
                descriptors: [B x M x D]
                image: [B x C x H x W] or image_size: [B x 2]
            image1: dict
                keypoints: [B x N x 2]
                descriptors: [B x N x D]
                image: [B x C x H x W] or image_size: [B x 2]
        Output (dict):
            matches0: [B x M]
            matching_scores0: [B x M]
            matches1: [B x N]
            matching_scores1: [B x N]
            matches: List[[Si x 2]]
            scores: List[[Si]]
            stop: int
            prune0: [B x M]
            prune1: [B x N]
        """
        with torch.autocast(enabled=self.conf.mp, device_type="cuda"):
            return self._forward(data)

    def _forward(self, data: dict) -> dict:
        for key in self.required_data_keys:
            assert key in data, f"Missing key {key} in data"
        data0, data1 = data["image0"], data["image1"]
        kpts0, kpts1 = data0["keypoints"], data1["keypoints"]
        b, m, _ = kpts0.shape
        b, n, _ = kpts1.shape
        device = kpts0.device
        size0, size1 = data0.get("image_size"), data1.get("image_size")
        kpts0 = normalize_keypoints(kpts0, size0).clone()
        kpts1 = normalize_keypoints(kpts1, size1).clone()

        if self.conf.add_scale_ori:
            kpts0 = torch.cat(
                [kpts0] + [data0[k].unsqueeze(-1) for k in ("scales", "oris")], -1
            )
            kpts1 = torch.cat(
                [kpts1] + [data1[k].unsqueeze(-1) for k in ("scales", "oris")], -1
            )
        desc0 = data0["descriptors"].detach().contiguous()
        desc1 = data1["descriptors"].detach().contiguous()

        assert desc0.shape[-1] == self.conf.input_dim
        assert desc1.shape[-1] == self.conf.input_dim

        if torch.is_autocast_enabled():
            desc0 = desc0.half()
            desc1 = desc1.half()

        mask0, mask1 = None, None
        c = max(m, n)
        do_compile = self.static_lengths and c <= max(self.static_lengths)
        if do_compile:
            kn = min([k for k in self.static_lengths if k >= c])
            desc0, mask0 = pad_to_length(desc0, kn)
            desc1, mask1 = pad_to_length(desc1, kn)
            kpts0, _ = pad_to_length(kpts0, kn)
            kpts1, _ = pad_to_length(kpts1, kn)
        desc0 = self.input_proj(desc0)
        desc1 = self.input_proj(desc1)
        # cache positional embeddings
        encoding0 = self.posenc(kpts0)
        encoding1 = self.posenc(kpts1)

        # GNN + final_proj + assignment
        do_early_stop = self.conf.depth_confidence > 0
        do_point_pruning = self.conf.width_confidence > 0 and not do_compile
        pruning_th = self.pruning_min_kpts(device)
        if do_point_pruning:
            ind0 = torch.arange(0, m, device=device)[None]
            ind1 = torch.arange(0, n, device=device)[None]
            # We store the index of the layer at which pruning is detected.
            prune0 = torch.ones_like(ind0)
            prune1 = torch.ones_like(ind1)
        token0, token1 = None, None
        for i in range(self.conf.n_layers):
            if desc0.shape[1] == 0 or desc1.shape[1] == 0:  # no keypoints
                break
            desc0, desc1 = self.transformers[i](
                desc0, desc1, encoding0, encoding1, mask0=mask0, mask1=mask1
            )
            if i == self.conf.n_layers - 1:
                continue  # no early stopping or adaptive width at last layer

            if do_early_stop:
                token0, token1 = self.token_confidence[i](desc0, desc1)
                if self.check_if_stop(token0[..., :m], token1[..., :n], i, m + n):
                    break
            if do_point_pruning and desc0.shape[-2] > pruning_th:
                scores0 = self.log_assignment[i].get_matchability(desc0)
                prunemask0 = self.get_pruning_mask(token0, scores0, i)
                keep0 = torch.where(prunemask0)[1]
                ind0 = ind0.index_select(1, keep0)
                desc0 = desc0.index_select(1, keep0)
                encoding0 = encoding0.index_select(-2, keep0)
                prune0[:, ind0] += 1
            if do_point_pruning and desc1.shape[-2] > pruning_th:
                scores1 = self.log_assignment[i].get_matchability(desc1)
                prunemask1 = self.get_pruning_mask(token1, scores1, i)
                keep1 = torch.where(prunemask1)[1]
                ind1 = ind1.index_select(1, keep1)
                desc1 = desc1.index_select(1, keep1)
                encoding1 = encoding1.index_select(-2, keep1)
                prune1[:, ind1] += 1

        if desc0.shape[1] == 0 or desc1.shape[1] == 0:  # no keypoints
            m0 = desc0.new_full((b, m), -1, dtype=torch.long)
            m1 = desc1.new_full((b, n), -1, dtype=torch.long)
            mscores0 = desc0.new_zeros((b, m))
            mscores1 = desc1.new_zeros((b, n))
            matches = desc0.new_empty((b, 0, 2), dtype=torch.long)
            mscores = desc0.new_empty((b, 0))
            if not do_point_pruning:
                prune0 = torch.ones_like(mscores0) * self.conf.n_layers
                prune1 = torch.ones_like(mscores1) * self.conf.n_layers
            return {
                "matches0": m0,
                "matches1": m1,
                "matching_scores0": mscores0,
                "matching_scores1": mscores1,
                "stop": i + 1,
                "matches": matches,
                "scores": mscores,
                "prune0": prune0,
                "prune1": prune1,
            }

        desc0, desc1 = desc0[..., :m, :], desc1[..., :n, :]  # remove padding
        scores, _ = self.log_assignment[i](desc0, desc1)
        m0, m1, mscores0, mscores1 = filter_matches(scores, self.conf.filter_threshold)
        matches, mscores = [], []
        for k in range(b):
            valid = m0[k] > -1
            m_indices_0 = torch.where(valid)[0]
            m_indices_1 = m0[k][valid]
            if do_point_pruning:
                m_indices_0 = ind0[k, m_indices_0]
                m_indices_1 = ind1[k, m_indices_1]
            matches.append(torch.stack([m_indices_0, m_indices_1], -1))
            mscores.append(mscores0[k][valid])

        # TODO: Remove when hloc switches to the compact format.
        if do_point_pruning:
            m0_ = torch.full((b, m), -1, device=m0.device, dtype=m0.dtype)
            m1_ = torch.full((b, n), -1, device=m1.device, dtype=m1.dtype)
            m0_[:, ind0] = torch.where(m0 == -1, -1, ind1.gather(1, m0.clamp(min=0)))
            m1_[:, ind1] = torch.where(m1 == -1, -1, ind0.gather(1, m1.clamp(min=0)))
            mscores0_ = torch.zeros((b, m), device=mscores0.device)
            mscores1_ = torch.zeros((b, n), device=mscores1.device)
            mscores0_[:, ind0] = mscores0
            mscores1_[:, ind1] = mscores1
            m0, m1, mscores0, mscores1 = m0_, m1_, mscores0_, mscores1_
        else:
            prune0 = torch.ones_like(mscores0) * self.conf.n_layers
            prune1 = torch.ones_like(mscores1) * self.conf.n_layers

        return {
            "matches0": m0,
            "matches1": m1,
            "matching_scores0": mscores0,
            "matching_scores1": mscores1,
            "stop": i + 1,
            "matches": matches,
            "scores": mscores,
            "prune0": prune0,
            "prune1": prune1,
        }

    def confidence_threshold(self, layer_index: int) -> float:
        """scaled confidence threshold"""
        threshold = 0.8 + 0.1 * np.exp(-4.0 * layer_index / self.conf.n_layers)
        return np.clip(threshold, 0, 1)

    def get_pruning_mask(
        self, confidences: torch.Tensor, scores: torch.Tensor, layer_index: int
    ) -> torch.Tensor:
        """mask points which should be removed"""
        keep = scores > (1 - self.conf.width_confidence)
        if confidences is not None:  # Low-confidence points are never pruned.
            keep |= confidences <= self.confidence_thresholds[layer_index]
        return keep

    def check_if_stop(
        self,
        confidences0: torch.Tensor,
        confidences1: torch.Tensor,
        layer_index: int,
        num_points: int,
    ) -> torch.Tensor:
        """evaluate stopping condition"""
        confidences = torch.cat([confidences0, confidences1], -1)
        threshold = self.confidence_thresholds[layer_index]
        ratio_confident = 1.0 - (confidences < threshold).float().sum() / num_points
        return ratio_confident > self.conf.depth_confidence

    def pruning_min_kpts(self, device: torch.device):
        if self.conf.flash and FLASH_AVAILABLE and device.type == "cuda":
            return self.pruning_keypoint_thresholds["flash"]
        else:
            return self.pruning_keypoint_thresholds[device.type]

**Module 4/7:** `src/lightglue_pipeline/model.py` (carried verbatim; see the note above)

In [ ]:
from __future__ import annotations

import hashlib
import io
import json
import os
import pickletools
import time
import urllib.request
import zipfile
from collections.abc import Callable
from pathlib import Path
from typing import Any

import torch

# standalone rewrite (build_notebook.py): `from .config import (` removed — names are kernel globals defined by the carried modules

MANIFEST_NAME = "dimer-base-manifest.json"
#: Fleet snapshot scheme (DIMER NOTEBOOK_SPEC 1.1 MOD13): the pinned files live in a repository-
#: local snapshot directory named by the model key and described by the committed manifest; a
#: standalone notebook carries that manifest inline and stages/verifies a working-directory copy.
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / DEFAULT_MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory snapshot, no repository checkout

#: The two pinned sources (name -> (url, sha256, bytes, pickle-audit digest)) and the two converted
#: files they become (name -> (sha256, bytes, state tensors)).
SOURCES: dict[str, tuple[str, str, int, str]] = {
    MATCHER_SOURCE_FILENAME: (
        MATCHER_SOURCE_URL,
        MATCHER_SOURCE_SHA256,
        MATCHER_SOURCE_SIZE_BYTES,
        MATCHER_PICKLE_AUDIT_SHA256,
    ),
    EXTRACTOR_SOURCE_FILENAME: (
        EXTRACTOR_SOURCE_URL,
        EXTRACTOR_SOURCE_SHA256,
        EXTRACTOR_SOURCE_SIZE_BYTES,
        EXTRACTOR_PICKLE_AUDIT_SHA256,
    ),
}
CONVERTED: dict[str, tuple[str, int, int]] = {
    MATCHER_FILENAME: (MATCHER_SHA256, MATCHER_SIZE_BYTES, MATCHER_STATE_TENSORS),
    EXTRACTOR_FILENAME: (EXTRACTOR_SHA256, EXTRACTOR_SIZE_BYTES, EXTRACTOR_STATE_TENSORS),
}
SOURCE_OF: dict[str, str] = {
    MATCHER_FILENAME: MATCHER_SOURCE_FILENAME,
    EXTRACTOR_FILENAME: EXTRACTOR_SOURCE_FILENAME,
}


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_checkpoint(
    snapshot_path: str | Path,
    *,
    require_configs: bool = False,
    return_manifest_verified: bool = False,
) -> Path | tuple[Path, bool]:
    """Assert the two converted safetensors files against their pinned byte counts and digests,
    refuse any weight file in an unsafe format other than the two pinned (digest-checked, never
    loaded) sources, and size/digest-check every manifest entry that is present."""
    root = Path(snapshot_path)
    if not root.is_dir():
        raise RuntimeError(f"Checkpoint directory does not exist: {root}")

    for name in CONVERTED_FILENAMES:
        if not (root / name).is_file():
            raise RuntimeError(
                f"Pinned checkpoint is missing {name}; run convert_sources() on the audited "
                f"{SOURCE_OF[name]} first"
            )

    unsafe = sorted(
        p.name
        for p in root.iterdir()
        if p.is_file()
        and p.suffix.lower() in UNSAFE_WEIGHT_EXTENSIONS
        and p.name not in SOURCE_FILENAMES
    )
    if unsafe:
        raise RuntimeError(f"Refusing unsafe weight files: {unsafe}")

    manifest_path = root / MANIFEST_NAME
    manifest_verified = False

    if manifest_path.is_file():
        try:
            manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        except Exception as exc:
            raise RuntimeError(f"Corrupt manifest {MANIFEST_NAME}: {exc}") from exc

        files = manifest.get("files") or []
        if not files:
            raise RuntimeError(f"Manifest {MANIFEST_NAME} contains no files")

        for entry in files:
            rel_path = entry.get("path")
            if not rel_path:
                continue
            target = root / rel_path
            if not target.is_file():
                if rel_path in SOURCE_FILENAMES:
                    continue  # converted-only (DIMER-hosted) shape: the sources are not required
                raise RuntimeError(f"Manifest file missing: {rel_path}")
            exp_bytes = entry.get("bytes")
            if exp_bytes is not None and target.stat().st_size != exp_bytes:
                raise RuntimeError(
                    f"Size mismatch for {rel_path}: {target.stat().st_size} != {exp_bytes}"
                )
            exp_sha = entry.get("sha256")
            if exp_sha is not None and _sha256(target) != exp_sha:
                raise RuntimeError(f"SHA-256 mismatch for {rel_path}")

        manifest_verified = True

    for name, (sha, size_bytes, _tensors) in CONVERTED.items():
        weight_path = root / name
        size = weight_path.stat().st_size
        if size != size_bytes:
            raise RuntimeError(f"Unexpected {name} size: {size}; expected {size_bytes}")
        digest = _sha256(weight_path)
        if digest != sha:
            raise RuntimeError(f"Unexpected {name} SHA-256: {digest}; expected {sha}")

    # The networks' configurations are carried in code (modeling.ALIKED.cfgs and
    # modeling.LightGlue.features); with require_configs there is nothing else to require.

    if return_manifest_verified:
        return root, manifest_verified
    return root


def _read_manifest(root: Path) -> dict[str, Any]:
    """Load and identity-check ``<root>/dimer-base-manifest.json``."""
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    try:
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    except ValueError as exc:
        raise RuntimeError(f"Corrupt manifest {MANIFEST_NAME}: {exc}") from exc
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing"
        )
    if not manifest.get("files"):
        raise RuntimeError(f"Manifest {MANIFEST_NAME} contains no files")
    return manifest


def _check_manifest_files(root: Path, manifest: dict[str, Any]) -> None:
    for entry in manifest["files"]:
        target = root / entry["path"]
        if not target.is_file():
            raise RuntimeError(f"Manifest file missing: {entry['path']}")
        if target.stat().st_size != entry.get("bytes"):
            raise RuntimeError(f"Size mismatch for {entry['path']}")
        if _sha256(target) != entry.get("sha256"):
            raise RuntimeError(f"SHA-256 mismatch for {entry['path']}")


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Manifest-driven verification of a fleet snapshot directory; raise on the first mismatch.

    The identity in the manifest must be the pinned one. Before conversion the two source pickles
    are size- and SHA-256-checked (never unpickled here); once both converted files exist,
    :func:`verify_checkpoint` asserts their pinned digests and byte counts and checks every manifest
    entry still present. Returns ``{"path": ..., **manifest, "converted": bool}``.
    """
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _read_manifest(root)
    converted = all((root / name).is_file() for name in CONVERTED_FILENAMES)
    if not converted:
        _check_manifest_files(root, manifest)
        return {"path": str(root), **manifest, "converted": False}
    _, manifest_verified = verify_checkpoint(
        root, require_configs=True, return_manifest_verified=True
    )
    if not manifest_verified:
        raise RuntimeError(f"manifest at {root} was not verified")  # pragma: no cover
    return {"path": str(root), **manifest, "converted": True}


def _release_download(relative_path: str, root: Path) -> None:
    """Fetch one pinned source from its immutable URL: the LightGlue release asset or the ALIKED
    file at the pinned commit (never a branch). The digest is re-checked by ``verify_snapshot`` and
    again by the static audit before anything is unpickled: a substituted file is caught first."""
    if relative_path not in SOURCES:
        raise ValueError(f"{relative_path} is not a downloadable manifest entry")
    url = SOURCES[relative_path][0]
    root.mkdir(parents=True, exist_ok=True)
    target = root / relative_path
    tmp = target.with_suffix(target.suffix + ".part")
    with urllib.request.urlopen(url, timeout=120) as response, open(tmp, "wb") as fh:  # noqa: S310
        while chunk := response.read(1 << 20):
            fh.write(chunk)
    tmp.replace(target)


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed sources that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). A source is not needed when its converted file is already present.
    Returns the relative paths fetched; :func:`verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _read_manifest(root)
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    for converted_name, source_name in SOURCE_OF.items():
        if (root / converted_name).is_file() and source_name in missing:
            missing.remove(source_name)  # converted-only shape: the source pickle is not needed
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them from the {MODEL_REVISION} release assets"
        )
    fetch = downloader or _release_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def _pickle_globals(data: bytes) -> dict[str, int]:
    """Every global a pickle stream would import, collected with `pickletools.genops` (no
    execution)."""
    found: dict[str, int] = {}
    stack: list[Any] = []
    for op, arg, _pos in pickletools.genops(io.BytesIO(data)):
        if op.name == "GLOBAL":  # pickletools renders the (module, name) pair space-separated
            key = arg.replace("\n", " ").replace(" ", ".", 1)
            found[key] = found.get(key, 0) + 1
        elif op.name == "STACK_GLOBAL":
            key = f"{stack[-2]}.{stack[-1]}"
            found[key] = found.get(key, 0) + 1
        if op.name in ("SHORT_BINUNICODE", "BINUNICODE", "UNICODE", "SHORT_BINSTRING", "BINSTRING"):
            stack.append(arg)
        elif op.name in ("MEMOIZE", "BINPUT", "LONG_BINPUT", "PUT"):
            pass
        else:
            stack.append(None)
    return found


def audit_pickle(
    path: str | Path, *, allowed: frozenset[str] = CKPT_ALLOWED_GLOBALS
) -> dict[str, Any]:
    """Statically list the globals a pickle (plain, or inside a torch zip archive) would import and
    refuse any outside `allowed`. Executes nothing. Returns the sorted globals and their digest."""
    file_path = Path(path)
    if not file_path.is_file():
        raise FileNotFoundError(f"file not found: {file_path}")
    data = file_path.read_bytes()
    found: dict[str, int] = {}
    nested = 0
    if data[:4] == b"PK\x03\x04":
        archive = zipfile.ZipFile(io.BytesIO(data))
        for name in archive.namelist():
            if name.endswith(".pkl"):
                nested += 1
                for key, count in _pickle_globals(archive.read(name)).items():
                    found[key] = found.get(key, 0) + count
    else:
        found = _pickle_globals(data)
    violations = sorted(name for name in found if name not in allowed)
    summary = {
        "file": file_path.name,
        "torch_archive": data[:4] == b"PK\x03\x04",
        "pickles": nested if nested else 1,
        "globals": sorted(found),
        "violations": violations,
        "audit_sha256": hashlib.sha256("\n".join(sorted(found)).encode("utf-8")).hexdigest(),
    }
    if violations:
        raise ValueError(
            f"{file_path.name}: pickle audit failed, globals outside the allow-list: {violations}"
        )
    return summary


def _check_pinned_source(root: Path, name: str) -> dict[str, Any]:
    _url, sha, size_bytes, audit_sha = SOURCES[name]
    source = root / name
    if not source.is_file():
        raise FileNotFoundError(f"source file not found: {source}")
    size = source.stat().st_size
    if size != size_bytes:
        raise ValueError(f"{name}: size {size} != pinned {size_bytes}")
    digest = _sha256(source)
    if digest != sha:
        raise ValueError(f"{name}: sha256 {digest} != pinned {sha}")
    audit = audit_pickle(source)
    if audit["audit_sha256"] != audit_sha:
        raise ValueError(
            f"{name}: pickle audit digest {audit['audit_sha256']} != pinned {audit_sha}"
        )
    return {"path": name, "bytes": size, "sha256": digest, "audit": audit}


def build_models(
    *,
    max_keypoints: int = MAX_KEYPOINTS,
    detection_threshold: float = DETECTION_THRESHOLD,
    filter_threshold: float = FILTER_THRESHOLD,
) -> tuple[Any, Any]:
    """The vendored ALIKED-N(16) extractor and the ALIKED-feature LightGlue matcher at random
    initialisation, in the inference configuration of this contract (nine layers, no pruning)."""
    pass  # standalone rewrite (build_notebook.py): `from .modeling import ALIKED, LightGlue` removed — names are kernel globals defined by the carried modules

    extractor = ALIKED(
        max_num_keypoints=max_keypoints,
        detection_threshold=detection_threshold,
        nms_radius=NMS_RADIUS,
    )
    matcher = LightGlue(
        features="aliked",
        filter_threshold=filter_threshold,
        depth_confidence=DEPTH_CONFIDENCE,
        width_confidence=WIDTH_CONFIDENCE,
    )
    return extractor, matcher


def convert_sources(path: str | Path | None = None) -> dict[str, Any]:
    """Convert the two pinned pickles into their safetensors files, deterministically, after size,
    digest and static-audit checks: torch's weights-only unpickler, a strict load into the vendored
    network, and the network's own state dict saved (sorted keys, contiguous tensors). Each pickle
    is unpickled exactly once, here. Converted files that already exist are left as they are."""
    from safetensors.torch import save_file

    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    started = time.perf_counter()
    extractor, matcher = build_models()
    report: dict[str, Any] = {"converted": [], "skipped": []}
    for converted_name, module in ((MATCHER_FILENAME, matcher), (EXTRACTOR_FILENAME, extractor)):
        target = root / converted_name
        sha, size_bytes, n_tensors = CONVERTED[converted_name]
        if target.is_file():
            if target.stat().st_size != size_bytes or _sha256(target) != sha:
                raise ValueError(
                    f"{converted_name}: existing file does not match its pinned digest"
                )
            report["skipped"].append(converted_name)
            continue
        source_name = SOURCE_OF[converted_name]
        source = _check_pinned_source(root, source_name)
        state = torch.load(root / source_name, map_location="cpu", weights_only=True)
        if not isinstance(state, dict) or any(
            not isinstance(v, torch.Tensor) for v in state.values()
        ):
            raise ValueError(f"{source_name} did not unpickle to a state dict of tensors")
        if len(state) != n_tensors:
            raise ValueError(
                f"{source_name}: state dict has {len(state)} tensors, expected {n_tensors}"
            )
        module.load_state_dict(state, strict=True)
        canonical = {k: v.contiguous() for k, v in module.state_dict().items()}
        if len(canonical) != n_tensors:
            raise ValueError(
                f"converted state dict has {len(canonical)} tensors; expected {n_tensors}"
            )
        save_file(canonical, str(target), metadata={"format": "pt"})
        size = target.stat().st_size
        digest = _sha256(target)
        if size != size_bytes or digest != sha:
            target.unlink()
            raise ValueError(
                f"{converted_name}: converted file {size} B / {digest} != pinned "
                f"{size_bytes} B / {sha}"
            )
        report["converted"].append(
            {
                "source": {k: v for k, v in source.items() if k != "audit"},
                "audit": source["audit"],
                "state_dict_tensors": len(state),
                "parameters": sum(p.numel() for p in module.parameters()),
                "converted": {"path": converted_name, "bytes": size, "sha256": digest},
            }
        )
    report["seconds"] = round(time.perf_counter() - started, 2)
    return report


def resolve_weights_path(
    weights_path: str | Path | None = None,
    cache_dir: str | Path | None = None,
) -> tuple[Path, str]:
    """Resolve weights path with precedence:

    1. Explicit argument `weights_path` -> 'explicit_path'
    2. Environment variable `LIGHTGLUE_WEIGHTS_DIR` -> 'env_var'
    3. Source checkout convention `weights/lightglue-aliked` -> 'repo_offline'
       (only if pyproject.toml exists at repo root and the directory holds the manifest)

    There is no Hub fallback: the checkpoints are GitHub-hosted files staged by
    :func:`stage_missing_files` into a manifest-described snapshot directory.
    """
    if weights_path is not None:
        return Path(weights_path), "explicit_path"

    env_dir = os.environ.get("LIGHTGLUE_WEIGHTS_DIR")
    if env_dir:
        return Path(env_dir), "env_var"

    repo_root = Path.cwd()  # standalone rewrite (build_notebook.py): no repository checkout to resolve
    if (repo_root / "pyproject.toml").is_file():
        repo_weights = repo_root / "weights" / DEFAULT_MODEL_KEY
        if (repo_weights / MANIFEST_NAME).is_file():
            return repo_weights, "repo_offline"

    raise FileNotFoundError(
        "no weights directory: pass weights_path / weights_dir, set LIGHTGLUE_WEIGHTS_DIR, or run "
        f"from a checkout holding weights/{DEFAULT_MODEL_KEY}/{MANIFEST_NAME}"
    )


_resolve_weights_path = resolve_weights_path


def _resolve_device(device: str | torch.device | None) -> torch.device:
    if device is not None:
        return torch.device(device)
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def load_components(
    *,
    device: str | torch.device | None = None,
    cache_dir: str | Path | None = None,
    weights_path: str | Path | None = None,
    return_metadata: bool = False,
    max_keypoints: int = MAX_KEYPOINTS,
    detection_threshold: float = DETECTION_THRESHOLD,
    filter_threshold: float = FILTER_THRESHOLD,
) -> tuple[Any, Any, torch.device, Path] | tuple[Any, Any, torch.device, Path, dict[str, Any]]:
    """Verify and load the two converted checkpoints into the vendored networks (strict state-dict
    loads from safetensors; no pickle is opened here)."""
    from safetensors.torch import load_file

    candidate_path, source = resolve_weights_path(
        weights_path=weights_path,
        cache_dir=cache_dir,
    )

    verified, manifest_verified = verify_checkpoint(
        candidate_path,
        require_configs=True,
        return_manifest_verified=True,
    )
    target_device = _resolve_device(device)

    extractor, matcher = build_models(
        max_keypoints=max_keypoints,
        detection_threshold=detection_threshold,
        filter_threshold=filter_threshold,
    )
    loaded: dict[str, Any] = {}
    for name, module, n_tensors, n_params in (
        (EXTRACTOR_FILENAME, extractor, EXTRACTOR_STATE_TENSORS, EXTRACTOR_PARAMETER_COUNT),
        (MATCHER_FILENAME, matcher, MATCHER_STATE_TENSORS, MATCHER_PARAMETER_COUNT),
    ):
        state = load_file(str(verified / name))
        if len(state) != n_tensors:
            raise RuntimeError(f"{name}: {len(state)} tensors, expected {n_tensors}")
        module.load_state_dict(state, strict=True)
        count = sum(p.numel() for p in module.parameters())
        if count != n_params:
            raise RuntimeError(f"{name}: network has {count} parameters, expected {n_params}")
        module.eval().to(target_device)
        loaded[name] = {"state_tensors": len(state), "parameters": count}

    weight_file = verified / MATCHER_FILENAME
    metadata: dict[str, Any] = {
        "checkpoint_path": verified,
        "checkpoint_source": source,
        "manifest_verified": manifest_verified,
        "weight_sha256": _sha256(weight_file),
        "weight_size_bytes": weight_file.stat().st_size,
        "extractor_sha256": _sha256(verified / EXTRACTOR_FILENAME),
        "extractor_size_bytes": (verified / EXTRACTOR_FILENAME).stat().st_size,
        "device": str(target_device),
        "state_tensors": loaded[MATCHER_FILENAME]["state_tensors"],
        "parameters": loaded[MATCHER_FILENAME]["parameters"],
        "extractor_state_tensors": loaded[EXTRACTOR_FILENAME]["state_tensors"],
        "extractor_parameters": loaded[EXTRACTOR_FILENAME]["parameters"],
    }

    if return_metadata:
        return extractor, matcher, target_device, verified, metadata
    return extractor, matcher, target_device, verified

**Module 5/7:** `src/lightglue_pipeline/provenance.py` (carried verbatim; see the note above)

In [ ]:
from __future__ import annotations

import json
import platform
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
from typing import Any

# standalone rewrite (build_notebook.py): `from .config import (` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .modeling import UPSTREAM_COMMIT, UPSTREAM_REPOSITORY` removed — names are kernel globals defined by the carried modules

_RUNTIME_PACKAGES = (
    "numpy",
    "pillow",
    "safetensors",
    "torch",
    "torchvision",
)


def _package_version(name: str) -> str | None:
    try:
        return version(name)
    except PackageNotFoundError:
        return None


def build_provenance(
    *,
    pipeline: Any | None = None,
    checkpoint_path: str | Path | None = None,
    include_runtime: bool = True,
) -> dict[str, Any]:
    checkpoint_source = None
    manifest_verified = False
    weight_sha256 = MATCHER_SHA256
    weight_size = MATCHER_SIZE_BYTES
    extractor_sha256 = EXTRACTOR_SHA256
    device_str = None
    resolved_checkpoint_path = None
    adapter = None

    if pipeline is not None:
        if getattr(pipeline, "checkpoint_path", None) is not None:
            resolved_checkpoint_path = str(pipeline.checkpoint_path)
        checkpoint_source = getattr(pipeline, "checkpoint_source", None)
        manifest_verified = bool(getattr(pipeline, "manifest_verified", False))
        if getattr(pipeline, "weight_sha256", None):
            weight_sha256 = pipeline.weight_sha256
        if getattr(pipeline, "weight_size_bytes", None):
            weight_size = pipeline.weight_size_bytes
        if getattr(pipeline, "extractor_sha256", None):
            extractor_sha256 = pipeline.extractor_sha256
        if getattr(pipeline, "device", None) is not None:
            device_str = str(pipeline.device)
        if getattr(pipeline, "adapter", None):
            skip = ("history", "trainable_names")
            adapter = {k: v for k, v in pipeline.adapter.items() if k not in skip}
    if checkpoint_path is not None:
        resolved_checkpoint_path = str(checkpoint_path)

    provenance: dict[str, Any] = {
        "schema_version": 1,
        "model": {
            "id": MODEL_ID,
            "revision": MODEL_REVISION,
            "revision_kind": MODEL_REVISION_KIND,
            "weight_file": MATCHER_FILENAME,
            "weight_sha256": weight_sha256,
            "weight_size_bytes": weight_size,
            "weight_format": "safetensors converted once from the audited release pickle",
            "extractor": {
                "repository": EXTRACTOR_REPOSITORY,
                "commit": EXTRACTOR_COMMIT,
                "weight_file": EXTRACTOR_FILENAME,
                "weight_sha256": extractor_sha256,
                "weight_size_bytes": EXTRACTOR_SIZE_BYTES,
            },
            "checkpoint_source": checkpoint_source,
            "checkpoint_path": resolved_checkpoint_path,
            "manifest_verified": manifest_verified,
            "vendored_code": {"repository": UPSTREAM_REPOSITORY, "commit": UPSTREAM_COMMIT},
        },
        "preprocessing": {
            "rgb": True,
            "resize": "none (the sample pairs are built at 640 px on the long side)",
            "side_range_px": [MIN_SIDE, MAX_SIDE],
            "padding": "inside ALIKED to a multiple of 32; keypoints reported in the input frame",
        },
        "inference": {
            "max_keypoints": MAX_KEYPOINTS,
            "detection_threshold": DETECTION_THRESHOLD,
            "nms_radius": NMS_RADIUS,
            "match_threshold": FILTER_THRESHOLD,
            "depth_confidence": DEPTH_CONFIDENCE,
            "width_confidence": WIDTH_CONFIDENCE,
            "match_confidence_semantics": "assignment_scores_not_calibrated_probability",
            "geometric_verification": (
                "none in the pipeline; the evaluation's RANSAC-DLT is a metric, not a filter"
            ),
            "device": device_str,
        },
        "adapter": adapter,
    }
    if include_runtime:
        provenance["runtime"] = {
            "python": platform.python_version(),
            "implementation": platform.python_implementation(),
            "platform": sys.platform,
            "packages": {name: _package_version(name) for name in _RUNTIME_PACKAGES},
        }
    return provenance


def write_provenance(
    path: str | Path,
    *,
    pipeline: Any | None = None,
    checkpoint_path: str | Path | None = None,
    include_runtime: bool = True,
) -> Path:
    record = build_provenance(
        pipeline=pipeline, checkpoint_path=checkpoint_path, include_runtime=include_runtime
    )
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8")
    return out

**Module 6/7:** `src/lightglue_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Image-pair dataset contract for adapting the matcher: the pinned iNaturalist photograph corpus, seeded
homography pairs with exact references, validation, splitting, BYOD loaders and CSV export.

The images are **real**: 360 CC0-licensed, research-grade iNaturalist photographs of six North American bird
species (the fleet's SigLIP sample; 60 per species, one per observer), pinned here by photo id, byte size and
SHA-256 of the served `medium` JPEG and fetched from the iNaturalist open-data bucket at run time, refused on any
byte-size or SHA-256 mismatch; the repository redistributes none of them, and every record keeps its
observation page and observer login. Each photograph becomes one **pair**: the photograph (long side scaled to
640 px, sides cropped to multiples of 8 — the same photographs and split the fleet's XoFTR row uses) and a copy warped by a seeded homography with
seeded photometric changes, so the reference `H` (image0 → image1) is exact and every returned match has a
reprojection error. Two difficulty tiers alternate per photograph: `easy` (corner jitter up to 6 % of the side,
rotation ±10°, scale 0.9–1.1, mild brightness / contrast / noise) and `hard` (jitter up to 18 %, **rotation ±150°**,
scale 0.6–1.4, strong brightness / contrast / gamma / blur / noise). The hard tier is this row's own: ALIKED
descriptors and LightGlue's positional encoding are not rotation-invariant, and the XoFTR row's ±35° left the
frozen matcher with nothing for an adaptation to move.

A record is ``{id, image0, image1, homography}`` (PIL images or paths, and a 3 × 3 list); `SPECIES` maps the
photograph's species key to its names for provenance.
"""
# ruff: noqa: E501  -- fleet dataset module written at the 110-column fleet width; this repo lints at 100

from __future__ import annotations

import csv
import hashlib
import io
import json
import math
import random
import re
import urllib.request
import zipfile
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

import numpy as np
from PIL import Image, ImageEnhance, ImageFilter

# standalone rewrite (build_notebook.py): `from .config import DIVISIBLE_BY, MAX_SIDE, MIN_SIDE, MODEL_ID` removed — names are kernel globals defined by the carried modules

MAX_IMAGE_SIDE = 4096  # pixels; larger uploads are rejected before any decode-to-tensor work
WORKING_LONG_SIDE = 640  # the pairs are built at the checkpoint's training resolution

CORPUS_NAME = "iNaturalist CC0 bird photographs (six species)"
CORPUS_RELEASE = (
    "iNaturalist open-data bucket, research-grade CC0 photos selected 2026-09-19 (60 per species)"
)
CORPUS_BASE_URL = "https://inaturalist-open-data.s3.amazonaws.com/photos/"
CORPUS_LICENSE = (
    "CC0 1.0 (each photo's own license_code on iNaturalist; observers credited in the records)"
)
CORPUS_BYTES = 39_223_447
DEFAULT_CACHE_DIR = Path("weights") / "inat-birds"
SPECIES: dict[str, tuple[str, str]] = {
    "song_sparrow": ("Melospiza melodia", "Song Sparrow"),
    "chipping_sparrow": ("Spizella passerina", "Chipping Sparrow"),
    "white_throated_sparrow": ("Zonotrichia albicollis", "White-throated Sparrow"),
    "dark_eyed_junco": ("Junco hyemalis", "Dark-eyed Junco"),
    "house_finch": ("Haemorhous mexicanus", "House Finch"),
    "american_goldfinch": ("Spinus tristis", "American Goldfinch"),
}
# (id, species, iNat photo id, iNat observation id, observer login, bytes, sha256 of the served
#  <photo id>/medium.<ext>, ext) — the bucket serves each photo under its original extension
#  (jpg or jpeg); the digest pins the served bytes
SAMPLE_RECORDS: tuple[tuple[str, str, int, int, str, int, str, str], ...] = (
    (
        "song_sparrow-00",
        "song_sparrow",
        129376982,
        79016324,
        "andywilson",
        43427,
        "7a9d9304a82f202e992655ec5f65477cd3d7c1dce03aa89a214c2daa38f9d61d",
        "jpg",
    ),
    (
        "song_sparrow-01",
        "song_sparrow",
        480991086,
        267636534,
        "lyneisfilm",
        162073,
        "11f77ff277dd2703c1000f2c787136ff0c3ca7ffad7017056892fe789d65efec",
        "jpg",
    ),
    (
        "song_sparrow-02",
        "song_sparrow",
        546060381,
        302980489,
        "swpollinators",
        27899,
        "4e70b9519c6e5f7384a4495b91b45465f2b1599f86491d8f9f9b12635a4046f6",
        "jpg",
    ),
    (
        "song_sparrow-03",
        "song_sparrow",
        308625896,
        177450028,
        "radrat",
        70961,
        "1211da4fdb24ae85ef0c6c3e2d03542c430457856aec661fe8f5f2de0027eee5",
        "jpeg",
    ),
    (
        "song_sparrow-04",
        "song_sparrow",
        494793016,
        275349085,
        "k-simpkins",
        58410,
        "255538cf450197257e86ed3d41dc69fb78e594434e9cb338c6314288c6cff26e",
        "jpg",
    ),
    (
        "song_sparrow-05",
        "song_sparrow",
        674054489,
        369029444,
        "ben142",
        220573,
        "1ae24622888d9d449ffd6b5c65cac1b9b14870fd8e12dbed8aa0acc2f5030123",
        "jpg",
    ),
    (
        "song_sparrow-06",
        "song_sparrow",
        339623726,
        193339933,
        "rawcomposition",
        25012,
        "d2cde085277a71886a2bf211a1eec26941a73752375708726a8af29aa4995a8b",
        "jpg",
    ),
    (
        "song_sparrow-07",
        "song_sparrow",
        181658744,
        107953669,
        "gcart043",
        98482,
        "a662a6abb24f42b256fb6e2d6f02c3e128a1053ef534f454461aa5cadcc03fe4",
        "jpeg",
    ),
    (
        "song_sparrow-08",
        "song_sparrow",
        222768957,
        130949329,
        "davidfbird",
        110773,
        "0feee62753f409d0aa365e9aa017ddef436ad70a2673dc847feb3b5386af27ff",
        "jpg",
    ),
    (
        "song_sparrow-09",
        "song_sparrow",
        148994242,
        90171417,
        "glennberry",
        102702,
        "64333977d24957d723a1d26886004f222d810557617685579f72a6b553e508fa",
        "jpg",
    ),
    (
        "song_sparrow-10",
        "song_sparrow",
        637315932,
        349374463,
        "sooji",
        136572,
        "a5cebbc0cc2325d3805c4ac854e103f7ba1c22e3a34fb204f30d58681e865033",
        "jpg",
    ),
    (
        "song_sparrow-11",
        "song_sparrow",
        471146686,
        262252507,
        "jeanpaulboerekamps",
        98601,
        "4301f06b52b8dcf1e137567c32412e384edb71cb90e2e35459a0405b2e56529b",
        "jpg",
    ),
    (
        "song_sparrow-12",
        "song_sparrow",
        640811426,
        351179648,
        "erikschiff",
        107695,
        "27aecce184a485ee888c0e8101cb4a34899b8a185b62dc064f3dc2ec9902182b",
        "jpg",
    ),
    (
        "song_sparrow-13",
        "song_sparrow",
        123859010,
        75689904,
        "w_mark_c",
        190819,
        "6b6057a1c50b83ffeb4d9e34367b3e6a9b355236dbcae9820b509e1484f8fe33",
        "jpg",
    ),
    (
        "song_sparrow-14",
        "song_sparrow",
        108520869,
        67204020,
        "dugald",
        52496,
        "e483364889fb95c540db84b5edf5a2400febf7d2eb62a1e49303b13ad329d1b8",
        "jpg",
    ),
    (
        "song_sparrow-15",
        "song_sparrow",
        63537800,
        40010230,
        "nathanael15",
        51441,
        "5ae55f868e779a4e8ee34f6ad077b40c41f20aa699d967f373ebd659a89137a8",
        "jpg",
    ),
    (
        "song_sparrow-16",
        "song_sparrow",
        435251292,
        244042351,
        "carterdorscht",
        166662,
        "d0cdd9ddcf7202a91ac2c47910a0c230639bce50283e8511fb8311151763233a",
        "jpeg",
    ),
    (
        "song_sparrow-17",
        "song_sparrow",
        120710033,
        73898230,
        "tys_rbg",
        126820,
        "515b3b32b64e88d401990bfd1d8e2c1281443b5d1e763a09e86d3d3664e1df41",
        "jpg",
    ),
    (
        "song_sparrow-18",
        "song_sparrow",
        349361283,
        198243065,
        "sean579",
        42820,
        "5eb0031a8d6066650c66b265fb1724413273e095e4e531054d2817eb75910074",
        "jpg",
    ),
    (
        "song_sparrow-19",
        "song_sparrow",
        393408927,
        222067370,
        "irenemacaulay_",
        101681,
        "22e326807de963352b4307790bd40c5506adb0fab1f9844968edaf4bc5665e1c",
        "jpg",
    ),
    (
        "song_sparrow-20",
        "song_sparrow",
        614258628,
        337847585,
        "joy4birds",
        70552,
        "aa767aa5a74a76cfd985aba5282e589f670a9ce0a8fcdc6a096c0357992dec67",
        "jpg",
    ),
    (
        "song_sparrow-21",
        "song_sparrow",
        608802621,
        335154467,
        "jamesadney",
        112564,
        "6e94bce1b5135f48b0b19b64e21a0c76cff4f9b9a28fcbc2b7f5c8ddda6ef513",
        "jpg",
    ),
    (
        "song_sparrow-22",
        "song_sparrow",
        634100352,
        347744524,
        "zorthesosen",
        214044,
        "6a70dba26dfb3e98a244a9ec9badb812ddaa8b3612e7a36f66aba057bb757ecd",
        "jpg",
    ),
    (
        "song_sparrow-23",
        "song_sparrow",
        50065474,
        31954532,
        "truthseqr",
        82521,
        "4186fbf344e92038358d4338102aa440098bff5f558ab1d197f72fefbcec0b15",
        "jpeg",
    ),
    (
        "song_sparrow-24",
        "song_sparrow",
        8656044,
        6803564,
        "glmory",
        297964,
        "e8cab773436ccfaad4699e5112ef4edb516236d413d6dbf034eea7bf9c88c8f5",
        "jpg",
    ),
    (
        "song_sparrow-25",
        "song_sparrow",
        131051149,
        79988590,
        "funvill",
        72572,
        "0a694b5bb03aee6eb5a3a6132e855d47172b21b84cc4cc7c64367d1a31de7894",
        "jpeg",
    ),
    (
        "song_sparrow-26",
        "song_sparrow",
        12923156,
        9491600,
        "gambolingquail",
        69559,
        "6a92bff4fc76820c6f21e15c5f254385dd976a32264911bfc08417a1c2bf053d",
        "jpg",
    ),
    (
        "song_sparrow-27",
        "song_sparrow",
        12077500,
        8959545,
        "reuvenm",
        74010,
        "3e3c5e94c839f45610ef3ef7ffaf3575f4bfe365fc94454c30bfdb718ca3c593",
        "jpeg",
    ),
    (
        "song_sparrow-28",
        "song_sparrow",
        132730299,
        80955309,
        "steph123456",
        154309,
        "5f8854dd231a302c643b22521b49ed3007203d74b8b8413a23abc79ffe0b0270",
        "jpeg",
    ),
    (
        "song_sparrow-29",
        "song_sparrow",
        124840110,
        76316566,
        "terrimewbornagain",
        81046,
        "2020092b0b67397a78cc2b0df267fb671b3ba18e03fb0e35e664e0784061e3c6",
        "jpeg",
    ),
    (
        "song_sparrow-30",
        "song_sparrow",
        130188079,
        79482300,
        "fake_id",
        180462,
        "1d9ad7b45536b85d46fa0fd9a29d4a8ae828bc223836796a3eb9df1fe8fe4fc4",
        "jpeg",
    ),
    (
        "song_sparrow-31",
        "song_sparrow",
        136405967,
        83075831,
        "tom_lazar",
        178554,
        "f86c3c5899d20ac8fb73afb76d45666c610245636d7285a6d517b0081b9d6cc9",
        "jpeg",
    ),
    (
        "song_sparrow-32",
        "song_sparrow",
        118885385,
        72867352,
        "ellyne",
        105156,
        "aab53fb1e7c12adc697711215778c81983f51af83e20bfffe841b80c1d144673",
        "jpeg",
    ),
    (
        "song_sparrow-33",
        "song_sparrow",
        676467457,
        370293984,
        "sholsenbeck",
        59104,
        "b7c230ca942cedc1c53f42902e1f70bc2d09497e57d2e9f31dfe72a5f6c84d16",
        "jpg",
    ),
    (
        "song_sparrow-34",
        "song_sparrow",
        688316087,
        376461614,
        "doublecritch",
        109416,
        "7f11c69df733f99d6e8663e91d807327f8cc57f404049fdbe95e2fd1ce1066e1",
        "jpg",
    ),
    (
        "song_sparrow-35",
        "song_sparrow",
        691785499,
        378254663,
        "karuquebec",
        144522,
        "9862b079bcf496aa8d9ce9e716e65b03a3d9a70a5c2077a223630cf9dcc6d5cc",
        "jpg",
    ),
    (
        "song_sparrow-36",
        "song_sparrow",
        691561091,
        378140427,
        "vaughnshirey",
        87827,
        "a2c197f91c0543ec7bd3a9aac4867a34417b2eb2444bf3062060f9f1e3580045",
        "jpg",
    ),
    (
        "song_sparrow-37",
        "song_sparrow",
        582182748,
        321880490,
        "dougbrown",
        44795,
        "465ce1214c40efab6afdddfbfd34fc5076e5640d130b0442e8e8ac2731ec8a1f",
        "jpg",
    ),
    (
        "song_sparrow-38",
        "song_sparrow",
        704199014,
        384692464,
        "enspring",
        85269,
        "1aa25fd1ed376b9245127432925832cf9253e5d156f4f7d35298eff9a8002043",
        "jpg",
    ),
    (
        "song_sparrow-39",
        "song_sparrow",
        583754170,
        322666295,
        "rociherrera",
        97743,
        "5e093fc6d59b13bbd7ee0889613cef115ef7bb14432634d0af753a8124951737",
        "jpg",
    ),
    (
        "song_sparrow-40",
        "song_sparrow",
        590270305,
        325961498,
        "damienxw",
        133375,
        "a12a152c94ef9c27e5f6dc557e74bf57f6c9ca3015a5e28ee9c30a27ed0e7e19",
        "jpg",
    ),
    (
        "song_sparrow-41",
        "song_sparrow",
        423534602,
        237940479,
        "don54",
        25229,
        "69b0f0dfb05e8219700e3878c9b7e4a2d43d0a2df3905414630913397a565dea",
        "jpeg",
    ),
    (
        "song_sparrow-42",
        "song_sparrow",
        418537114,
        235351206,
        "memoosborne",
        95540,
        "3432cf5a95939b898989c5a962c66b2d71b92b17195d5e695020a776361af3fa",
        "jpeg",
    ),
    (
        "song_sparrow-43",
        "song_sparrow",
        280724703,
        162358184,
        "shirleymorrison",
        43015,
        "f64af9b3fc4559f251ba0f1e95bf8889bfa629123586532adeb2b81e4c2871b2",
        "jpg",
    ),
    (
        "song_sparrow-44",
        "song_sparrow",
        641009256,
        351278414,
        "robinlanark",
        49640,
        "e4af33fd4a504a876b6e7a111a5024cf4197fc888c90f2039df63e9be13d5f06",
        "jpg",
    ),
    (
        "song_sparrow-45",
        "song_sparrow",
        531314531,
        295151228,
        "steveplumb",
        128710,
        "8806861909c0833b29f43b10e9428e104deaccb99d41b42039b76f7da0ae4600",
        "jpg",
    ),
    (
        "song_sparrow-46",
        "song_sparrow",
        114487462,
        70425556,
        "leahmfulton",
        50091,
        "d0b3c9408b72514f8485b19f61116b94be002be99368db96e126d625a17bd343",
        "jpg",
    ),
    (
        "song_sparrow-47",
        "song_sparrow",
        118804811,
        72823547,
        "heibudas",
        191643,
        "8dbe5a56d5fea83b6c25257aa0f9e552ea971f1e0311f583691d24fc9668556f",
        "jpeg",
    ),
    (
        "song_sparrow-48",
        "song_sparrow",
        118498513,
        72671334,
        "seanwashington1",
        80800,
        "940762ff238d94815922d06051ca00fe4a025279b1da08921a44733084adcdb6",
        "jpeg",
    ),
    (
        "song_sparrow-49",
        "song_sparrow",
        119696805,
        73327300,
        "rambryum",
        68983,
        "65f7007d6b90654b77b7f4ec0557ff2d0f310c572368cb3492dd69f9d082e90d",
        "jpeg",
    ),
    (
        "song_sparrow-50",
        "song_sparrow",
        4251416,
        3670620,
        "swells",
        49271,
        "8c53fa5c02e603227117293720d269c59a13555abd537a83640210805fbd844d",
        "JPG",
    ),
    (
        "song_sparrow-51",
        "song_sparrow",
        303954368,
        174929167,
        "dianeclark6280",
        157478,
        "14446920b7c55472de3b172f417fbbeac312090add7356a3c947cfb16911dc95",
        "png",
    ),
    (
        "song_sparrow-52",
        "song_sparrow",
        392544542,
        221653008,
        "emckenziewhat",
        138683,
        "7fd2f3ad3c835e541a7c720372d662a8f953f88f4d8223864a03e4b0f70a42d3",
        "jpeg",
    ),
    (
        "song_sparrow-53",
        "song_sparrow",
        270614122,
        156495271,
        "radkins21",
        92783,
        "7bf8ceb5944de93cbf01aebe4d464b9acb848569c07fbe6e732b701d309f0b25",
        "jpg",
    ),
    (
        "song_sparrow-54",
        "song_sparrow",
        139634990,
        84920252,
        "raffib128",
        56293,
        "865cf92a61d89cb3420c9d3cbf585a0724df2e314918b05d705804aee356f1a5",
        "jpeg",
    ),
    (
        "song_sparrow-55",
        "song_sparrow",
        25251350,
        16733911,
        "kemper",
        54034,
        "99f19fa1a37d23712538e43f6a82339f1fd37defe283de3baacc17719f05857e",
        "jpeg",
    ),
    (
        "song_sparrow-56",
        "song_sparrow",
        139129879,
        84639025,
        "wewantashrubbery",
        104736,
        "aabbffe279d90ffdf737175cc720bea92d634483b830c90acecf920901419643",
        "jpg",
    ),
    (
        "song_sparrow-57",
        "song_sparrow",
        128242284,
        78367372,
        "stevestevens",
        112226,
        "918dfc8094d0a697beb1dcd04499cfd7b0c4b12f9dac4d915d1566ab5690b29b",
        "jpeg",
    ),
    (
        "song_sparrow-58",
        "song_sparrow",
        478733235,
        266446909,
        "saintaardvark",
        298253,
        "d3c8d3a58b44d28c6af4748e4e4195a9a2af7dc6fe91541429dc44aa3a659d21",
        "jpg",
    ),
    (
        "song_sparrow-59",
        "song_sparrow",
        623006598,
        342147975,
        "ryman56",
        60213,
        "73e18df350a69ce00158f03c7ec4fda8672539e0a7292b75f1dfa22dbec7ee30",
        "jpg",
    ),
    (
        "chipping_sparrow-00",
        "chipping_sparrow",
        198992636,
        117809422,
        "k-simpkins",
        62563,
        "cf9f3b0c1863808e21af596b2e609b047ddbc28cb2ed076625e2546425ff0adf",
        "jpg",
    ),
    (
        "chipping_sparrow-01",
        "chipping_sparrow",
        248210057,
        144599194,
        "w_mark_c",
        193389,
        "497d0a0fef81c326bcc87b5d1eb97fe987d559b8f2b71bb60fe422ae34dce150",
        "jpg",
    ),
    (
        "chipping_sparrow-02",
        "chipping_sparrow",
        156350853,
        94266719,
        "ellyne",
        142332,
        "6a60ebac34476a372b4790a87d823432cdda8f72590930b5d18ae166cf4c7ba2",
        "jpeg",
    ),
    (
        "chipping_sparrow-03",
        "chipping_sparrow",
        16128796,
        11327134,
        "reuvenm",
        70532,
        "5ad36c9cdd6c92e225a1b8ab3c04d2f65bc4e971d0243958f8ac090b0996115c",
        "jpeg",
    ),
    (
        "chipping_sparrow-04",
        "chipping_sparrow",
        391300648,
        220982684,
        "carterdorscht",
        208567,
        "c974a676c3c227c2844ff822d429c784bc87bb41f40d5074a0ebadd4c6785b21",
        "jpeg",
    ),
    (
        "chipping_sparrow-05",
        "chipping_sparrow",
        339456083,
        193252042,
        "rawcomposition",
        46329,
        "41c9260ca9107430e3a8090ba01cebf3e3c25f2dad15bea4f77c8e824d9fc496",
        "jpg",
    ),
    (
        "chipping_sparrow-06",
        "chipping_sparrow",
        40457652,
        26076708,
        "andywilson",
        156894,
        "b70edd35e00fe672a39d5f0441ea4e9bb9fac19b0f2415ae13e22160bc6091a6",
        "jpeg",
    ),
    (
        "chipping_sparrow-07",
        "chipping_sparrow",
        84151914,
        52921135,
        "davidfbird",
        145714,
        "ea65ce5ed881ded9a8157b5756fa907948f41e0eefb6735b48a1c43993e977f9",
        "jpeg",
    ),
    (
        "chipping_sparrow-08",
        "chipping_sparrow",
        523674610,
        291074747,
        "rwp84",
        47983,
        "41eac0a5fb578b089f7524c4d2e6cb38c5508aa81815d6cfc46948c081daa800",
        "jpg",
    ),
    (
        "chipping_sparrow-09",
        "chipping_sparrow",
        292695018,
        168861389,
        "tim_kirsten",
        64812,
        "cbb2a03f5dbd2209d56f1cca8b49f342dfbaf44e51fe1014ead75b2018c6678d",
        "jpeg",
    ),
    (
        "chipping_sparrow-10",
        "chipping_sparrow",
        92501489,
        57964052,
        "tniernberger",
        78380,
        "44ddb9923026a97ee77ed362bc944c3d2cbbe4fbd4636000880e81613634e6c5",
        "jpg",
    ),
    (
        "chipping_sparrow-11",
        "chipping_sparrow",
        220257388,
        129611350,
        "gcart043",
        129377,
        "1bdce32e7ac58321dd6beacc084afaf45973469215d66b61a0f7c612da3db361",
        "jpeg",
    ),
    (
        "chipping_sparrow-12",
        "chipping_sparrow",
        478480946,
        266314208,
        "russnamitz",
        61359,
        "8200cc2ec779d47be1b5afa261d934910a251a39f5620d3dc705106fb2bf5e0b",
        "jpeg",
    ),
    (
        "chipping_sparrow-13",
        "chipping_sparrow",
        538480223,
        298914072,
        "hiltonward",
        201271,
        "2a6479556f14a20a8c9a69ff0c1026fe4deb376944234d1d4429c558042c4346",
        "jpg",
    ),
    (
        "chipping_sparrow-14",
        "chipping_sparrow",
        58192408,
        36778771,
        "bradenjudson",
        22352,
        "7c630d7a5b24d94e677e563a8ecb36f57fb29babe6ac1568f11537f10a5edf75",
        "jpeg",
    ),
    (
        "chipping_sparrow-15",
        "chipping_sparrow",
        80410971,
        50636049,
        "radrat",
        55646,
        "0c71ca735502e8c94db81302ecd006428206eb16d75472eb0c706e62e2656667",
        "jpg",
    ),
    (
        "chipping_sparrow-16",
        "chipping_sparrow",
        300376697,
        172991803,
        "matthias55",
        81986,
        "7a9a5cbfb6a7d0581c98c75b5fdeb53a049d60f352eda43a39b1f9c2a3347f3b",
        "jpeg",
    ),
    (
        "chipping_sparrow-17",
        "chipping_sparrow",
        370917657,
        209626382,
        "craigmartin",
        101736,
        "feb94d319fe5b12c01e63a80dc8e44c45e15a0bfb6534ac5d2c0641205dfe1d5",
        "jpg",
    ),
    (
        "chipping_sparrow-18",
        "chipping_sparrow",
        264119989,
        152960401,
        "laurelthrone",
        194107,
        "13e7336628f9ad784757dca82557c79eda22fdce561e0b52a10a503e72979b8e",
        "jpeg",
    ),
    (
        "chipping_sparrow-19",
        "chipping_sparrow",
        220323755,
        129631264,
        "enspring",
        85687,
        "886b7e6faa39943f0c9754e4aef637e2a2f5ad57fe8e8c7ec08360ea95af45b1",
        "jpeg",
    ),
    (
        "chipping_sparrow-20",
        "chipping_sparrow",
        210319996,
        124105091,
        "bunnymom20",
        188328,
        "2f6b6f7ac5ed9d91a361c8fed102484f4fca599ec39c9a2bfe9af1df53b28899",
        "jpeg",
    ),
    (
        "chipping_sparrow-21",
        "chipping_sparrow",
        117013164,
        71832078,
        "terrimewbornagain",
        49527,
        "4456adb52cdafdefdaeb6dd8cec5955f93c1b8f40abdbb1182b2f605a0e1d486",
        "jpeg",
    ),
    (
        "chipping_sparrow-22",
        "chipping_sparrow",
        660951395,
        362143224,
        "mike_cove",
        148079,
        "41d85eaa47b3a41a97ada54728a84c0bfbfed3bcff92de3b6ba88d4c9aface48",
        "jpg",
    ),
    (
        "chipping_sparrow-23",
        "chipping_sparrow",
        691160217,
        377927837,
        "ben142",
        235922,
        "f8137e51e90f0a1964c64fac1a775beaa9ae96c90a1d9aad4f1c870f143e34fd",
        "jpg",
    ),
    (
        "chipping_sparrow-24",
        "chipping_sparrow",
        707633397,
        386482138,
        "leannestacy",
        238800,
        "3fef93babe26eed119ab3b2363902007c46a58a02f91beb0d3bc650821f0dc28",
        "jpg",
    ),
    (
        "chipping_sparrow-25",
        "chipping_sparrow",
        696064427,
        380468188,
        "rdnwoods",
        309547,
        "b56f8f043112021367cbb02861021ac2ef6d2831359315f49d1445bf92477a07",
        "jpg",
    ),
    (
        "chipping_sparrow-26",
        "chipping_sparrow",
        702557361,
        383835541,
        "seanwashington1",
        159095,
        "2940c235b38f84101504b6f50e59c359d0f13e3dc384cadbd43976d1e5e9de91",
        "jpg",
    ),
    (
        "chipping_sparrow-27",
        "chipping_sparrow",
        660337945,
        361814902,
        "kristinpiston",
        158651,
        "a2272e4168edf260001ace410774fbe874ea6b1f6632b8502efc1bc02ae0f866",
        "jpg",
    ),
    (
        "chipping_sparrow-28",
        "chipping_sparrow",
        546647394,
        303298450,
        "kzoebel",
        86457,
        "c37a4aff4263de570ca82f94de92f82d9f9711922e5880547af836104f1883df",
        "jpg",
    ),
    (
        "chipping_sparrow-29",
        "chipping_sparrow",
        713495256,
        389515472,
        "lina47741",
        287986,
        "52e8a37dca8d985a57fdd0d90205e1d813388d961dbf4280d5cbccca9c6af5d3",
        "jpg",
    ),
    (
        "chipping_sparrow-30",
        "chipping_sparrow",
        269951645,
        156136769,
        "conhawn",
        106540,
        "11b48d8bf97084d9ff57aa69517b61539f03b72525022930192330fcb05a160c",
        "jpeg",
    ),
    (
        "chipping_sparrow-31",
        "chipping_sparrow",
        535714979,
        297462154,
        "dissectedfrog",
        200498,
        "cd2f2e1be8c76240a1eeff9535e9a60b8f4fb9d771c7e487d65cde48b7e24cf6",
        "jpg",
    ),
    (
        "chipping_sparrow-32",
        "chipping_sparrow",
        557301829,
        308874987,
        "mar_y_sierra_silvestre",
        115174,
        "ce58e87584df623a86643705b08750228903705ca9b02d93b0206401a24eee9e",
        "jpg",
    ),
    (
        "chipping_sparrow-33",
        "chipping_sparrow",
        652401709,
        357680572,
        "dianeclark6280",
        46782,
        "b77c0eea6899d49f8ab97fd1ad6fc772b6055e5195628fa9f6f989e2d51cedd1",
        "jpg",
    ),
    (
        "chipping_sparrow-34",
        "chipping_sparrow",
        657945260,
        360569392,
        "jasonleduc",
        36363,
        "e3471104742fa68b9eeb149255b0136c53306df23d14f2b2633cea80c9e46eea",
        "jpg",
    ),
    (
        "chipping_sparrow-35",
        "chipping_sparrow",
        434553038,
        243681621,
        "stariplativky",
        57489,
        "e253bd9d70dea9d0438b795f6bebcbc2b26c48b16a90729f0d3155695fa812fd",
        "jpg",
    ),
    (
        "chipping_sparrow-36",
        "chipping_sparrow",
        207498409,
        122551474,
        "askalotl",
        98446,
        "d84e28017be8142c22758e59f14b8c52990e79584a68e6f929123aab4997ac68",
        "jpeg",
    ),
    (
        "chipping_sparrow-37",
        "chipping_sparrow",
        179570776,
        106840941,
        "artemis224",
        138087,
        "74cad843089d2aada7a124f9e704ae1c1419b7cfcade7e2ae29e2c4eda0a6d36",
        "jpg",
    ),
    (
        "chipping_sparrow-38",
        "chipping_sparrow",
        81287617,
        51172999,
        "thyg",
        55659,
        "e1f075addb4bd1feea3ab256e37945a464d2e8c77768b7be131d0655382268d7",
        "jpg",
    ),
    (
        "chipping_sparrow-39",
        "chipping_sparrow",
        343116928,
        195090930,
        "umamimomma",
        52983,
        "a8948a0189d19b3d7b8df65271f4844b14bd5118f510d0b9ef458c942ac41ce2",
        "jpg",
    ),
    (
        "chipping_sparrow-40",
        "chipping_sparrow",
        480169680,
        267207764,
        "cvharris",
        144339,
        "d1ab8643c48f2ea823f9b61843016795def21b290c35ce3d6045e4933b5dfa0a",
        "jpeg",
    ),
    (
        "chipping_sparrow-41",
        "chipping_sparrow",
        352953833,
        200088501,
        "aster-asti",
        105786,
        "82c35e04e4e22fe35c9b364bce7737fa5226cb5c469c761adfc61ef0956da9db",
        "jpg",
    ),
    (
        "chipping_sparrow-42",
        "chipping_sparrow",
        622149696,
        341721858,
        "wilderbombyx",
        293652,
        "227c39a25941a81e9551b5ccef39f95d320e7daa02c96de247123ba62f7b9824",
        "jpg",
    ),
    (
        "chipping_sparrow-43",
        "chipping_sparrow",
        503555630,
        280377982,
        "jtdavis05",
        42352,
        "24ccb474659505cd785502295f385ec369a36c6248296aff85261ed3475a4372",
        "jpeg",
    ),
    (
        "chipping_sparrow-44",
        "chipping_sparrow",
        509135437,
        283351340,
        "martyndrabik",
        125948,
        "1604bf4f5512207c59637a84c78842285c40f76a4cf5335bf6d39afe896ed03c",
        "jpg",
    ),
    (
        "chipping_sparrow-45",
        "chipping_sparrow",
        8076820,
        6402608,
        "msieges",
        166588,
        "6c379f94141a0bcdeb8c9743ea140ded50f8c7864baecd6cef0ee1c724b9fd80",
        "jpeg",
    ),
    (
        "chipping_sparrow-46",
        "chipping_sparrow",
        131917390,
        80486493,
        "tom_lazar",
        137355,
        "10ab0a9c5af2ca2203a18e4e3df799dcf129494d7b637a3c46ffdcf0f3b3856c",
        "jpeg",
    ),
    (
        "chipping_sparrow-47",
        "chipping_sparrow",
        141348323,
        85865218,
        "j-dehoog",
        71473,
        "d3fae9ca7f515c43c216d749f085231d19ddfd101bd37cc4ffae25fc946a34fb",
        "jpeg",
    ),
    (
        "chipping_sparrow-48",
        "chipping_sparrow",
        28879424,
        18850970,
        "andy71",
        194983,
        "ea5af97ddef4f743596c176653c86e457f6bddde65d86f5ab16ddff939574683",
        "jpeg",
    ),
    (
        "chipping_sparrow-49",
        "chipping_sparrow",
        148831027,
        90084486,
        "ian-wolfe",
        181942,
        "cfab51f0c0598120f5312b354ae249bafb15124dd74723a8c29cdd7319206e49",
        "jpg",
    ),
    (
        "chipping_sparrow-50",
        "chipping_sparrow",
        641370647,
        351455126,
        "hrachski",
        130262,
        "9bc25019bf6a71d73daac63b810d18039d360887504014525598cffce2035908",
        "jpg",
    ),
    (
        "chipping_sparrow-51",
        "chipping_sparrow",
        648268506,
        355462611,
        "potatocthulhu",
        193138,
        "f526135f7b71e0f9f8d80c17d8e6c1a3cd89ba068333b9104e66d02ced7a7eaa",
        "jpg",
    ),
    (
        "chipping_sparrow-52",
        "chipping_sparrow",
        645412478,
        353758550,
        "ctlqh",
        154844,
        "e34228baa058ddb18c6aacf193c20574dab67d8c1f4d6940eec6e1acb03d993b",
        "jpg",
    ),
    (
        "chipping_sparrow-53",
        "chipping_sparrow",
        121777317,
        74504842,
        "hickl",
        48576,
        "a7f0980bf777cacb9f61855d157b94d895f6dd629485b2a038ce20519c53856c",
        "jpg",
    ),
    (
        "chipping_sparrow-54",
        "chipping_sparrow",
        147992761,
        89606238,
        "benkeen",
        461732,
        "8b6e8f03650c270624ebf82f85dd0b37db3e693157d4d3d7a1787c855846da9d",
        "png",
    ),
    (
        "chipping_sparrow-55",
        "chipping_sparrow",
        27976676,
        18310312,
        "schoenitz",
        66424,
        "64a0186d15abaf1537a34bd329b67625f7580b7aa544dded6bc626dd389b72fe",
        "jpg",
    ),
    (
        "chipping_sparrow-56",
        "chipping_sparrow",
        190706448,
        112884695,
        "jbeusmans",
        109296,
        "1777945eeb6f916ec6f9c0dc10c81014e8dc0ba22d87268e7c7dc1173193b1e6",
        "jpeg",
    ),
    (
        "chipping_sparrow-57",
        "chipping_sparrow",
        184174664,
        109291439,
        "chrismcv",
        91566,
        "c5d9a1dbaf1fad839680c417c7be7d4a680396e38e7c2aba0ee78fd213f7d32c",
        "jpeg",
    ),
    (
        "chipping_sparrow-58",
        "chipping_sparrow",
        72765715,
        45879059,
        "henryfrye",
        146855,
        "f0b957698f5a600e3a6c45471848013f9c74d1602895e40d513e2116a9b7320a",
        "jpg",
    ),
    (
        "chipping_sparrow-59",
        "chipping_sparrow",
        74100878,
        46721618,
        "aredmer",
        132045,
        "ce6f4d4324d684cfc0f40d6fe9c4a2b03a7024080e7448083ae7e3fc0c06ac6e",
        "jpg",
    ),
    (
        "white_throated_sparrow-00",
        "white_throated_sparrow",
        339621218,
        193338380,
        "rawcomposition",
        31152,
        "d1c08bfaca721bf0873437455b4cc010c6860d08b4777136c775007b0b9d07b6",
        "jpg",
    ),
    (
        "white_throated_sparrow-01",
        "white_throated_sparrow",
        166821399,
        99992799,
        "dziakj1",
        125954,
        "c595a41fbc8948b0d918b59117340dc320e2ba80d29e92bb9dacaaed5b404852",
        "jpeg",
    ),
    (
        "white_throated_sparrow-02",
        "white_throated_sparrow",
        469820434,
        261505977,
        "joy4birds",
        111767,
        "7ce091492c73c68395667bb45578dfff11457511b0958d1d0d320d1df3e55ceb",
        "jpg",
    ),
    (
        "white_throated_sparrow-03",
        "white_throated_sparrow",
        260413022,
        150969515,
        "andywilson",
        45958,
        "fb7c533c92239775da6a5b353ff398f6bdd8f65c13445fc1dd72986af5a46466",
        "jpeg",
    ),
    (
        "white_throated_sparrow-04",
        "white_throated_sparrow",
        99351488,
        62040646,
        "bradenjudson",
        23399,
        "e103968e2a6c9efb6f0bcb548a6457aef950a4a720838a624b6796b90411538e",
        "jpeg",
    ),
    (
        "white_throated_sparrow-05",
        "white_throated_sparrow",
        628148203,
        344731686,
        "lavenderdame",
        106872,
        "36379abf3af51d865ba6e6804ba3dd48fc9efd12ecc0b7d97e03fc04a16178a8",
        "jpg",
    ),
    (
        "white_throated_sparrow-06",
        "white_throated_sparrow",
        104660609,
        65043951,
        "allan7",
        42443,
        "10791891945e83a0c908c14fea07a257a0f25f51c0e708bee35184213833a635",
        "jpeg",
    ),
    (
        "white_throated_sparrow-07",
        "white_throated_sparrow",
        250718938,
        145903421,
        "stevestevens",
        138735,
        "bd52e6d2f48c247f72db0fa393ec4fa13af8510c95f0ca9134cbef71caddb6d4",
        "jpeg",
    ),
    (
        "white_throated_sparrow-08",
        "white_throated_sparrow",
        15105971,
        10793852,
        "schylerbrown",
        31467,
        "6496e7e6d3abf13b1a538f769e3cc402280cf1e9a2a1ebdf81c68dcfd7ed01e2",
        "jpeg",
    ),
    (
        "white_throated_sparrow-09",
        "white_throated_sparrow",
        171460784,
        102554447,
        "w_mark_c",
        251287,
        "3828c41af9209b408fe0d8ec6541edf35d13fc730b74fd27a82980d4f3771137",
        "jpg",
    ),
    (
        "white_throated_sparrow-10",
        "white_throated_sparrow",
        244260317,
        142471526,
        "deejay",
        75694,
        "d67efb1ec61f6700b8a6c6552e2da9cd981e68ae81af16d6f1e0ef17fcaff84f",
        "jpeg",
    ),
    (
        "white_throated_sparrow-11",
        "white_throated_sparrow",
        194732675,
        115373159,
        "wildreturn",
        128989,
        "7ed4bde480b35734576bb4c5f9d1453e77c1f095380432b43b1b682050255831",
        "jpeg",
    ),
    (
        "white_throated_sparrow-12",
        "white_throated_sparrow",
        341990462,
        194541980,
        "ethologist",
        149260,
        "3e2710fac082cc347e6cd114d42d39f25ec47b82c25936922232eba916808cdb",
        "jpg",
    ),
    (
        "white_throated_sparrow-13",
        "white_throated_sparrow",
        267625208,
        154862375,
        "laurelthrone",
        148616,
        "0cac1bc7053c891ce5ae1c33e3f94b69f1b19b958bd08c686db2ccaa7ba0fbd1",
        "jpeg",
    ),
    (
        "white_throated_sparrow-14",
        "white_throated_sparrow",
        330539586,
        188793537,
        "efalquet",
        119214,
        "c3db4583124472b28dbfb4c6fd9b3829e451d508a6b0a095eccc28f17e7982b4",
        "jpeg",
    ),
    (
        "white_throated_sparrow-15",
        "white_throated_sparrow",
        193091403,
        114346779,
        "ian-wolfe",
        73751,
        "e28974c8782fccb00f5ea5420140e01248ead859f151563a95a26bf67acba479",
        "jpg",
    ),
    (
        "white_throated_sparrow-16",
        "white_throated_sparrow",
        691050773,
        377873006,
        "dinomariobob",
        167744,
        "cc5e25afa5041ce2d6ea4e0d726793843f3a867f30b8d9ccf55892d6617da887",
        "jpg",
    ),
    (
        "white_throated_sparrow-17",
        "white_throated_sparrow",
        440986574,
        246671481,
        "suzannehale",
        141857,
        "4a354c189225de2e7b4e94a6df9cbd3a671dac0c8a7d2d4c75e933f93d8b83bf",
        "jpeg",
    ),
    (
        "white_throated_sparrow-18",
        "white_throated_sparrow",
        113217820,
        69733707,
        "kemper",
        97802,
        "4de7da3ac53a086f2c343555be5a2b62a683715db9f80636a76673cc380fa7f6",
        "jpeg",
    ),
    (
        "white_throated_sparrow-19",
        "white_throated_sparrow",
        575307217,
        318327478,
        "k-simpkins",
        90416,
        "af2b4e72a098bd90b920c8a49632e1bbff18b73954d8ac541ca81ed4baf0530b",
        "jpg",
    ),
    (
        "white_throated_sparrow-20",
        "white_throated_sparrow",
        340420076,
        193765555,
        "don54",
        62714,
        "7f6eea434fc700166af1d343951d15f3a9c83eff06cfb0518c3c4291019a5556",
        "jpeg",
    ),
    (
        "white_throated_sparrow-21",
        "white_throated_sparrow",
        562344160,
        311510652,
        "rrfc",
        45842,
        "f7c6394649765db6e3313a03ae013291db1ae20d89d3b8f0bdf6641757f31ed0",
        "jpg",
    ),
    (
        "white_throated_sparrow-22",
        "white_throated_sparrow",
        74598266,
        47039878,
        "ianrwhyte",
        141502,
        "b0633999572a4499a80310955ab928a86f4fe08774e669b4c4949cd26254428b",
        "jpeg",
    ),
    (
        "white_throated_sparrow-23",
        "white_throated_sparrow",
        588001234,
        324825124,
        "portablecity",
        370187,
        "11e3a97f43de5d61266d15028fe9023eef3900cfea2d027bd94ad847ecba9607",
        "jpg",
    ),
    (
        "white_throated_sparrow-24",
        "white_throated_sparrow",
        694103481,
        379467387,
        "memoosborne",
        120656,
        "82684f82f62f6e036e206eec315fff8e1ba36d308b667eba06750b75d1fada3a",
        "jpg",
    ),
    (
        "white_throated_sparrow-25",
        "white_throated_sparrow",
        655749108,
        359408087,
        "russnamitz",
        58449,
        "2a7d2b4226f1941d0950adc04bfd8fe62345f1138c521d0f5f0adf8bc5838da1",
        "jpg",
    ),
    (
        "white_throated_sparrow-26",
        "white_throated_sparrow",
        599110899,
        330395670,
        "jd_flores",
        121343,
        "5eeef4f5533a4b0a0222dcf7be000a2b34f3331f3be2b939703155ba8c8ba3ae",
        "jpg",
    ),
    (
        "white_throated_sparrow-27",
        "white_throated_sparrow",
        653888173,
        358443775,
        "toknowtheland",
        144197,
        "20a4933b00967fe4488296b2b6e89c12ecbcca0e40da591bd95927cca46ae2f7",
        "jpg",
    ),
    (
        "white_throated_sparrow-28",
        "white_throated_sparrow",
        295037357,
        170132018,
        "sturuss",
        117077,
        "d309f75ad5f4008c906ba7b3d6138aa017f919a0a17a1d5b431fa7ab47d5f2d3",
        "jpg",
    ),
    (
        "white_throated_sparrow-29",
        "white_throated_sparrow",
        405042885,
        228270930,
        "carterdorscht",
        122157,
        "f1fb32bcfb78f66f50ccd20f2418832afe72c7a5847bdeb8e4dc19546455cfaf",
        "jpeg",
    ),
    (
        "white_throated_sparrow-30",
        "white_throated_sparrow",
        84971412,
        53425343,
        "davidfbird",
        168442,
        "95c5363704d17d6076f3c7a4c9f5e41d235edf7d5dc9d6022116774f2fd544e6",
        "jpg",
    ),
    (
        "white_throated_sparrow-31",
        "white_throated_sparrow",
        476335868,
        265210970,
        "aster-asti",
        91852,
        "cd300cbc823e8ebc0902b7bd80c99eb323e5900bbabf63303b0d9399624c3e54",
        "jpg",
    ),
    (
        "white_throated_sparrow-32",
        "white_throated_sparrow",
        481170792,
        267732071,
        "kcthetc1",
        46021,
        "2194a78339aae99549df56d2d3d7fc91e0bf98a371e7d0da7958b4312fa3b487",
        "jpeg",
    ),
    (
        "white_throated_sparrow-33",
        "white_throated_sparrow",
        623602892,
        342440773,
        "imperialwoodpecker26",
        112678,
        "66653f47ce6a54d38cc1f54b0a85e965f5c5ee7cce65b02c78c1c185cb586c85",
        "jpg",
    ),
    (
        "white_throated_sparrow-34",
        "white_throated_sparrow",
        622732944,
        342010560,
        "tom_lazar",
        34257,
        "584070c4b5231db49087d1507aabec8f51adc4e09dcdda8b93b3a78159d47146",
        "jpg",
    ),
    (
        "white_throated_sparrow-35",
        "white_throated_sparrow",
        505347090,
        281327144,
        "erikschiff",
        85652,
        "91d50bda6afc3cd5596bf4d3176999bf473c31fb6680b64445ec1cbeaabea847",
        "jpg",
    ),
    (
        "white_throated_sparrow-36",
        "white_throated_sparrow",
        7072170,
        5706496,
        "jmutter",
        171582,
        "de90ea2739e8c65ca89f0e511abaf44219501e18377eb49d9e5c7f10d3a6d5b2",
        "jpeg",
    ),
    (
        "white_throated_sparrow-37",
        "white_throated_sparrow",
        16303076,
        11431253,
        "robw",
        61348,
        "e7d81cfe027c7b0914c79a0431f0ce1431db1ebe78368a2fe5f31e2aab59c3c7",
        "jpg",
    ),
    (
        "white_throated_sparrow-38",
        "white_throated_sparrow",
        7149657,
        5761350,
        "bradleysaul",
        108304,
        "adf755b8393fb6b4db99e788d4112c2c72197c33ecffeec2ca7b3ba5dd9d650e",
        "jpg",
    ),
    (
        "white_throated_sparrow-39",
        "white_throated_sparrow",
        27513220,
        17998912,
        "mefisher",
        49587,
        "f1716d49782f70455ec97ecdda634505b6b2afc375105b1d779353e704dda07b",
        "jpg",
    ),
    (
        "white_throated_sparrow-40",
        "white_throated_sparrow",
        642121459,
        351843522,
        "jeffcherry",
        53348,
        "c9e82d7e47ca2e51c856f9bd4d8f2965d8f69948f081c08c9fa6ebf44fcde711",
        "jpg",
    ),
    (
        "white_throated_sparrow-41",
        "white_throated_sparrow",
        642355950,
        351962427,
        "robinlanark",
        54504,
        "c0da16d5146afac102fa274fffea2e0b203282078ed692f3b7ad4a1340b0d4cd",
        "jpg",
    ),
    (
        "white_throated_sparrow-42",
        "white_throated_sparrow",
        638199267,
        349826931,
        "wildaboutwildlife",
        32981,
        "845f7d38d6f35d2e6e9dca61234c9566778fec1e274500c97e47977fcecd232f",
        "jpg",
    ),
    (
        "white_throated_sparrow-43",
        "white_throated_sparrow",
        119839965,
        73407944,
        "terrimewbornagain",
        61951,
        "dbfbaaa2f72caa812c026122f41512caef00f8f8fdd21c2307fcdf9d79921757",
        "jpeg",
    ),
    (
        "white_throated_sparrow-44",
        "white_throated_sparrow",
        6236370,
        5078910,
        "ruggedbynature",
        126403,
        "3d6030593bf57bc5734fd6c0706c1e350bb117a32370a16b194ef4488668f123",
        "jpeg",
    ),
    (
        "white_throated_sparrow-45",
        "white_throated_sparrow",
        32221828,
        20882923,
        "christinan",
        132485,
        "b92ce41128bb1c5b53a0c487b46ba80eea83b9a443ee9c72f100005e5d80e411",
        "jpeg",
    ),
    (
        "white_throated_sparrow-46",
        "white_throated_sparrow",
        36598789,
        23643589,
        "jessm-c",
        45861,
        "eda75171590418596e372c6cba058364acb64f1279d250dff6c5df61fa17868b",
        "jpeg",
    ),
    (
        "white_throated_sparrow-47",
        "white_throated_sparrow",
        163198257,
        98065830,
        "ellyne",
        109261,
        "864fb42a82561684c1ac22ae71bd023ddeb15c6052b7732ebd2570a98b552a1a",
        "jpeg",
    ),
    (
        "white_throated_sparrow-48",
        "white_throated_sparrow",
        162150240,
        97493574,
        "c_burns802",
        188464,
        "0fb2fb2e04bca82c436315b77a974adc665d2d1fb4e419dd61afa068ab412254",
        "jpeg",
    ),
    (
        "white_throated_sparrow-49",
        "white_throated_sparrow",
        161756778,
        97282492,
        "raffib128",
        67237,
        "77384a65eb120bdb345c3015cb8c22fcec9c89df49b0e0c00954e3476834da9f",
        "jpeg",
    ),
    (
        "white_throated_sparrow-50",
        "white_throated_sparrow",
        165146429,
        99107203,
        "philippthompson",
        84518,
        "8582544ca8b3cb1c73174fc42a9d66971fdae0c90cd4739083eb4ef4983eb6e7",
        "jpeg",
    ),
    (
        "white_throated_sparrow-51",
        "white_throated_sparrow",
        177451999,
        105722646,
        "natepow",
        44983,
        "c81a68c46264d1a5fcae3042e7ab24653c3615665cef6293c31622d5f342d840",
        "jpg",
    ),
    (
        "white_throated_sparrow-52",
        "white_throated_sparrow",
        67630852,
        42591255,
        "schoenitz",
        138616,
        "34594c49a4f601b17e49dfc4d4098db7b73ff5a69d3830b773378e4d695ff181",
        "jpg",
    ),
    (
        "white_throated_sparrow-53",
        "white_throated_sparrow",
        99106006,
        61898962,
        "glennberry",
        122576,
        "fd8cd1c0b0c1fdd7636b8588c86c6d4f3cddc233e9bd4bccade5e2ed275d0e28",
        "jpg",
    ),
    (
        "white_throated_sparrow-54",
        "white_throated_sparrow",
        102162606,
        63634844,
        "eug302",
        73329,
        "692114df14fb8a75fdf38428a196467a4a394e5ded95605fbb48cf64903ccb1b",
        "jpeg",
    ),
    (
        "white_throated_sparrow-55",
        "white_throated_sparrow",
        370403033,
        209307218,
        "kuykenwil",
        124312,
        "d0700068b06ad0f79eeac331a92e5d633bd76be264c3d8801a9a6eb301193969",
        "jpg",
    ),
    (
        "white_throated_sparrow-56",
        "white_throated_sparrow",
        500790892,
        278893388,
        "quillipede",
        186273,
        "0db918123c7f529ca76c04129886eb9aeb947bd3e90cdfd968c4605931c16f85",
        "jpeg",
    ),
    (
        "white_throated_sparrow-57",
        "white_throated_sparrow",
        364738073,
        205369541,
        "sandra1142",
        144424,
        "4cd383751871426904f4d8ff7526a312ba33a6c894faf7121c6ec0ff6ce0d7b1",
        "jpeg",
    ),
    (
        "white_throated_sparrow-58",
        "white_throated_sparrow",
        374502828,
        211884837,
        "halliefromcali",
        62476,
        "0073652fddd7378dc936f280c18d7e44c4a7160aadc2f051e7b5e2896b1a476c",
        "jpg",
    ),
    (
        "white_throated_sparrow-59",
        "white_throated_sparrow",
        109241547,
        67591707,
        "blkillin",
        130154,
        "e5b8c1caa0f7cc7926d46eac42df582ccb0882b4a0733fc0740b8b4597a4e014",
        "jpeg",
    ),
    (
        "dark_eyed_junco-00",
        "dark_eyed_junco",
        172110799,
        102901486,
        "schylerbrown",
        182973,
        "185209c7a1111fc626a068e136ec3cfcdff3174d15f1c60af209d7b32fe7bef9",
        "jpeg",
    ),
    (
        "dark_eyed_junco-01",
        "dark_eyed_junco",
        46691943,
        29901256,
        "haida_gwaii",
        46823,
        "a92dca21e6e58c375fc313f0d1da2c86c11408beb0df8fd96fc60ae035ae80d5",
        "jpg",
    ),
    (
        "dark_eyed_junco-02",
        "dark_eyed_junco",
        707222551,
        386266764,
        "ben142",
        289608,
        "7bf320edf4d34a4848f3e9d175cfafa66cc5bab6a1a8f97c41c670263de3ff89",
        "jpg",
    ),
    (
        "dark_eyed_junco-03",
        "dark_eyed_junco",
        346777340,
        196961623,
        "zacharyfoster",
        46712,
        "ea8f9f0eebb4d2f86193c705344a8fab09bf634bc2217a0874ee57c4f0f5b4ab",
        "jpg",
    ),
    (
        "dark_eyed_junco-04",
        "dark_eyed_junco",
        8793471,
        6892999,
        "truthseqr",
        45302,
        "f8241eab39797c4e097a1432b13658466287d61ed515448457907c17e60cf5e2",
        "jpeg",
    ),
    (
        "dark_eyed_junco-05",
        "dark_eyed_junco",
        192557376,
        114006980,
        "k-simpkins",
        172398,
        "206f012add8a5d0fa434e07c51f1e7bd73a60c4d299a2202163fc662e1b03479",
        "jpeg",
    ),
    (
        "dark_eyed_junco-06",
        "dark_eyed_junco",
        243303909,
        141959574,
        "andywilson",
        71321,
        "ee5308b6f93fb40a6d795a7d8ca6f2ef55b4844513f4268ef34827e1c5e4c4a6",
        "jpeg",
    ),
    (
        "dark_eyed_junco-07",
        "dark_eyed_junco",
        274980085,
        159160633,
        "andy71",
        51209,
        "c54b45ca7fdc635bdb31eb89166c9fce84f0b0f8f0c17331fd5e42b582633c9b",
        "jpeg",
    ),
    (
        "dark_eyed_junco-08",
        "dark_eyed_junco",
        213798376,
        126031618,
        "nathanael15",
        57646,
        "d6b9e7d2dcc63cdd88b47cce328149aa9e8eb486ee2a5fc0f08b90601f2c7d9b",
        "jpg",
    ),
    (
        "dark_eyed_junco-09",
        "dark_eyed_junco",
        63482066,
        39977347,
        "chrisleearm",
        41870,
        "90573e814dbde965c70a934d9702110c40670aa22cc9f80ff8849db877d1fd72",
        "jpeg",
    ),
    (
        "dark_eyed_junco-10",
        "dark_eyed_junco",
        458710472,
        255914048,
        "joy4birds",
        90973,
        "fd25ecb1bf86896b84e9e4e8329c742011011e6883631f47dbe3744ba7463153",
        "jpg",
    ),
    (
        "dark_eyed_junco-11",
        "dark_eyed_junco",
        20784016,
        14046286,
        "gambolingquail",
        93957,
        "a454d6f987153317c9c57c05019b08ab0ebebc46d3cf72356b014231b69a1e9d",
        "jpeg",
    ),
    (
        "dark_eyed_junco-12",
        "dark_eyed_junco",
        332842159,
        189933284,
        "jan-konilu",
        110145,
        "d18706f536164a72a4ec9bacf47437edd89d6b547f86eefdf10ed0166cebf0dd",
        "jpeg",
    ),
    (
        "dark_eyed_junco-13",
        "dark_eyed_junco",
        513004508,
        270404136,
        "thevertebratepokedex",
        141252,
        "854f75e7527932d0409e14756725388de9913c0b450536da717e54d683de169f",
        "jpg",
    ),
    (
        "dark_eyed_junco-14",
        "dark_eyed_junco",
        593499256,
        327583635,
        "orionid",
        108583,
        "7eab4cea6d46faa878732d82c5c2ece63ecc3e66781ea08d0bd950eee581d16d",
        "jpg",
    ),
    (
        "dark_eyed_junco-15",
        "dark_eyed_junco",
        256948665,
        149140989,
        "igor322",
        86925,
        "9339b8cf4aa347b4eb176ef1209dfaa4fcb11e277889cc17de399d6af2c8633a",
        "jpeg",
    ),
    (
        "dark_eyed_junco-16",
        "dark_eyed_junco",
        12533281,
        9255403,
        "braincellsgone",
        40857,
        "be34c51655f59612858d888b05a4f927da65e1aede850c9a5f1be68fa00bfcbe",
        "jpg",
    ),
    (
        "dark_eyed_junco-17",
        "dark_eyed_junco",
        12580989,
        9282523,
        "artemis224",
        216253,
        "15198c123c01fa0f8c03edc086408c8d95ffa553083c380e8ae236dbb7056093",
        "jpg",
    ),
    (
        "dark_eyed_junco-18",
        "dark_eyed_junco",
        63590391,
        40041059,
        "bobbyblackmore",
        82853,
        "87746be0bca30fc40ddba739c1a35ed1fdd41882ebfe3561518b69d285f5ae5d",
        "jpeg",
    ),
    (
        "dark_eyed_junco-19",
        "dark_eyed_junco",
        106718005,
        66230973,
        "vicki936",
        41475,
        "e6903e9a69e987c45edd468ace0bf71adf8252063ff5d39c2130409d8fea68be",
        "jpeg",
    ),
    (
        "dark_eyed_junco-20",
        "dark_eyed_junco",
        611049594,
        336277421,
        "skylar_schell",
        15425,
        "7e876479febb3d44b1e494a25f778efe4e8bfd323279031cb08f9f6fe95f742e",
        "jpg",
    ),
    (
        "dark_eyed_junco-21",
        "dark_eyed_junco",
        591147962,
        326403963,
        "toknowtheland",
        106373,
        "07770f3313dc969802355fa8b3e62a86edb115a64771bd38938e1fab33220c39",
        "jpg",
    ),
    (
        "dark_eyed_junco-22",
        "dark_eyed_junco",
        459696953,
        256398190,
        "w_mark_c",
        262126,
        "5025e757c026e02160ff3f0e82f1f4434b888a6c9689e7ac76274c1fcec64f0c",
        "jpg",
    ),
    (
        "dark_eyed_junco-23",
        "dark_eyed_junco",
        484171775,
        269292238,
        "aschuman",
        59459,
        "89dd933d12cf21c0e73eb01e96a9279e2e57b36676a72a9c860281d0d7703282",
        "jpg",
    ),
    (
        "dark_eyed_junco-24",
        "dark_eyed_junco",
        469746672,
        261469406,
        "shannon_j",
        74167,
        "ec4c7ce15aa1c4eadd3bc5d46d433baf9560d6a0e4593d8d57aab8fa9bec9e18",
        "jpg",
    ),
    (
        "dark_eyed_junco-25",
        "dark_eyed_junco",
        357382685,
        202362003,
        "dougbrown",
        56324,
        "70275d8e07192c99e121b67d206de6823b194f043e65b9766f1f9ce772953c78",
        "jpeg",
    ),
    (
        "dark_eyed_junco-26",
        "dark_eyed_junco",
        7660371,
        6119391,
        "jeffreyleeisanaturalist",
        108394,
        "0f667e9af8e8d3585807da58a2f315c5dbcb7eece5f6862006fb9da68a4bef42",
        "jpg",
    ),
    (
        "dark_eyed_junco-27",
        "dark_eyed_junco",
        437957476,
        245430473,
        "eug302",
        106520,
        "394abfbcf192ecf5a76cb2f19dfa2a62070fa3874b084d1468219d6ee5db6cbf",
        "jpeg",
    ),
    (
        "dark_eyed_junco-28",
        "dark_eyed_junco",
        339439778,
        193239701,
        "rawcomposition",
        32258,
        "73fad57ed975df01c529e77eaca9eab9b00741ea2cfc91358856aefca620a10a",
        "jpg",
    ),
    (
        "dark_eyed_junco-29",
        "dark_eyed_junco",
        634100291,
        347744522,
        "zorthesosen",
        169264,
        "932237001fe68e3d6d19c496fd448b8a82f254316211ac6f9d35708a1395ed81",
        "jpg",
    ),
    (
        "dark_eyed_junco-30",
        "dark_eyed_junco",
        6198359,
        5055484,
        "glmory",
        108168,
        "d31767f42eaaf7f3527133fffb9a0271a9dbbe66fc036d5c694b605563d09965",
        "jpg",
    ),
    (
        "dark_eyed_junco-31",
        "dark_eyed_junco",
        6485959,
        5243663,
        "fake_id",
        148254,
        "ce4dab6c5b37b7ebac07f69d4040d4686fea9fd4b753f91bdf15a3a440820712",
        "jpg",
    ),
    (
        "dark_eyed_junco-32",
        "dark_eyed_junco",
        246318772,
        143610209,
        "wildreturn",
        62090,
        "839e57443659659522ed65dfd054a8fc0a6bfe400518d9f9a7519383d33a2c0d",
        "jpeg",
    ),
    (
        "dark_eyed_junco-33",
        "dark_eyed_junco",
        253199424,
        147174876,
        "sean579",
        121660,
        "6efc17eb4c0b184d64a605a76d3734d9bff5d26e8c8609d2f0b0dd4ccc8b3faa",
        "jpeg",
    ),
    (
        "dark_eyed_junco-34",
        "dark_eyed_junco",
        244315013,
        142500114,
        "enspring",
        63514,
        "1d5012c2b4504b4347fef36d8764eebd1bf07649a14cf874e6d0b3eba9fb3c4f",
        "jpg",
    ),
    (
        "dark_eyed_junco-35",
        "dark_eyed_junco",
        256191686,
        148740024,
        "nartb",
        194532,
        "e88b1449a989fc62c5da1c592e4987fa4db71eadf1ef29c0c5c1159b8f77a298",
        "jpg",
    ),
    (
        "dark_eyed_junco-36",
        "dark_eyed_junco",
        241013731,
        140770209,
        "nana10",
        196479,
        "fc773ca1b06e0e3673aac5c2213c66981ede11bf3ad4a0cdef2102c4f1d99843",
        "jpeg",
    ),
    (
        "dark_eyed_junco-37",
        "dark_eyed_junco",
        240712071,
        140608440,
        "saintaardvark",
        129995,
        "94c4dc30ca31804d6645a6be802cf87ccaffe974f0d3cb554b4fb3e9696ec6a5",
        "jpg",
    ),
    (
        "dark_eyed_junco-38",
        "dark_eyed_junco",
        242909841,
        141750179,
        "calinsdad",
        61773,
        "c49c5d22969552de0fc04ff95eefb8a2ffc76df528477b93c2e2c38cdebe8f6c",
        "jpg",
    ),
    (
        "dark_eyed_junco-39",
        "dark_eyed_junco",
        109385584,
        67667584,
        "ninetoes",
        64185,
        "64e9ee29cf017c292e1711b4c6482488a74c8f652e88e8b7c1df883dc91d75e2",
        "jpg",
    ),
    (
        "dark_eyed_junco-40",
        "dark_eyed_junco",
        102027986,
        63561089,
        "michaelnaumoff",
        114346,
        "eccaa3c60fe575e28fc60f8f6896a99065bff4aee0a4fdc057d7220e7870ca4d",
        "jpg",
    ),
    (
        "dark_eyed_junco-41",
        "dark_eyed_junco",
        105652410,
        65622040,
        "don54",
        159856,
        "5c4282e5e85af7a886eceeadc90977bcd11bf26658176ba5ef8182d2f4a2bfe1",
        "jpeg",
    ),
    (
        "dark_eyed_junco-42",
        "dark_eyed_junco",
        267851501,
        154986181,
        "shanebustapbj",
        203657,
        "217002a366779db0ccc3fddffa71e79472b1fcaa5d444049971dac4c31c910f5",
        "jpeg",
    ),
    (
        "dark_eyed_junco-43",
        "dark_eyed_junco",
        268597047,
        155389350,
        "j-dehoog",
        54916,
        "c0890844944f3d603f8220b0a75213051766c6a45b2ba1fed51381134832f69f",
        "jpeg",
    ),
    (
        "dark_eyed_junco-44",
        "dark_eyed_junco",
        515881506,
        286951393,
        "jubileej",
        108221,
        "9cbecae915f1c96dcc31d5fd77cb5902181152cc3b918693191cc11fce18822c",
        "jpg",
    ),
    (
        "dark_eyed_junco-45",
        "dark_eyed_junco",
        640462423,
        351001488,
        "russnamitz",
        44508,
        "63276eb90201f890b977a54b95bf1d65592eef98089dbb73f2e27e66aaf49a65",
        "jpg",
    ),
    (
        "dark_eyed_junco-46",
        "dark_eyed_junco",
        649025513,
        355899297,
        "rocksand2134",
        233215,
        "2f2545fa5e7901c0383578dea84131eb4be551b8f2bd55d9f308ebc838144f39",
        "jpg",
    ),
    (
        "dark_eyed_junco-47",
        "dark_eyed_junco",
        625101518,
        343192310,
        "jtdavis05",
        106387,
        "4642076ebc5e9bcde85dbba65eb16a49d3fabe6c82cc707b4863ad4b7a4531ae",
        "jpg",
    ),
    (
        "dark_eyed_junco-48",
        "dark_eyed_junco",
        509902416,
        283764048,
        "loarie",
        271588,
        "0c1e3897caca8ff3ff4f3eff7548e30ca50dfe2c2c972106303f9e21c259dbda",
        "jpg",
    ),
    (
        "dark_eyed_junco-49",
        "dark_eyed_junco",
        627787858,
        344552039,
        "margohj",
        54717,
        "a41e557077525b064fa16a7c0b140d608d323663519cfbf424f4454d79951244",
        "jpg",
    ),
    (
        "dark_eyed_junco-50",
        "dark_eyed_junco",
        630444682,
        345899876,
        "robinlanark",
        102578,
        "aa4ffd5dbd553d60ba687d57b21b52ce21572d3c711471abbc5e959bd1126da2",
        "jpg",
    ),
    (
        "dark_eyed_junco-51",
        "dark_eyed_junco",
        635784743,
        348597723,
        "lina47741",
        317439,
        "392d1114b7fc1bb3b041d9895ceb049432a96951764e8949d5939315b96bcc54",
        "jpg",
    ),
    (
        "dark_eyed_junco-52",
        "dark_eyed_junco",
        518407567,
        288278997,
        "zzphantom",
        231733,
        "de2a5be329b583911babe0758c9ce8c0982191f743b8082eb826623256709f71",
        "jpg",
    ),
    (
        "dark_eyed_junco-53",
        "dark_eyed_junco",
        620711072,
        341026776,
        "ethologist",
        186675,
        "13f47961b2a1654ec40f6d1a2005afbf34c75060987fbb662001e66e10a023ed",
        "jpg",
    ),
    (
        "dark_eyed_junco-54",
        "dark_eyed_junco",
        638143588,
        349800424,
        "tom_lazar",
        157629,
        "0be0b84bbc956c616ce35e9e704e8c9e01f0d24db7eeb157fcaa3b3952bd9b23",
        "jpg",
    ),
    (
        "dark_eyed_junco-55",
        "dark_eyed_junco",
        668612910,
        366183670,
        "gendereuphorbia",
        76372,
        "4bfa9f780e9fd0327afec60a8b42e3378b8a9917a0461fd77a6df73b1a94e6bb",
        "jpg",
    ),
    (
        "dark_eyed_junco-56",
        "dark_eyed_junco",
        116078128,
        71308227,
        "msieges",
        47042,
        "ecd4ade5abf058ac887fabe2706eb021ada3b9ab083d10728c9b1dc26de56bbc",
        "jpeg",
    ),
    (
        "dark_eyed_junco-57",
        "dark_eyed_junco",
        117580446,
        72145931,
        "hewwowhy",
        67000,
        "1a8ca3be627c192459440c2eb07bd85b348f0e876a7de32bfc6fe0fb2959a618",
        "jpg",
    ),
    (
        "dark_eyed_junco-58",
        "dark_eyed_junco",
        117044036,
        71848977,
        "radrat",
        81035,
        "7362dd0a27d11b18d5a79abcce9a80304dc80dc0ea1fa84265a1cf0638687641",
        "jpeg",
    ),
    (
        "dark_eyed_junco-59",
        "dark_eyed_junco",
        11110817,
        8359397,
        "giselle9",
        71741,
        "b2e74f27ddca8d1e6dd4807251b07867cbfec8b6bf51d93ef7f142109f705520",
        "jpeg",
    ),
    (
        "house_finch-00",
        "house_finch",
        697940852,
        381438133,
        "ben142",
        196735,
        "a89f8e0263fdabb404b462acaa592f5dd2ac88ee4615da444470de4a1fae82d5",
        "jpg",
    ),
    (
        "house_finch-01",
        "house_finch",
        176982307,
        105476125,
        "vicki936",
        22211,
        "c377fb361df0324c7a856d9344968886ece3b94bd67188c9325b8d2d284d3a2f",
        "jpeg",
    ),
    (
        "house_finch-02",
        "house_finch",
        117990649,
        72375345,
        "kristen163",
        75945,
        "eeafad0dd2e91ecfe45c9d1f27dd0392a01bd81099549c60fd2e36a4b4342a9f",
        "jpeg",
    ),
    (
        "house_finch-03",
        "house_finch",
        389479656,
        220010434,
        "aster-asti",
        82128,
        "a8848197b4e7890e07538d492480c4275b75d04e10c1ae95aee91aaafe3319c5",
        "jpg",
    ),
    (
        "house_finch-04",
        "house_finch",
        98576538,
        61594129,
        "enspring",
        46784,
        "8b355426d8fe6327f202c6a9458cee1b95de445bfbf7e48a4b5a2c7d0eb78a8c",
        "jpg",
    ),
    (
        "house_finch-05",
        "house_finch",
        72470599,
        45698380,
        "henrya",
        61724,
        "1b96d37a7078e1b725b80af4b10848da58b0d0c17a70c8ac01e326c0a749ee6b",
        "jpeg",
    ),
    (
        "house_finch-06",
        "house_finch",
        80751781,
        50842166,
        "leahmfulton",
        44269,
        "6d6202de26f042d83ee6c5af550cd74e6ab10eb796eed2f78c83ac9e2368e4b7",
        "jpg",
    ),
    (
        "house_finch-07",
        "house_finch",
        214612538,
        126483167,
        "hamiltonturner",
        124116,
        "6b7687640c4641b974865da04cf9eaf1f86b774ebc19678f2fc39e55c8648930",
        "jpeg",
    ),
    (
        "house_finch-08",
        "house_finch",
        630196420,
        345777550,
        "truthseqr",
        149455,
        "abb84d1e327dd82c07cbea3dd5583c07453e69b2cc220397b101e597da81bd6c",
        "jpg",
    ),
    (
        "house_finch-09",
        "house_finch",
        213077180,
        125637342,
        "jnicat",
        25823,
        "e1e3baff8d0bd72339e3e49089f7f4f2c1dd383ff005e49a964a1bacc4f8ebb8",
        "jpeg",
    ),
    (
        "house_finch-10",
        "house_finch",
        500744872,
        278868417,
        "pbaff",
        149626,
        "2684cfb1fc40d7766610a5920ead0ad0c27c338ccb4fb9ed18baddda150568b0",
        "jpeg",
    ),
    (
        "house_finch-11",
        "house_finch",
        268678834,
        155431721,
        "stevestevens",
        120325,
        "5d1e8c097c214d14ef7b895bf95769ae9bc25fa799a98bbfa6f12f2f84e6af79",
        "jpeg",
    ),
    (
        "house_finch-12",
        "house_finch",
        358373136,
        202884575,
        "kcthetc1",
        52329,
        "b9929163e4fdadef26c753437ac7af3ad550aa047b15cba131c05d4d335799fe",
        "jpeg",
    ),
    (
        "house_finch-13",
        "house_finch",
        104227663,
        64793290,
        "verdantpulsar",
        101003,
        "90fe37c477bad9ed30ab119e1a7445ffc2720e548a22bb65d66b2ff7b82337d5",
        "jpeg",
    ),
    (
        "house_finch-14",
        "house_finch",
        196156834,
        116218899,
        "kgarrett",
        56801,
        "ce03d1d70a89b5e6e0088f4307f8077573c3571b91997725ca2e6c69be80e2d0",
        "jpeg",
    ),
    (
        "house_finch-15",
        "house_finch",
        454332148,
        253709711,
        "rlaortiz",
        149985,
        "cf007ef8ac57bc0eb085c9eecf9fc95eb69a29df706998de237df24f9491c61d",
        "jpeg",
    ),
    (
        "house_finch-16",
        "house_finch",
        665287910,
        295246528,
        "dinomariobob",
        74972,
        "5f6121c1f8dbfaccbb61c279a579c22600744eb4118c2eeef09e69c86f6e1a49",
        "jpg",
    ),
    (
        "house_finch-17",
        "house_finch",
        509969159,
        283798927,
        "damienxw",
        104277,
        "56d3656be473c362f1ccd09d15e62cbcfe9137d83bef7a3312d72444921c7805",
        "jpg",
    ),
    (
        "house_finch-18",
        "house_finch",
        168402062,
        100871757,
        "k-simpkins",
        86922,
        "209a884cf7618d0b85679ae3a72f237a6d03a5a3096f6ab1b2e5503c38c50d9d",
        "jpg",
    ),
    (
        "house_finch-19",
        "house_finch",
        661042217,
        362189851,
        "dougbrown",
        56426,
        "ad704994fda99779aa340ca1c637b1371f0513df6f36973263bb9ee88cf6713b",
        "jpg",
    ),
    (
        "house_finch-20",
        "house_finch",
        701862058,
        383472210,
        "dblanco",
        25222,
        "8562ed9c4e889b34f83b40c55dd93b821a6b5829a3b1c681374a53c3ca9bf01f",
        "jpg",
    ),
    (
        "house_finch-21",
        "house_finch",
        709783562,
        387605980,
        "fraskar",
        99555,
        "60a1926358c4ec04aeec9414381177600580c8237899c852e03472e53cf87f31",
        "jpg",
    ),
    (
        "house_finch-22",
        "house_finch",
        703924789,
        384540425,
        "meshhy",
        110802,
        "2df38d1aaab2c268c7a47f05e55bef1e83690eeeb91f34874c5b479a5ff0a2cb",
        "jpg",
    ),
    (
        "house_finch-23",
        "house_finch",
        705516095,
        385380385,
        "m_aniket",
        77797,
        "ba009764b66ed89efc9b0aaacd9565830c65039967961488d774ce3c9add5d64",
        "jpg",
    ),
    (
        "house_finch-24",
        "house_finch",
        577527110,
        319485540,
        "glmory",
        94407,
        "6d4cd80f8ef77f41575befae0d8534e9bc397685dceb7f629b5cbc24447a3613",
        "jpg",
    ),
    (
        "house_finch-25",
        "house_finch",
        570686433,
        315887836,
        "emilyheaton",
        229263,
        "27e0bd21c1b322aa5d74c5504dbf27bfb5a0fdeea9718b6ffcee7926f2cb2733",
        "jpg",
    ),
    (
        "house_finch-26",
        "house_finch",
        266259557,
        154114419,
        "andywilson",
        47792,
        "b0dc2de7134cc5af50b1482681b450d32b1f58765990d5d553c4838ae7936db0",
        "jpeg",
    ),
    (
        "house_finch-27",
        "house_finch",
        269729570,
        156011712,
        "kbkash",
        59048,
        "3af407624c46ba4218d24a198ed22b812d4967380bd0e7f67e960f7fbb2defe7",
        "jpeg",
    ),
    (
        "house_finch-28",
        "house_finch",
        270277531,
        156183453,
        "aparrot1",
        83424,
        "93c92a1bb7cf89e7106b0c3df65ce030c82bccade181d661952e940ddcc6282a",
        "jpg",
    ),
    (
        "house_finch-29",
        "house_finch",
        658061500,
        360629741,
        "mike_cove",
        107594,
        "deaaefaf712059b6c8a929a83e1ffd0295e12f0f291bc9869011703eecfcf372",
        "jpg",
    ),
    (
        "house_finch-30",
        "house_finch",
        545518752,
        302693775,
        "cvharris",
        208917,
        "fc5ffdbd30bc76fa4371a99f59543c9137411ddac0a691e2e8122ae7796e038b",
        "jpg",
    ),
    (
        "house_finch-31",
        "house_finch",
        662587006,
        362998939,
        "curran",
        113662,
        "644931afff18c942930717ac3ca563b63c18a0c78ab12ff88557aeede18ead63",
        "jpg",
    ),
    (
        "house_finch-32",
        "house_finch",
        665549166,
        364566546,
        "sealgyu",
        130087,
        "e6fb3e42979ce2709433e8141c4c665a70c4942166153578c7f0173328ffb2ad",
        "jpg",
    ),
    (
        "house_finch-33",
        "house_finch",
        566433127,
        313653086,
        "fake_id",
        150986,
        "cb8a50f48f65795850f57c4d8f84e49debc3deed59b6a60d5bee8ef95299650f",
        "jpg",
    ),
    (
        "house_finch-34",
        "house_finch",
        404592864,
        228025269,
        "logan_artz",
        78971,
        "d0cf631b138d199951d96538903eeb1b56e3b8da3f28c7b042d8868060f27a44",
        "jpeg",
    ),
    (
        "house_finch-35",
        "house_finch",
        423118612,
        237726292,
        "conhawn",
        153347,
        "826ddd6fa8fad5bd1450c0c7beb77fa3261f41a0afa61d7681b7096bc4225c40",
        "jpeg",
    ),
    (
        "house_finch-36",
        "house_finch",
        290080883,
        167451466,
        "vijaybarve",
        89129,
        "f0b32b4af4bdf0bc7eaf298b37a9be6dbcb175d9f72553efc239b6afd2746a69",
        "jpeg",
    ),
    (
        "house_finch-37",
        "house_finch",
        296347840,
        170825854,
        "susanaber",
        96563,
        "47832989fc212f19631e55b63c23155817209c40c66af49e4e03e42863243770",
        "jpeg",
    ),
    (
        "house_finch-38",
        "house_finch",
        414288752,
        233136904,
        "wafflemaster135",
        29548,
        "9f2039a678a69e8f8f383d43f80e3f4e6fcc81d5b99285a0e5b5e54834cf9125",
        "jpeg",
    ),
    (
        "house_finch-39",
        "house_finch",
        284751592,
        164572867,
        "efalquet",
        80806,
        "b50e0a0a4a0bae79b5a8cd034645252044601a29b081f0842443848e802e82d2",
        "jpeg",
    ),
    (
        "house_finch-40",
        "house_finch",
        205914234,
        121669780,
        "darylnolan",
        44763,
        "c04e8d7c002ad0038dcb62843e776ffb8fc86f425c2399442dd1653bbaa4b904",
        "jpeg",
    ),
    (
        "house_finch-41",
        "house_finch",
        96688995,
        60467105,
        "erikschiff",
        145431,
        "dd056b1c0d89dcb9fa06b1c38936142e5c825eb2aeed0d7d2265dc7e147be792",
        "jpg",
    ),
    (
        "house_finch-42",
        "house_finch",
        732358790,
        399184296,
        "katrinamccollough",
        41331,
        "76c899fd9b9aa84ad41888c71a36ab99431fe7201dd417f2c19a66f77388420d",
        "jpg",
    ),
    (
        "house_finch-43",
        "house_finch",
        83824778,
        52720178,
        "beesbirdsbugs",
        48816,
        "72731eb63fa9c5058f9911d11be929af9a58cfea805159d54010e9095948c39a",
        "jpg",
    ),
    (
        "house_finch-44",
        "house_finch",
        349400403,
        198261102,
        "paulgraham",
        105965,
        "b0d1f556ee4926e6a6f87201f6b73955dd9c530788e0ce57d5e09ced74ede001",
        "jpeg",
    ),
    (
        "house_finch-45",
        "house_finch",
        484652323,
        269544283,
        "robinlanark",
        29863,
        "a1d6e8ffbcef10a83a2d3674c7228af89f8d39297f44cd569d1e031da02bc516",
        "jpeg",
    ),
    (
        "house_finch-46",
        "house_finch",
        341357046,
        194221751,
        "haconra",
        80643,
        "8e11937a5c6d5a47da47f562d52d4089d58ad3f2adefadc287b30c562c7ece85",
        "jpg",
    ),
    (
        "house_finch-47",
        "house_finch",
        622140195,
        341714166,
        "rachel141",
        133938,
        "aa776ef9fd3022b6b3db824a762bdde82eeb95518a8764763bd8199683bf1d04",
        "jpg",
    ),
    (
        "house_finch-48",
        "house_finch",
        515309527,
        286632225,
        "solunasilver",
        29507,
        "72a6b495ed99de80504c6b1a86ac1fbfc9e5736bc6f43274bd6093cc91f9ea05",
        "jpg",
    ),
    (
        "house_finch-49",
        "house_finch",
        514877896,
        286416757,
        "jubileej",
        153667,
        "26cd596525a244d2ba82ec2d575834c15671080814400cf7862f614c0db68d07",
        "jpg",
    ),
    (
        "house_finch-50",
        "house_finch",
        125605349,
        76831730,
        "sarahangulo",
        60174,
        "236eb345f1cf1ab4c82fb3b2cc48b948e614f8ddfd958c87a62ce5b44a04f080",
        "jpeg",
    ),
    (
        "house_finch-51",
        "house_finch",
        12881677,
        9467101,
        "artemis224",
        173956,
        "91b5f9f66e8b84cd96169894024ebc3519de6f737d3a32ee3b70100c22b039b9",
        "jpg",
    ),
    (
        "house_finch-52",
        "house_finch",
        124405788,
        76034586,
        "radrat",
        36008,
        "bb9b3fe0b37a1ab6342b1e573f3faf3a287669b70fa9f00b5bfce83de4a376c6",
        "jpeg",
    ),
    (
        "house_finch-53",
        "house_finch",
        17468099,
        12163814,
        "andy71",
        199644,
        "6bc02aaf56c1549dd78d6711f9121d48c33fbd76a8fb623895369f05604ad30d",
        "jpg",
    ),
    (
        "house_finch-54",
        "house_finch",
        134699271,
        82117365,
        "seanwashington1",
        35334,
        "d6e0fe4a8cc690fd6369deff8ea67f4810e1f1297a6f1b9f125aae905742db2a",
        "jpeg",
    ),
    (
        "house_finch-55",
        "house_finch",
        134287410,
        81854554,
        "omcelroy",
        90394,
        "6b4a5ea4c8aa1722332ccdf420af0ca0906cfdb87348e9f6c0904d5a92d8c203",
        "jpg",
    ),
    (
        "house_finch-56",
        "house_finch",
        17763955,
        12337763,
        "jennifer510",
        167368,
        "35355c603049fdf21fc3bf1fed827b50740cfdba134542707026bb4e1f32149b",
        "jpeg",
    ),
    (
        "house_finch-57",
        "house_finch",
        18946521,
        13021113,
        "deejay",
        60576,
        "2420363f6b02433f4b03f3323148ed5e4ac9d37c58a27e859f8e48354db866c8",
        "jpeg",
    ),
    (
        "house_finch-58",
        "house_finch",
        14590841,
        10492098,
        "silvercat",
        120515,
        "23934e0376c70dd6d5ce5e9a24c4176291c877432d25257f07e27694495c879a",
        "jpeg",
    ),
    (
        "house_finch-59",
        "house_finch",
        124702263,
        76228853,
        "janeyair",
        162431,
        "5c86d7cbd28e09d44814735f1e0a21e8266bd4e6a4396b047d092d2f5f3f2111",
        "jpeg",
    ),
    (
        "american_goldfinch-00",
        "american_goldfinch",
        84579952,
        53187208,
        "glennberry",
        59673,
        "72d36079e592e0a83c2f774f9073bfd4cc81253452c925d1673217ddd4b52a36",
        "jpeg",
    ),
    (
        "american_goldfinch-01",
        "american_goldfinch",
        12533322,
        9255418,
        "braincellsgone",
        55661,
        "6344e0125e74791f43ac6e07e5e1b9fbfce6d19bc62b6bb5d83b3caff9f7bcbc",
        "jpg",
    ),
    (
        "american_goldfinch-02",
        "american_goldfinch",
        131102823,
        80016788,
        "radrat",
        98990,
        "232a944f7e3351d4916a12ef2f6d598e7b007caaa95a9b064a14c1ba3af2a6ff",
        "jpeg",
    ),
    (
        "american_goldfinch-03",
        "american_goldfinch",
        175048222,
        104466897,
        "eug302",
        44231,
        "11c723482cc75fcc3a723ac1c0818a68e2fcf74c4ca684cf60195bf0d33f274e",
        "jpg",
    ),
    (
        "american_goldfinch-04",
        "american_goldfinch",
        68849595,
        43390778,
        "mefisher",
        154503,
        "66f07bc59bb3fdedd65a4537ebabd0cafd457826b8bf4bb633181f584a3edfd1",
        "jpg",
    ),
    (
        "american_goldfinch-05",
        "american_goldfinch",
        431916465,
        242278180,
        "k-simpkins",
        45413,
        "d76e7adf33e3a84ebec24dde5438965e62ebec8595755453973846340e4f460d",
        "jpg",
    ),
    (
        "american_goldfinch-06",
        "american_goldfinch",
        230801384,
        135330401,
        "enspring",
        42886,
        "78aebfb9b28c3e16dd9618a0e1ae06df67bfa4b8550c0879427d96915c475fed",
        "jpeg",
    ),
    (
        "american_goldfinch-07",
        "american_goldfinch",
        377136648,
        213398931,
        "nathan1177",
        66516,
        "7ad75838fdf2020a8e426e97507c7dd4355da93e6eece241128c28adbe302438",
        "jpg",
    ),
    (
        "american_goldfinch-08",
        "american_goldfinch",
        294667312,
        169935316,
        "dande",
        163470,
        "39e7892e81eeef6af4887e61bc0998e17797688eb6394fcc8c9438391e875ee0",
        "jpeg",
    ),
    (
        "american_goldfinch-09",
        "american_goldfinch",
        660472044,
        361884286,
        "ben142",
        270599,
        "733d64cd50c61334682f0862f5c7859ded34cae5776ee0bc6594fee400dc4876",
        "jpg",
    ),
    (
        "american_goldfinch-10",
        "american_goldfinch",
        143215717,
        86889530,
        "memoosborne",
        129438,
        "ef4a9a771cef9ba3c2c047eb106a6aa220236dd6aaa6aade5f4ef3a37c8abcba",
        "jpeg",
    ),
    (
        "american_goldfinch-11",
        "american_goldfinch",
        403873164,
        227647158,
        "drew_baxter",
        106009,
        "74e34c776f1b9a5d375a7dfae0e309cd42dbd133b82a1ecad4a49111ac7eed49",
        "jpeg",
    ),
    (
        "american_goldfinch-12",
        "american_goldfinch",
        481153019,
        267722510,
        "vicki936",
        255397,
        "3326ddbee3270241b681cb636466e742491f110e9841f453f5158864aadaea36",
        "jpg",
    ),
    (
        "american_goldfinch-13",
        "american_goldfinch",
        213311122,
        125763980,
        "hickl",
        24740,
        "0c295b3761bced4215519ff24a4f734773b54be98de456a4444d0819456af7dd",
        "jpeg",
    ),
    (
        "american_goldfinch-14",
        "american_goldfinch",
        417352824,
        234736320,
        "joy4birds",
        81109,
        "357014c108519543471b94f39591667d1a67d87fd1fdc4702a3a449235d6bddd",
        "jpeg",
    ),
    (
        "american_goldfinch-15",
        "american_goldfinch",
        72130720,
        45486482,
        "dctphoto",
        178111,
        "17b2f3599e20161acc17fd63bf61e9f40488b9c4d5bfaefc8cae54de42997509",
        "jpeg",
    ),
    (
        "american_goldfinch-16",
        "american_goldfinch",
        45148898,
        28952026,
        "megachile",
        49358,
        "6bbc6ca0ac074e486c20ce4c3fad5dc863cb2e908efc2b1ef97494403a351252",
        "jpeg",
    ),
    (
        "american_goldfinch-17",
        "american_goldfinch",
        133251099,
        81250513,
        "raffib128",
        128110,
        "863c76587fe1de80a84b97c2e72f38ae1cf4fa972789e006d9b0b544d696c6ef",
        "jpeg",
    ),
    (
        "american_goldfinch-18",
        "american_goldfinch",
        155797129,
        93957328,
        "nathanael15",
        74086,
        "7059ae3bdeeb48fe949e5b70b788bd28c96aecd0fe49cb7be22db9a8697bf5a5",
        "jpg",
    ),
    (
        "american_goldfinch-19",
        "american_goldfinch",
        11400438,
        8535447,
        "akneidel",
        26423,
        "4af74c1d04ddc7bbb7bb0e9eb2977a1daf48dd9b9477d71d69a7c4cb5d4f785b",
        "jpg",
    ),
    (
        "american_goldfinch-20",
        "american_goldfinch",
        55990791,
        35505213,
        "conhawn",
        84915,
        "69f768c39a2180440bdcbfc6addc5d426341e8080d4cf9ba8241d57564b3e6fb",
        "jpg",
    ),
    (
        "american_goldfinch-21",
        "american_goldfinch",
        145464774,
        88186208,
        "wildreturn",
        68204,
        "565d2a3b6d404e9ea0c24737ebcc5a3a82057b1452b4790df5e9552e8c50e592",
        "jpg",
    ),
    (
        "american_goldfinch-22",
        "american_goldfinch",
        637247336,
        289067166,
        "dinomariobob",
        142312,
        "01c63d305fce8edcc3a494f0543154c6bd6aa52368e02be84cf6c23e95b94cdd",
        "jpg",
    ),
    (
        "american_goldfinch-23",
        "american_goldfinch",
        460988055,
        257029007,
        "eric112",
        60314,
        "0db1b5f0e32d1faf861a437a20794fa18e80ea8966313933c145451c9319e2a9",
        "jpg",
    ),
    (
        "american_goldfinch-24",
        "american_goldfinch",
        59072170,
        37280587,
        "bradenjudson",
        23468,
        "827d77bf9ca9cc456e867434a07b68cdceb4a20e09eb9e6b67757c80cd31ff42",
        "jpeg",
    ),
    (
        "american_goldfinch-25",
        "american_goldfinch",
        66515260,
        41915252,
        "reuvenm",
        58833,
        "90f0b687d9626fdf5d0111b794cf960cf18f3de4c8eab601fcb23c41b45c4df7",
        "jpeg",
    ),
    (
        "american_goldfinch-26",
        "american_goldfinch",
        109875643,
        67928198,
        "artemis224",
        217909,
        "99e1d7d30eb19d47c7974e9ddd7efe4d06329c952a41e15d0a4b2095b747df61",
        "jpg",
    ),
    (
        "american_goldfinch-27",
        "american_goldfinch",
        219478221,
        129171341,
        "cgmayers",
        96993,
        "93bdee27bb526b60e0edfa518a923b3c10c41ba57221d6a23f4de15bf4a91600",
        "jpeg",
    ),
    (
        "american_goldfinch-28",
        "american_goldfinch",
        99106353,
        61899131,
        "thyg",
        50544,
        "0272bf89b379c2689247f1dbd6f4b36d7f197898baad91d7a8ebdf1c87f88fbd",
        "jpg",
    ),
    (
        "american_goldfinch-29",
        "american_goldfinch",
        223110598,
        131133187,
        "marissa3",
        89214,
        "6a2bb601ac3a70e80e6ccb33d4a24af43128d1e7afe3e92d42e803234bdf532e",
        "jpg",
    ),
    (
        "american_goldfinch-30",
        "american_goldfinch",
        104649681,
        65037800,
        "kemper",
        113821,
        "f4430daf9fca2cf6deb6b988148a5c2e5ae382af8c5db7e5cd37b26e8fbc9461",
        "jpeg",
    ),
    (
        "american_goldfinch-31",
        "american_goldfinch",
        98669936,
        61655828,
        "cloaca_enthusiast",
        31964,
        "3ddcb2e2e6863ef08a40fae7640e05b36ccec9fab9892ac494f095c4588b746f",
        "jpg",
    ),
    (
        "american_goldfinch-32",
        "american_goldfinch",
        347798463,
        197465311,
        "aster-asti",
        116039,
        "e1236dd9850a412ba68386ffd342d180f582db187b83c1d33cb1e09dd51d0043",
        "jpg",
    ),
    (
        "american_goldfinch-33",
        "american_goldfinch",
        461261879,
        257163676,
        "erikschiff",
        40881,
        "eb0b1ea08fcee160edd008ffc386b69b7b50bb89096c44873b978072c581d826",
        "jpg",
    ),
    (
        "american_goldfinch-34",
        "american_goldfinch",
        343726970,
        195390446,
        "robinlanark",
        53508,
        "8b1f4e496755d1a3c4f567c6a48673998b8480ce7d38e7c52bbd94d7ee9901ed",
        "jpeg",
    ),
    (
        "american_goldfinch-35",
        "american_goldfinch",
        352259924,
        199722919,
        "carterdorscht",
        122243,
        "11edfdb53f892b1123df734cb7300e002171baeef21671a31411ff30d156bdc6",
        "jpeg",
    ),
    (
        "american_goldfinch-36",
        "american_goldfinch",
        148829913,
        90084462,
        "ian-wolfe",
        172110,
        "4f6a493313fc596665eb99f8e303ab0ee980b4932a9957993cd00b03b5360fa4",
        "jpg",
    ),
    (
        "american_goldfinch-37",
        "american_goldfinch",
        276388190,
        159958623,
        "dianeclark6280",
        39786,
        "284ca7c9e814fede2c5a15e427a46d7dd82fa6984c4c862c380b0dddff2f86db",
        "jpg",
    ),
    (
        "american_goldfinch-38",
        "american_goldfinch",
        409778763,
        230765968,
        "squidtk",
        210988,
        "b92ccce145f7a320f25edf92633e0c56aa92e8c83890b4a439ce4b3d669cb1a1",
        "jpeg",
    ),
    (
        "american_goldfinch-39",
        "american_goldfinch",
        382773537,
        216432059,
        "rocksand2134",
        195020,
        "ad33d43d9e5650dd441bbd9adb9cc9854a6a1bcc7088a8802a7fb4ccc1ee8b51",
        "jpeg",
    ),
    (
        "american_goldfinch-40",
        "american_goldfinch",
        290287368,
        167563678,
        "sean579",
        105190,
        "8703f245d4b98476f67775ca0c19b7abd7c5edf756c50b2ff502b3d058c08014",
        "jpeg",
    ),
    (
        "american_goldfinch-41",
        "american_goldfinch",
        423112648,
        237722733,
        "andrew2285",
        55719,
        "10709f5df3f4c509afd260ba51c0db5cee89c7c9076c74e9911e9613303d7754",
        "jpeg",
    ),
    (
        "american_goldfinch-42",
        "american_goldfinch",
        422361237,
        237335568,
        "wildaboutwildlife",
        34855,
        "55b0962389b589037e03378472e65ffa0e7ed0fb7e227a7441106a009f093924",
        "jpg",
    ),
    (
        "american_goldfinch-43",
        "american_goldfinch",
        291016386,
        167956837,
        "samallonthesciencemon",
        128110,
        "489934bec3eb5682067cca11cb210c171198e616bb4920753b48bb0e1480bc32",
        "jpg",
    ),
    (
        "american_goldfinch-44",
        "american_goldfinch",
        419250127,
        235728207,
        "paulgraham",
        39211,
        "be1f9ae51a6c4ee92a543ecfe1654698581d7647a3a0b46859a7cf054d8b0fd9",
        "jpeg",
    ),
    (
        "american_goldfinch-45",
        "american_goldfinch",
        416004544,
        234039610,
        "randv",
        105453,
        "942a87723af8aedc86459479437bc474817f19429806f36d8e1236538448e24f",
        "jpeg",
    ),
    (
        "american_goldfinch-46",
        "american_goldfinch",
        416408092,
        234251817,
        "kmayner",
        143385,
        "55259e21e667b88864e2b88ba8dacc8ac5be0d2606699be35c46d2b17fe67fb2",
        "jpeg",
    ),
    (
        "american_goldfinch-47",
        "american_goldfinch",
        116188790,
        71368917,
        "leahmfulton",
        48440,
        "ae01ef8ba2fb714253543a5e773e4e047e95448f4ff4d1e51bc889a20b14e1bc",
        "jpg",
    ),
    (
        "american_goldfinch-48",
        "american_goldfinch",
        118530525,
        72675993,
        "seanwashington1",
        38505,
        "d688edde61a079907cdf5011a32914432f8f7f35c12d355e308aae29b1988738",
        "jpeg",
    ),
    (
        "american_goldfinch-49",
        "american_goldfinch",
        121016576,
        74071097,
        "geenance",
        110180,
        "fb4d1fef09311949043b3c9edc0d3a85d4c20386346093bb6ea39e1b95f756b9",
        "jpeg",
    ),
    (
        "american_goldfinch-50",
        "american_goldfinch",
        3279608,
        2872951,
        "joed",
        115964,
        "c2f1463bfef4c9b157f7e3ebd33fa37570592a4cdfa276f2b7c673f543cbc3ca",
        "jpeg",
    ),
    (
        "american_goldfinch-51",
        "american_goldfinch",
        118140282,
        72458899,
        "melaniegaddy",
        243713,
        "a3d70061032ac8c665f3d17a9118c8e592d53a3669182b23b502e3573a80d8f8",
        "jpg",
    ),
    (
        "american_goldfinch-52",
        "american_goldfinch",
        7669539,
        6125334,
        "cuihenggang",
        61287,
        "36d9570131caf33f3b5cfa6ab99a26c417e7cd8e38d0c1351d1a2938bb6e9bf9",
        "jpeg",
    ),
    (
        "american_goldfinch-53",
        "american_goldfinch",
        9258865,
        7182023,
        "fake_id",
        151987,
        "131d2a52097c9ab18fad318b62e8213c1bd33acddc72065ec0e404c76220c5ee",
        "jpeg",
    ),
    (
        "american_goldfinch-54",
        "american_goldfinch",
        12222432,
        9054511,
        "myacadianforest",
        107758,
        "69b1b5274d74bb28601afc2758e57a88f8d63743b5410c70284a9f4877c721fb",
        "jpeg",
    ),
    (
        "american_goldfinch-55",
        "american_goldfinch",
        128301260,
        78400716,
        "natepow",
        67416,
        "0c988080a47484802806ef14e0d1fdb9205680eacecedbe24f8bbac2aa2873f9",
        "jpg",
    ),
    (
        "american_goldfinch-56",
        "american_goldfinch",
        129842959,
        79296314,
        "nartb",
        117002,
        "f23b2e68f4c10e2679f0a9fe2f34a894bd2deea9c90c2143462e0aa5a83a3daa",
        "jpg",
    ),
    (
        "american_goldfinch-57",
        "american_goldfinch",
        78410307,
        49351195,
        "chrisleearm",
        108066,
        "8a4b273f1a1bd548c48abcf57b86ed35028cbfb949720180a43f24976a26ac8e",
        "jpg",
    ),
    (
        "american_goldfinch-58",
        "american_goldfinch",
        72300928,
        45598176,
        "henrya",
        68593,
        "a0c2869c330aaea08fba3bfeae99392c5588980b1ec21770c220c744125c3b42",
        "jpeg",
    ),
    (
        "american_goldfinch-59",
        "american_goldfinch",
        194724999,
        115367030,
        "vlatassa",
        159105,
        "c456a23238f8f81f93df7919b92dd3fa0123fa16a875c22ec1f9d0ae6ef17324",
        "jpeg",
    ),
)
SAMPLE_SEED = 42
SAMPLE_SPLIT = {"train": 36, "validation": 8, "test": 16}  # photographs per species; 6 species -> 216 / 48 / 96 pairs
TIERS = ("easy", "hard")
TIER_PARAMS: dict[str, dict[str, Any]] = {
    "easy": {"jitter": 0.06, "rotation": 10.0, "scale": (0.9, 1.1), "brightness": (0.85, 1.15), "contrast": (0.85, 1.15), "gamma": (1.0, 1.0), "blur": 0.0, "noise": 4.0},
    "hard": {"jitter": 0.18, "rotation": 150.0, "scale": (0.6, 1.4), "brightness": (0.6, 1.4), "contrast": (0.6, 1.4), "gamma": (0.7, 1.4), "blur": 1.2, "noise": 10.0},
}
MIN_RECORDS = 4
MAX_RECORDS = 5_000
_ID_RE = re.compile(r"^[A-Za-z0-9_.:-]{1,64}$")


def _sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def photo_url(photo_id: int, ext: str = "jpg") -> str:
    """The served object for a pinned photo; `ext` is its recorded original extension (jpg, jpeg or png,
    either case — the bucket key is case-sensitive)."""
    if ext.lower() not in ("jpg", "jpeg", "png"):
        raise ValueError(f"unsupported photo extension {ext!r}")
    return f"{CORPUS_BASE_URL}{photo_id}/medium.{ext}"


def observation_url(observation_id: int) -> str:
    return f"https://www.inaturalist.org/observations/{observation_id}"


def fetch_corpus(*, cache_dir: str | Path | None = None, fetcher: Any = None) -> dict[str, bytes]:
    """Return every pinned photo (bytes keyed by record id) from the cache or the open-data bucket."""
    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    cache.mkdir(parents=True, exist_ok=True)
    out = {}
    for rid, _label, photo_id, _obs, _user, size, digest, ext in SAMPLE_RECORDS:
        local = cache / f"{photo_id}.jpg"
        data = local.read_bytes() if local.is_file() else b""
        if len(data) != size or _sha256_bytes(data) != digest:
            url = photo_url(photo_id, ext)
            if fetcher is not None:
                data = fetcher(url)
            else:
                request = urllib.request.Request(
                    url, headers={"User-Agent": "dimer-lightglue-tutorial/1.0"}
                )
                with urllib.request.urlopen(request, timeout=120) as response:  # noqa: S310 (pinned https URL)
                    data = response.read()
            if len(data) != size or _sha256_bytes(data) != digest:
                raise ValueError(
                    f"{rid} ({photo_id}/medium.{ext}): fetched {len(data)} bytes with sha256 "
                    f"{_sha256_bytes(data)[:16]}…, pinned {size} / {digest[:16]}…"
                )
            local.write_bytes(data)
        out[rid] = data
    return out


def read_corpus(files: Mapping[str, bytes]) -> list[dict[str, Any]]:
    """Decode the verified photo bytes into `{id, image, species}` records with their provenance."""
    out = []
    for rid, label, photo_id, obs_id, user, _size, _digest, _ext in SAMPLE_RECORDS:
        if rid not in files:
            raise ValueError(f"corpus is missing {rid}")
        image = Image.open(io.BytesIO(files[rid]))
        image.load()
        out.append(
            {
                "id": rid,
                "image": image.convert("RGB"),
                "species": label,
                "scientific_name": SPECIES[label][0],
                "common_name": SPECIES[label][1],
                "inat_photo_id": photo_id,
                "inat_observation_url": observation_url(obs_id),
                "observer": user,
            }
        )
    return out


# --------------------------------------------------------------------------------------------------
# pair synthesis: a seeded homography with an exact reference
# --------------------------------------------------------------------------------------------------


def working_size(size: tuple[int, int], long_side: int = WORKING_LONG_SIDE) -> tuple[int, int]:
    """The size a photograph is brought to: long side `long_side`, both sides multiples of DIVISIBLE_BY."""
    width, height = size
    scale = long_side / max(width, height)
    w = max(DIVISIBLE_BY, int(width * scale) // DIVISIBLE_BY * DIVISIBLE_BY)
    h = max(DIVISIBLE_BY, int(height * scale) // DIVISIBLE_BY * DIVISIBLE_BY)
    return w, h


def prepare_image(image: Image.Image, long_side: int = WORKING_LONG_SIDE) -> Image.Image:
    """Resize to the working size (bicubic, aspect kept up to the multiple-of-8 crop) as RGB."""
    w, h = working_size(image.size, long_side)
    return image.convert("RGB").resize((w, h), Image.BICUBIC)


def sample_homography(size: tuple[int, int], rng: random.Random, params: Mapping[str, Any]) -> np.ndarray:
    """A random homography built from a rotation + scale about the centre followed by corner jitter."""
    width, height = size
    cx, cy = (width - 1) / 2.0, (height - 1) / 2.0
    angle = math.radians(rng.uniform(-params["rotation"], params["rotation"]))
    scale = rng.uniform(*params["scale"])
    cos_a, sin_a = math.cos(angle) * scale, math.sin(angle) * scale
    similarity = np.array(
        [[cos_a, -sin_a, cx - cos_a * cx + sin_a * cy], [sin_a, cos_a, cy - sin_a * cx - cos_a * cy], [0.0, 0.0, 1.0]]
    )
    corners = np.array([[0.0, 0.0], [width - 1.0, 0.0], [width - 1.0, height - 1.0], [0.0, height - 1.0]])
    jitter = params["jitter"] * min(width, height)
    moved = corners + np.array([[rng.uniform(-jitter, jitter), rng.uniform(-jitter, jitter)] for _ in range(4)])
    pass  # standalone rewrite (build_notebook.py): `from .metrics import dlt_homography` removed — names are kernel globals defined by the carried modules

    perspective = dlt_homography(corners, moved)
    if perspective is None:  # degenerate draw (practically impossible); fall back to the similarity alone
        return similarity
    return perspective @ similarity


def _perspective_coefficients(homography: np.ndarray) -> list[float]:
    """PIL's PERSPECTIVE transform takes the inverse mapping (output pixel -> input pixel), 8 coefficients."""
    inverse = np.linalg.inv(homography)
    inverse = inverse / inverse[2, 2]
    return [float(v) for v in inverse.ravel()[:8]]


def warp_image(image: Image.Image, homography: np.ndarray) -> Image.Image:
    """image1 = image0 warped by `homography` (image0 coords -> image1 coords), same canvas, black outside."""
    return image.transform(image.size, Image.PERSPECTIVE, _perspective_coefficients(homography), Image.BICUBIC)


def photometric(image: Image.Image, rng: random.Random, params: Mapping[str, Any]) -> Image.Image:
    """Seeded brightness / contrast / gamma / blur / Gaussian-noise changes (never geometric)."""
    out = ImageEnhance.Brightness(image).enhance(rng.uniform(*params["brightness"]))
    out = ImageEnhance.Contrast(out).enhance(rng.uniform(*params["contrast"]))
    gamma = rng.uniform(*params["gamma"])
    if params["blur"] > 0:
        out = out.filter(ImageFilter.GaussianBlur(rng.uniform(0.0, params["blur"])))
    array = np.asarray(out, dtype=np.float64) / 255.0
    if gamma != 1.0:
        array = np.power(np.clip(array, 0.0, 1.0), gamma)
    if params["noise"] > 0:
        noise_rng = np.random.default_rng(rng.getrandbits(32))
        array = array + noise_rng.normal(0.0, params["noise"] / 255.0, array.shape)
    return Image.fromarray((np.clip(array, 0.0, 1.0) * 255.0).round().astype(np.uint8))


def make_pair(image: Image.Image, *, seed: int, tier: str = "hard", record_id: str = "pair") -> dict[str, Any]:
    """One `{id, image0, image1, homography, tier}` record from a photograph and a seed."""
    if tier not in TIER_PARAMS:
        raise ValueError(f"tier must be one of {TIERS}")
    params = TIER_PARAMS[tier]
    rng = random.Random(seed)
    image0 = prepare_image(image)
    homography = sample_homography(image0.size, rng, params)
    image1 = photometric(warp_image(image0, homography), rng, params)
    return {
        "id": record_id,
        "image0": image0,
        "image1": image1,
        "homography": homography.tolist(),
        "tier": tier,
        "seed": seed,
    }


def make_pairs(records: Sequence[Mapping[str, Any]], *, seed: int = SAMPLE_SEED, tier: str | None = None) -> list[dict[str, Any]]:
    """One pair per image record (`{id, image, ...}`); tiers alternate easy / hard unless `tier` is fixed."""
    out = []
    for index, record in enumerate(records):
        chosen = tier or TIERS[index % len(TIERS)]
        pair = make_pair(record["image"], seed=seed * 100_003 + index, tier=chosen, record_id=str(record["id"]))
        for key in ("species", "observer", "inat_photo_id", "inat_observation_url", "source_id"):
            if key in record:
                pair[key] = record[key]
        out.append(pair)
    return out


def build_sample_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """Seeded stratified draw of photographs per species into train / validation / test, then one pair per
    photograph (tiers alternating within each split). No photograph lands in two splits."""
    sizes = dict(sizes or SAMPLE_SPLIT)
    rng = random.Random(seed)
    by_label: dict[str, list[dict[str, Any]]] = {}
    for record in records:
        by_label.setdefault(str(record.get("species", "image")), []).append(dict(record))
    photos: dict[str, list[dict[str, Any]]] = {name: [] for name in sizes}
    for label in sorted(by_label):
        pool = by_label[label]
        rng.shuffle(pool)
        needed = sum(sizes.values())
        if len(pool) < needed:
            raise ValueError(f"{label}: only {len(pool)} records available, need {needed}")
        cursor = 0
        for name, per_class in sizes.items():
            photos[name].extend(pool[cursor : cursor + per_class])
            cursor += per_class
    out: dict[str, list[dict[str, Any]]] = {}
    for offset, name in enumerate(photos):
        rng.shuffle(photos[name])
        relabelled = [{**r, "id": f"{name}-{i:03d}", "source_id": r["id"]} for i, r in enumerate(photos[name])]
        out[name] = make_pairs(relabelled, seed=seed + offset)
    return out


def fetch_sample_dataset(
    *,
    cache_dir: str | Path | None = None,
    fetcher: Any = None,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """The tutorial splits from the pinned corpus."""
    return build_sample_dataset(
        read_corpus(fetch_corpus(cache_dir=cache_dir, fetcher=fetcher)), seed=seed, sizes=sizes
    )


# --------------------------------------------------------------------------------------------------
# validation
# --------------------------------------------------------------------------------------------------


def _open(image: Any, label_name: str) -> Image.Image:
    if isinstance(image, str | Path):
        path = Path(image)
        if not path.is_file():
            raise ValueError(f"{label_name}: image file not found: {path}")
        image = Image.open(path)
        image.load()
    if not isinstance(image, Image.Image):
        raise ValueError(f"{label_name}: image must be a PIL.Image.Image or a file path")
    width, height = image.size
    if width < 1 or height < 1 or max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"{label_name}: image side outside 1..MAX_IMAGE_SIDE={MAX_IMAGE_SIDE} px: {image.size}")
    return image.convert("RGB")


def check_homography(value: Any, label_name: str = "homography") -> np.ndarray:
    array = np.asarray(value, dtype=np.float64)
    if array.shape != (3, 3) or not np.all(np.isfinite(array)):
        raise ValueError(f"{label_name}: must be a finite 3x3 matrix")
    if abs(array[2, 2]) < 1e-12:
        raise ValueError(f"{label_name}: H[2, 2] must be non-zero")
    array = array / array[2, 2]
    if abs(np.linalg.det(array)) < 1e-9:
        raise ValueError(f"{label_name}: matrix is singular")
    return array


def _check_record(record: Any, index: int) -> dict[str, Any]:
    label_name = f"records[{index}]"
    if not isinstance(record, Mapping):
        raise ValueError(f"{label_name} must be a mapping with id/image0/image1/homography")
    for key in ("id", "image0", "image1", "homography"):
        if key not in record:
            raise ValueError(f"{label_name} is missing {key!r}")
    rid = record["id"]
    if not isinstance(rid, str) or not _ID_RE.match(rid):
        raise ValueError(f"{label_name}: id must match {_ID_RE.pattern}")
    image0 = _open(record["image0"], label_name + ".image0")
    image1 = _open(record["image1"], label_name + ".image1")
    for which, image in (("image0", image0), ("image1", image1)):
        if min(image.size) < MIN_SIDE or max(image.size) > MAX_SIDE:
            raise ValueError(
                f"{label_name}.{which}: sides must lie in [{MIN_SIDE}, {MAX_SIDE}] px after preparation; got {image.size}"
            )
    homography = check_homography(record["homography"], label_name + ".homography")
    tier = record.get("tier", "unspecified")
    if not isinstance(tier, str) or not tier:
        raise ValueError(f"{label_name}: tier must be a non-empty string when given")
    item = {"id": rid, "image0": image0, "image1": image1, "homography": homography.tolist(), "tier": tier}
    for key in ("source_id", "seed", "species", "observer", "inat_photo_id", "inat_observation_url"):
        if key in record:
            item[key] = record[key]
    return item


def validate_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    min_records: int = MIN_RECORDS,
    max_records: int = MAX_RECORDS,
) -> dict[str, Any]:
    """Structural validation of a pair dataset; raises ValueError before any model import. Nothing checks
    that `image1` really is `image0` under `homography` — a wrong reference is scored without complaint."""
    if (
        isinstance(records, Mapping)
        or not isinstance(records, Sequence)
        or isinstance(records, (str, bytes))
    ):
        raise ValueError("records must be a list of {id, image0, image1, homography} mappings")
    if not min_records <= len(records) <= max_records:
        raise ValueError(f"{len(records)} records; {min_records}..{max_records} are required")
    checked = []
    ids: set[str] = set()
    tiers: dict[str, int] = {}
    for index, record in enumerate(records):
        item = _check_record(record, index)
        if item["id"] in ids:
            raise ValueError(f"duplicate id {item['id']!r}")
        ids.add(item["id"])
        tiers[item["tier"]] = tiers.get(item["tier"], 0) + 1
        checked.append(item)
    sides = [max(r["image0"].size) for r in checked]
    return {
        "records": checked,
        "n_records": len(checked),
        "tiers": dict(sorted(tiers.items())),
        "image_side": {"min": min(sides), "max": max(sides)},
        "digest": dataset_digest(checked),
        "model_id": MODEL_ID,
    }


def image_digest(image: Image.Image) -> str:
    """SHA-256 of the decoded RGB pixels (size + bytes), so a re-encoded copy of the same image matches."""
    rgb = image.convert("RGB")
    return _sha256_bytes(f"{rgb.size[0]}x{rgb.size[1]}:".encode() + rgb.tobytes())


def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:
    payload = [[r["id"], image_digest(r["image0"]), image_digest(r["image1"]), r["homography"]] for r in records]
    return _sha256_bytes(json.dumps(payload, ensure_ascii=False, separators=(",", ":")).encode("utf-8"))


def check_split_disjoint(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Assert no source image (by decoded-pixel digest of image0) appears in two splits (leakage check)."""
    seen: dict[str, str] = {}
    for name, records in splits.items():
        for record in records:
            key = image_digest(record["image0"])
            if key in seen and seen[key] != name:
                raise ValueError(f"image {record['id']!r} appears in both {seen[key]} and {name}")
            seen[key] = name
    return {name: len(records) for name, records in splits.items()}


def observer_overlap(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, int]:
    """How many observers contributed photos to more than one split (an observation, not an assertion)."""
    seen: dict[str, set[str]] = {}
    for name, records in splits.items():
        for record in records:
            if record.get("observer"):
                seen.setdefault(str(record["observer"]), set()).add(name)
    return {
        "observers": len(seen),
        "in_more_than_one_split": sum(1 for s in seen.values() if len(s) > 1),
    }


def split_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    val_fraction: float = 0.15,
    test_fraction: float = 0.2,
    seed: int = 0,
) -> dict[str, list[dict[str, Any]]]:
    """Seeded shuffle of BYOD image records (`{id, image}`) into train / validation / test after de-duplicating
    images by decoded pixels, then one pair per image."""
    if not (0.0 <= val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):
        raise ValueError("fractions must satisfy 0 <= val < 1, 0 < test < 1, val + test < 1")
    seen: set[str] = set()
    unique = []
    for record in records:
        image = _open(record["image"], str(record.get("id")))
        key = image_digest(image)
        if key not in seen:
            seen.add(key)
            unique.append({**record, "image": image})
    rng = random.Random(seed)
    rng.shuffle(unique)
    n_test = max(1, round(len(unique) * test_fraction))
    n_val = round(len(unique) * val_fraction)
    parts = {"test": unique[:n_test], "validation": unique[n_test : n_test + n_val], "train": unique[n_test + n_val :]}
    if len(parts["train"]) < MIN_RECORDS:
        raise ValueError(f"split leaves {len(parts['train'])} training images; at least {MIN_RECORDS} are required")
    out = {}
    for offset, (name, part) in enumerate(parts.items()):
        relabelled = [{**r, "id": f"{name}-{i:03d}", "source_id": r["id"]} for i, r in enumerate(part)]
        out[name] = make_pairs(relabelled, seed=seed + offset)
    return out


def load_byod_dataset(path: str | Path) -> list[dict[str, Any]]:
    """Read `{id, image}` records from a directory or a zip of image files (optionally listed in `images.csv`
    with columns `id`, `file`); images are decoded, never extracted to disk."""
    source = Path(path)
    if source.is_dir():
        names = sorted(p.name for p in source.iterdir() if p.suffix.lower() in (".jpg", ".jpeg", ".png"))
        loader = lambda name: Image.open(source / name)  # noqa: E731
    elif source.is_file() and source.suffix.lower() == ".zip":
        archive = zipfile.ZipFile(source)
        members = {Path(n).name: n for n in archive.namelist() if Path(n).suffix.lower() in (".jpg", ".jpeg", ".png")}
        names = sorted(members)
        loader = lambda name: Image.open(io.BytesIO(archive.read(members[name])))  # noqa: E731
    else:
        raise ValueError("BYOD datasets must be a directory or a .zip holding JPEG / PNG image files")
    if not names:
        raise ValueError("BYOD dataset holds no JPEG / PNG image files")
    out = []
    for name in names:
        image = loader(name)
        image.load()
        out.append({"id": re.sub(r"[^A-Za-z0-9_.:-]", "_", Path(name).stem)[:64], "image": image.convert("RGB")})
    return out


def write_dataset_csv(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    """Write the pair table of a split (id, source, tier, seed, the homography, provenance)."""
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=["id", "source_id", "tier", "seed", "homography", "observer", "inat_observation_url"])
        writer.writeheader()
        for record in records:
            writer.writerow(
                {
                    "id": record["id"],
                    "source_id": record.get("source_id", ""),
                    "tier": record.get("tier", ""),
                    "seed": record.get("seed", ""),
                    "homography": json.dumps(record["homography"]),
                    "observer": record.get("observer", ""),
                    "inat_observation_url": record.get("inat_observation_url", ""),
                }
            )
    return out

**Module 7/7:** `src/lightglue_pipeline/pipeline.py` (carried verbatim; see the note above)

In [ ]:
"""LightGlue + ALIKED matching pipeline: the inference contract (`extract`, `match`), homography-supervised
evaluation with three baselines, a bounded adaptation of the matcher's last layers with the LightGlue
assignment loss, and a digest-manifested safetensors adapter."""
# ruff: noqa: E501  -- docstrings and record literals kept on single lines at the fleet width

from __future__ import annotations

import hashlib
import json
import math
import time
from collections.abc import Callable, Mapping, Sequence
from io import BytesIO
from pathlib import Path
from typing import Any

import numpy as np
import torch
from PIL import Image

# standalone rewrite (build_notebook.py): `from .config import (` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .metrics import (` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .model import convert_sources, load_components, stage_missing_files, verify_snapshot` removed — names are kernel globals defined by the carried modules

ImageInput = str | Path | bytes | Image.Image

# --------------------------------------------------------------------------
# Adaptation contract (E2E): bounded fine-tuning of the matcher's last transformer layers and its final
# assignment head with the LightGlue assignment loss, supervised by the pairs' exact homographies.
# --------------------------------------------------------------------------
MATCHER_LAYERS = 9  # LightGlue transformer layers (self + cross attention each)
DEFAULT_TRAINABLE_LAYERS = 2  # the last two layers + the final assignment head (2,567,169 params)
POSITIVE_PX = 3.0  # a keypoint pair is a ground-truth match under this symmetric reprojection error
NEGATIVE_PX = 5.0  # a keypoint with no partner under this error is ground-truth unmatched
MAX_EVAL_RECORDS = 5_000
MIN_SCORED_RECORDS = 30  # below this a scored set is labelled a small sample
ARTIFACT_FORMAT = "org.valcorza.lightglue.adapter.v1"
ARTIFACT_FORMAT_VERSION = "1.0"
ARTIFACT_WEIGHTS_NAME = "adapter.safetensors"
ARTIFACT_MANIFEST_NAME = "manifest.json"
PARAMETER_COUNT = MATCHER_PARAMETER_COUNT

INPUT_SCHEMA: dict[str, Any] = {
    "images": (
        "PIL.Image.Image, raw bytes, or a local path decodable by Pillow; any mode, converted to RGB; "
        "remote URLs are refused"
    ),
    "image_size": (
        f"sides in [{MIN_SIDE}, {MAX_SIDE}] px; no resizing or cropping (ALIKED pads to a multiple of 32 "
        "internally and reports keypoints in the input frame)"
    ),
    "keypoints": {"max_per_image": MAX_KEYPOINTS, "detection_threshold": DETECTION_THRESHOLD},
    "thresholds": {"match": FILTER_THRESHOLD},
    "output": (
        "kpts0 / kpts1 (M, 2) float32 pixel coordinates (x, y) of the matched ALIKED keypoints and "
        "confidence (M,) in (0, 1] — the assignment score of each mutual match"
    ),
    "validation": (
        "size and decodability only. Nothing checks that the two images show the same scene: any two "
        "images are matched, and a pair with no overlap still returns whatever passes the threshold"
    ),
}


def _sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _coerce_image(value: ImageInput) -> Image.Image:
    if isinstance(value, Image.Image):
        return value.convert("RGB")
    if isinstance(value, bytes):
        image = Image.open(BytesIO(value))
        image.load()
        return image.convert("RGB")
    if isinstance(value, str | Path):
        text = str(value)
        if text.lower().startswith(("http://", "https://")):
            raise ValueError("remote image URLs are not accepted; pass a local path, bytes or a PIL image")
        path = Path(text)
        if not path.is_file():
            raise ValueError(f"image file not found: {path}")
        image = Image.open(path)
        image.load()
        return image.convert("RGB")
    raise ValueError("image must be a local path, bytes or a PIL.Image.Image")


def _check_size(image: Image.Image, what: str) -> None:
    if min(image.size) < MIN_SIDE or max(image.size) > MAX_SIDE:
        raise ValueError(f"{what}: sides must lie in [{MIN_SIDE}, {MAX_SIDE}] px; got {image.size}")


def _to_tensor(image: Image.Image) -> torch.Tensor:
    """RGB float tensor (1, 3, H, W) in [0, 1] — the extractor's input, at the image's own size."""
    array = np.asarray(image.convert("RGB"), dtype=np.float32) / 255.0
    return torch.from_numpy(array).permute(2, 0, 1)[None].contiguous()


def validate_inputs(
    image0: ImageInput, image1: ImageInput, *, names: Sequence[str] | None = None
) -> dict[str, Any]:
    """Validation stage: exactly the checks `match` applies, reported as an input manifest before the model runs."""
    findings: list[dict[str, Any]] = []
    observations = []
    labels = list(names) if names else ["image0", "image1"]
    for label, value in zip(labels, (image0, image1), strict=True):
        image = _coerce_image(value)
        _check_size(image, label)
        width, height = image.size
        observations.append({"name": label, "size": [width, height], "mode": "RGB after conversion"})
    return {"schema": INPUT_SCHEMA, "images": observations, "findings": findings, "verdict": "accepted"}


class LightGluePipeline:
    def __init__(
        self,
        extractor: Any,
        matcher: Any,
        *,
        device: str | torch.device = "cpu",
        checkpoint_path: Path | str | None = None,
        checkpoint_source: str | None = None,
        manifest_verified: bool = False,
        weight_sha256: str | None = None,
        weight_size_bytes: int | None = None,
        extractor_sha256: str | None = None,
    ) -> None:
        self.extractor = extractor
        self.matcher = matcher
        self.model = matcher  # the adaptable network, under the fleet's attribute name
        self.device = torch.device(device)
        self.checkpoint_path = Path(checkpoint_path) if checkpoint_path is not None else None
        self.checkpoint_source = checkpoint_source
        self.manifest_verified = manifest_verified
        self.weight_sha256 = weight_sha256
        self.weight_size_bytes = weight_size_bytes
        self.extractor_sha256 = extractor_sha256
        self.adapter: dict[str, Any] | None = None
        self.conversion: dict[str, Any] | None = None
        for module in (self.extractor, self.matcher):
            if hasattr(module, "parameters"):
                module.eval()  # ALIKED's BatchNorm statistics must never drift: a train-mode extractor updates them on every forward
                for param in module.parameters():
                    param.requires_grad_(False)

    @classmethod
    def from_pretrained(
        cls,
        *,
        device: str | torch.device | None = None,
        cache_dir: str | Path | None = None,
        weights_path: str | Path | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
        max_keypoints: int = MAX_KEYPOINTS,
        detection_threshold: float = DETECTION_THRESHOLD,
        filter_threshold: float = FILTER_THRESHOLD,
        progress: Callable[[Any], None] | None = None,
    ) -> LightGluePipeline:
        """Load the two pinned checkpoints into the vendored networks.

        ``weights_dir`` names a fleet snapshot directory holding ``dimer-base-manifest.json``: absent sources
        are staged with :func:`stage_missing_files` (only when ``allow_download=True``), the directory is
        verified against the manifest by :func:`verify_snapshot`, the two safetensors files are produced by
        :func:`convert_sources` when absent (static pickle audit, one weights-only unpickle each, strict load,
        deterministic save, pinned-digest check), and :func:`load_components` loads them as an explicit path.
        """
        conversion = None
        if weights_dir is not None:
            if weights_path is not None:
                raise ValueError("pass either weights_dir or weights_path, not both")
            stage_missing_files(weights_dir, allow_download=allow_download)
            snapshot = verify_snapshot(weights_dir)
            if not snapshot.get("converted"):
                conversion = convert_sources(weights_dir)  # audit + one-time weights-only unpickle
                if progress is not None:
                    progress({"conversion": conversion})
                verify_snapshot(weights_dir)
            weights_path = weights_dir
        extractor, matcher, target_device, _, metadata = load_components(
            device=device,
            cache_dir=cache_dir,
            weights_path=weights_path,
            return_metadata=True,
            max_keypoints=max_keypoints,
            detection_threshold=detection_threshold,
            filter_threshold=filter_threshold,
        )
        pipe = cls(
            extractor,
            matcher,
            device=target_device,
            checkpoint_path=metadata.get("checkpoint_path"),
            checkpoint_source=metadata.get("checkpoint_source"),
            manifest_verified=metadata.get("manifest_verified", False),
            weight_sha256=metadata.get("weight_sha256"),
            weight_size_bytes=metadata.get("weight_size_bytes"),
            extractor_sha256=metadata.get("extractor_sha256"),
        )
        pipe.conversion = conversion
        return pipe

    # ------------------------------------------------------------------ inference contract

    def _features(self, image: Image.Image) -> dict[str, torch.Tensor]:
        """ALIKED keypoints, descriptors and scores of one image, as batched tensors on the device."""
        tensor = _to_tensor(image).to(self.device)
        with torch.inference_mode():
            feats = self.extractor({"image": tensor})
        feats = {k: v.clone() for k, v in feats.items()}  # leave inference mode: tensors may feed autograd
        feats["image_size"] = torch.tensor([[tensor.shape[-1], tensor.shape[-2]]], dtype=torch.float32, device=self.device)
        return feats

    def extract(self, image: ImageInput) -> dict[str, Any]:
        """The extractor half of the contract: up to `MAX_KEYPOINTS` ALIKED keypoints of one image with their
        128-d L2-normalised descriptors and detection scores, in the image's own pixel frame."""
        img = _coerce_image(image)
        _check_size(img, "image")
        feats = self._features(img)
        return {
            "keypoints": feats["keypoints"][0].cpu().numpy().astype(np.float32),
            "descriptors": feats["descriptors"][0].cpu().numpy().astype(np.float32),
            "scores": feats["keypoint_scores"][0].cpu().numpy().astype(np.float32),
            "size": [int(img.width), int(img.height)],
        }

    def match(self, image0: ImageInput, image1: ImageInput) -> dict[str, Any]:
        """Sparse matches between two images: `{kpts0, kpts1, confidence, size0, size1, n_keypoints0,
        n_keypoints1}` with keypoints as (M, 2) float32 (x, y) pixel coordinates in each image's own frame."""
        img0, img1 = _coerce_image(image0), _coerce_image(image1)
        _check_size(img0, "image0")
        _check_size(img1, "image1")
        feats0, feats1 = self._features(img0), self._features(img1)
        with torch.inference_mode():
            out = self.matcher({"image0": feats0, "image1": feats1})
        pairs = out["matches"][0]
        kpts0 = feats0["keypoints"][0][pairs[:, 0]].cpu().numpy().astype(np.float32)
        kpts1 = feats1["keypoints"][0][pairs[:, 1]].cpu().numpy().astype(np.float32)
        conf = out["scores"][0].cpu().numpy().astype(np.float32)
        return {
            "kpts0": kpts0.reshape(-1, 2),
            "kpts1": kpts1.reshape(-1, 2),
            "confidence": conf.reshape(-1),
            "size0": [int(img0.width), int(img0.height)],
            "size1": [int(img1.width), int(img1.height)],
            "n_keypoints0": int(feats0["keypoints"].shape[1]),
            "n_keypoints1": int(feats1["keypoints"].shape[1]),
            "layers": int(out["stop"]),
        }

    def descriptor_nn_match(self, image0: ImageInput, image1: ImageInput) -> dict[str, Any]:
        """The third baseline: the same ALIKED keypoints and descriptors, paired by mutual nearest neighbour in
        descriptor space — no LightGlue."""
        img0, img1 = _coerce_image(image0), _coerce_image(image1)
        _check_size(img0, "image0")
        _check_size(img1, "image1")
        feats0, feats1 = self._features(img0), self._features(img1)
        pairs, sims = mutual_nn_matches(feats0["descriptors"][0].cpu().numpy(), feats1["descriptors"][0].cpu().numpy())
        kpts0 = feats0["keypoints"][0].cpu().numpy()[pairs[:, 0]]
        kpts1 = feats1["keypoints"][0].cpu().numpy()[pairs[:, 1]]
        return {
            "kpts0": np.asarray(kpts0, dtype=np.float64).reshape(-1, 2),
            "kpts1": np.asarray(kpts1, dtype=np.float64).reshape(-1, 2),
            "confidence": np.asarray(sims, dtype=np.float64),
            "size0": [int(img0.width), int(img0.height)],
            "baseline": "ALIKED descriptors, mutual nearest neighbour (no learned matcher)",
        }

    # ------------------------------------------------------------------ evaluation

    def evaluate(
        self,
        records: Sequence[Mapping[str, Any]],
        *,
        matcher: Callable[[Image.Image, Image.Image], Mapping[str, Any]] | None = None,
    ) -> dict[str, Any]:
        """Homography-supervised scoring of a validated pair dataset with `metrics.matching_metrics`; `matcher`
        substitutes a baseline for the model (same record structure, same scoring)."""
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]
        started = time.perf_counter()
        rows = []
        for record in checked:
            result = (
                matcher(record["image0"], record["image1"])
                if matcher is not None
                else self.match(record["image0"], record["image1"])
            )
            size = tuple(result.get("size0", record["image0"].size))
            row = pair_metrics(result, np.asarray(record["homography"]), (int(size[0]), int(size[1])))
            row.update({"id": record["id"], "tier": record["tier"]})
            rows.append(row)
        out = matching_metrics(rows)
        tiers = sorted({r["tier"] for r in rows})
        out["by_tier"] = {
            tier: {
                k: v
                for k, v in matching_metrics([r for r in rows if r["tier"] == tier]).items()
                if k != "definitions"
            }
            for tier in tiers
        }
        out.update(
            {
                "per_pair": [{k: v for k, v in r.items() if k != "inlier_errors"} for r in rows],
                "verdict": "measured" if len(checked) >= MIN_SCORED_RECORDS else "measured-small-sample",
                "adapted": self.adapter is not None,
                "matcher": "model" if matcher is None else "baseline",
                "seconds": round(time.perf_counter() - started, 3),
                "model_id": MODEL_ID,
                "model_revision": MODEL_REVISION,
            }
        )
        return out

    def evaluate_baselines(self, records: Sequence[Mapping[str, Any]]) -> dict[str, dict[str, Any]]:
        """The three references scored exactly as the model is: two non-neural (identity guess, patch nearest
        neighbour) and the same keypoints without the learned matcher (descriptor mutual nearest neighbour)."""
        out = {}
        for name, fn in (
            ("identity", identity_baseline),
            ("patch_neighbour", patch_neighbour_baseline),
            ("descriptor_nn", self.descriptor_nn_match),
        ):
            result = self.evaluate(records, matcher=fn)
            result["baseline"] = name
            out[name] = result
        return out

    # ------------------------------------------------------------------ adaptation

    def _assignment_forward(self, feats0: Mapping[str, torch.Tensor], feats1: Mapping[str, torch.Tensor]) -> torch.Tensor:
        """The matcher's forward to the final log-assignment matrix with gradients: keypoint normalisation,
        input projection, positional encoding, all nine transformer layers (no early exit, no pruning) and the
        last assignment head. Returns (1, M + 1, N + 1) log scores with the dustbin row and column."""
        pass  # standalone rewrite (build_notebook.py): `from .modeling import normalize_keypoints` removed — names are kernel globals defined by the carried modules

        m = self.matcher
        kpts0 = normalize_keypoints(feats0["keypoints"], feats0["image_size"]).clone()
        kpts1 = normalize_keypoints(feats1["keypoints"], feats1["image_size"]).clone()
        desc0 = m.input_proj(feats0["descriptors"].detach().contiguous())
        desc1 = m.input_proj(feats1["descriptors"].detach().contiguous())
        encoding0, encoding1 = m.posenc(kpts0), m.posenc(kpts1)
        for layer in m.transformers:
            desc0, desc1 = layer(desc0, desc1, encoding0, encoding1)
        scores, _sim = m.log_assignment[-1](desc0, desc1)
        return scores

    @staticmethod
    def match_ground_truth(
        kpts0: np.ndarray,
        kpts1: np.ndarray,
        homography: np.ndarray,
        *,
        positive_px: float = POSITIVE_PX,
        negative_px: float = NEGATIVE_PX,
    ) -> dict[str, np.ndarray]:
        """Ground-truth assignment of two keypoint sets under a homography, the LightGlue training rule: the
        symmetric reprojection distance (`H · k0` against `k1` and `H⁻¹ · k1` against `k0`, the larger of the
        two), mutual nearest neighbours under `positive_px` are the positive pairs, keypoints with no partner
        under `negative_px` are unmatched, and the rest are ignored by the loss."""
        k0 = np.asarray(kpts0, dtype=np.float64).reshape(-1, 2)
        k1 = np.asarray(kpts1, dtype=np.float64).reshape(-1, 2)
        m, n = len(k0), len(k1)
        if m == 0 or n == 0:
            return {"positives": np.zeros((0, 2), dtype=np.int64), "unmatched0": np.ones(m, dtype=bool), "unmatched1": np.ones(n, dtype=bool)}
        p01 = warp_points(k0, homography)
        p10 = warp_points(k1, np.linalg.inv(np.asarray(homography, dtype=np.float64)))
        d01 = np.linalg.norm(p01[:, None, :] - k1[None, :, :], axis=2)
        d10 = np.linalg.norm(k0[:, None, :] - p10[None, :, :], axis=2)
        dist = np.maximum(np.nan_to_num(d01, nan=np.inf), np.nan_to_num(d10, nan=np.inf))
        nn0 = dist.argmin(axis=1)
        nn1 = dist.argmin(axis=0)
        i = np.arange(m)
        best0 = dist[i, nn0]
        positive = (nn1[nn0] == i) & (best0 < positive_px)
        positives = np.stack([i[positive], nn0[positive]], axis=1).astype(np.int64)
        unmatched0 = dist.min(axis=1) > negative_px
        unmatched1 = dist.min(axis=0) > negative_px
        return {"positives": positives, "unmatched0": unmatched0, "unmatched1": unmatched1}

    @staticmethod
    def assignment_loss(scores: torch.Tensor, ground_truth: Mapping[str, np.ndarray]) -> torch.Tensor:
        """The LightGlue assignment loss on one pair: the negative log-likelihood of the positive pairs in the
        log-assignment matrix, plus half the mean negative log-likelihood of the dustbin entries of the
        unmatched keypoints of each image (`log(1 − σ)`), each term averaged over its own set."""
        s = scores[0]
        zero = s.sum() * 0.0
        pos = torch.as_tensor(ground_truth["positives"], dtype=torch.long, device=s.device)
        un0 = torch.as_tensor(ground_truth["unmatched0"], dtype=torch.bool, device=s.device)
        un1 = torch.as_tensor(ground_truth["unmatched1"], dtype=torch.bool, device=s.device)
        nll_pos = -s[pos[:, 0], pos[:, 1]].mean() if len(pos) else zero
        nll_neg0 = -s[:-1, -1][un0].mean() if bool(un0.any()) else zero
        nll_neg1 = -s[-1, :-1][un1].mean() if bool(un1.any()) else zero
        return nll_pos + 0.5 * (nll_neg0 + nll_neg1)

    def _trainable_names(self, trainable_layers: int) -> list[str]:
        if (
            isinstance(trainable_layers, bool)
            or not isinstance(trainable_layers, int)
            or not 1 <= trainable_layers <= MATCHER_LAYERS
        ):
            raise ValueError(f"trainable_layers must be an int in 1..{MATCHER_LAYERS}")
        first = MATCHER_LAYERS - trainable_layers
        prefixes = tuple(f"transformers.{k}." for k in range(first, MATCHER_LAYERS)) + (
            f"log_assignment.{MATCHER_LAYERS - 1}.",
        )
        return [name for name, _p in self.matcher.named_parameters() if name.startswith(prefixes)]

    def adapt(
        self,
        train: Sequence[Mapping[str, Any]],
        val: Sequence[Mapping[str, Any]] | None = None,
        *,
        epochs: int = 3,
        lr: float = 1e-4,
        batch_size: int = 4,
        trainable_layers: int = DEFAULT_TRAINABLE_LAYERS,
        seed: int = 0,
        progress: Callable[[dict[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Bounded fine-tuning of the matcher on validated pairs.

        Only the last `trainable_layers` transformer layers of LightGlue and its final assignment head
        (`log_assignment.8`: matchability and final projection) train — 2,567,169 of 11,884,625 parameters by
        default; ALIKED, the input projection, the positional encoding, the earlier layers and the token
        confidences stay frozen. ALIKED features of the training pairs are extracted once and cached (the
        extractor is frozen). Each pair runs the matcher to the final log-assignment matrix and is scored with
        the LightGlue assignment loss against the homography's ground-truth assignment; `batch_size` pairs are
        accumulated per AdamW step (pairs have different keypoint counts, so they are not stacked), gradients
        are clipped at 1.0, the order is seeded, no scheduler. Epoch 0 records the frozen model's validation
        metrics; the epoch with the highest validation precision at 3 px is kept (ties broken by homography
        accuracy at 3 px). Transactional: any failure restores the base tensors."""
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        if isinstance(epochs, bool) or not isinstance(epochs, int) or not 1 <= epochs <= 20:
            raise ValueError("epochs must be an int in 1..20")
        if not (0.0 < lr <= 1e-3):
            raise ValueError("lr must be in (0, 1e-3]")
        if isinstance(batch_size, bool) or not isinstance(batch_size, int) or not 1 <= batch_size <= 32:
            raise ValueError("batch_size must be an int in 1..32")
        names = self._trainable_names(trainable_layers)
        train_checked = validate_dataset(train)["records"]
        val_checked = validate_dataset(val, min_records=1, max_records=MAX_EVAL_RECORDS)["records"] if val else []
        model = self.matcher
        torch.manual_seed(seed)
        started = time.perf_counter()
        wanted = set(names)
        for name, param in model.named_parameters():
            param.requires_grad_(name in wanted)
        params = [p for p in model.parameters() if p.requires_grad]
        n_trainable = sum(p.numel() for p in params)
        optimiser = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)
        cached = []
        for record in train_checked:
            feats0, feats1 = self._features(record["image0"]), self._features(record["image1"])
            gt = self.match_ground_truth(feats0["keypoints"][0].cpu().numpy(), feats1["keypoints"][0].cpu().numpy(), np.asarray(record["homography"]))
            cached.append((feats0, feats1, gt))
        n_positive = int(sum(len(gt["positives"]) for _f0, _f1, gt in cached))
        feature_seconds = round(time.perf_counter() - started, 2)

        def score_val() -> dict[str, Any] | None:
            if not val_checked:
                return None
            model.eval()
            result = self.evaluate(val_checked)
            return {k: result[k] for k in ("precision_3px", "homography_acc_3px", "inliers_per_pair", "matches_per_pair", "n")}

        def key(entry: dict[str, Any]) -> tuple[float, float]:
            return (entry["val"]["precision_3px"], entry["val"]["homography_acc_3px"]) if entry["val"] else (-math.inf, -math.inf)

        history: list[dict[str, Any]] = []
        entry: dict[str, Any] = {"epoch": 0, "train_loss": None, "val": score_val(), "note": "frozen model"}
        history.append(entry)
        if progress:
            progress(entry)
        best_key = key(entry)
        best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}
        initial_state = {k: v.clone() for k, v in best_state.items()}
        best_epoch = 0
        generator = torch.Generator().manual_seed(seed)
        try:
            for epoch in range(1, epochs + 1):
                model.train()
                self.extractor.eval()  # the features are cached, but any later _features call must not touch BatchNorm statistics
                order = torch.randperm(len(cached), generator=generator).tolist()
                losses = []
                for start in range(0, len(order), batch_size):
                    optimiser.zero_grad(set_to_none=True)
                    chunk = order[start : start + batch_size]
                    total = 0.0
                    for i in chunk:
                        feats0, feats1, gt = cached[i]
                        scores = self._assignment_forward(feats0, feats1)
                        loss = self.assignment_loss(scores, gt) / len(chunk)
                        loss.backward()
                        total += float(loss.detach())
                    torch.nn.utils.clip_grad_norm_(params, 1.0)
                    optimiser.step()
                    losses.append(total)
                model.eval()
                entry = {"epoch": epoch, "train_loss": sum(losses) / len(losses), "val": score_val()}
                history.append(entry)
                if progress:
                    progress(entry)
                if not entry["val"] or key(entry) > best_key:
                    best_key = key(entry)
                    best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}
                    best_epoch = epoch
        except BaseException:
            restore = dict(model.state_dict())
            restore.update(initial_state)
            model.load_state_dict(restore, strict=True)
            model.eval()
            for param in model.parameters():
                param.requires_grad_(False)
            self.adapter = None
            raise
        merged = dict(model.state_dict())
        merged.update(best_state)
        model.load_state_dict(merged, strict=True)
        model.eval()
        for param in model.parameters():
            param.requires_grad_(False)
        self.adapter = {
            "trainable_layers": trainable_layers,
            "trainable_names": names,
            "n_trainable": n_trainable,
            "n_total": sum(p.numel() for p in model.parameters()),
            "epochs": epochs,
            "best_epoch": best_epoch,
            "selection": "highest validation precision at 3 px (ties: homography accuracy at 3 px)"
            if val_checked
            else "final epoch (no validation split)",
            "lr": lr,
            "batch_size": batch_size,
            "n_train": len(train_checked),
            "n_val": len(val_checked),
            "ground_truth": {"positive_px": POSITIVE_PX, "negative_px": NEGATIVE_PX, "positive_pairs": n_positive},
            "feature_seconds": feature_seconds,
            "seed": seed,
            "history": history,
            "seconds": round(time.perf_counter() - started, 2),
        }
        return dict(self.adapter)

    # ------------------------------------------------------------------ artifacts

    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:
        """Write the adapted matcher tensors as safetensors plus a base manifest."""
        if self.adapter is None:
            raise ValueError("nothing to save: call adapt() first")
        from safetensors.torch import save_file

        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        names = set(self.adapter["trainable_names"])
        tensors = {k: v.detach().cpu().contiguous() for k, v in self.matcher.state_dict().items() if k in names}
        weights_path = out / ARTIFACT_WEIGHTS_NAME
        save_file(tensors, str(weights_path), metadata={"format": "pt"})
        manifest = {
            "format": ARTIFACT_FORMAT,
            "format_version": ARTIFACT_FORMAT_VERSION,
            "base_model": {
                "id": MODEL_ID,
                "revision": MODEL_REVISION,
                "key": DEFAULT_MODEL_KEY,
                "weight_file": MATCHER_FILENAME,
                "weight_sha256": MATCHER_SHA256,
                "extractor_file": EXTRACTOR_FILENAME,
                "extractor_sha256": EXTRACTOR_SHA256,
            },
            "adapter": {k: v for k, v in self.adapter.items() if k not in ("history", "trainable_names")},
            "history": self.adapter["history"],
            "tensors": sorted(tensors),
            "files": [
                {
                    "path": ARTIFACT_WEIGHTS_NAME,
                    "bytes": weights_path.stat().st_size,
                    "sha256": _sha256_file(weights_path),
                }
            ],
            "metadata": dict(metadata or {}),
        }
        (out / ARTIFACT_MANIFEST_NAME).write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
        return out

    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Verify an adapter's manifest, digest and exact tensor set **before** deserialising, then overwrite
        exactly the tensors it carries."""
        root = Path(artifact_dir)
        manifest = json.loads((root / ARTIFACT_MANIFEST_NAME).read_text(encoding="utf-8"))
        if manifest.get("format") != ARTIFACT_FORMAT:
            raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
        if manifest.get("format_version") != ARTIFACT_FORMAT_VERSION:
            raise ValueError(f"artifact format_version {manifest.get('format_version')!r} != {ARTIFACT_FORMAT_VERSION!r}")
        base = manifest.get("base_model") or {}
        if (base.get("id"), base.get("revision"), base.get("weight_sha256")) != (MODEL_ID, MODEL_REVISION, MATCHER_SHA256):
            raise ValueError("artifact was adapted from a different base model, revision or weight file")
        if base.get("weight_file", MATCHER_FILENAME) != MATCHER_FILENAME or base.get("extractor_sha256", EXTRACTOR_SHA256) != EXTRACTOR_SHA256:
            raise ValueError("artifact was adapted from a different base weight file or extractor")
        files = manifest.get("files")
        if not isinstance(files, list) or len(files) != 1 or files[0].get("path") != ARTIFACT_WEIGHTS_NAME:
            raise ValueError(f"artifact manifest must list exactly {ARTIFACT_WEIGHTS_NAME!r}")
        weights_path = (root / ARTIFACT_WEIGHTS_NAME).resolve()
        if weights_path.parent != root.resolve():
            raise ValueError("artifact weight path must resolve inside the artifact directory")
        layers = (manifest.get("adapter") or {}).get("trainable_layers")
        expected = sorted(self._trainable_names(layers))
        if sorted(manifest.get("tensors") or []) != expected:
            raise ValueError("artifact tensor list does not match its recorded configuration")
        entry = files[0]
        if not weights_path.is_file():
            raise FileNotFoundError(f"artifact weights missing: {weights_path}")
        if _sha256_file(weights_path) != entry["sha256"] or weights_path.stat().st_size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: digest or size mismatch; refusing to load")
        from safetensors.torch import load_file

        tensors = load_file(str(weights_path))
        if sorted(tensors) != expected:
            raise ValueError("artifact tensor names differ from its manifest")
        state = self.matcher.state_dict()
        for key, value in tensors.items():
            if key not in state or not key.startswith(("transformers.", "log_assignment.")):
                raise ValueError(f"artifact tensor {key} is not an adaptable matcher tensor of the base")
            if tuple(value.shape) != tuple(state[key].shape):
                raise ValueError(f"artifact tensor {key} has shape {tuple(value.shape)}, base has {tuple(state[key].shape)}")
        merged = dict(state)
        merged.update({k: v.to(state[k].dtype) for k, v in tensors.items()})
        self.matcher.load_state_dict(merged, strict=True)
        self.matcher.eval()
        self.adapter = {**manifest["adapter"], "trainable_names": manifest["tensors"], "history": manifest.get("history", [])}
        return manifest

    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        *,
        device: str | torch.device | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> LightGluePipeline:
        """A fresh pipeline from the pinned base with an adapter overlaid."""
        pipe = cls.from_pretrained(device=device, weights_dir=weights_dir, allow_download=allow_download)
        pipe.load_artifact(artifact_dir)
        return pipe


def load_pipeline(**kwargs: Any) -> LightGluePipeline:
    return LightGluePipeline.from_pretrained(**kwargs)


def evaluation_report(result: Mapping[str, Any], reference: Mapping[str, Any] | None = None, *, sample_kind: str = "synthetic") -> dict[str, Any]:
    """Evaluation stage for the drawn-shape sanity pair: a machine-readable report even when nothing is
    measurable. With a `reference` homography and image size the report carries the pair metrics with the
    verdict `sample-sanity`; without it the verdict is `not-measurable`."""
    base: dict[str, Any] = {
        "task": "sparse image matching (ALIKED keypoints, LightGlue assignment)",
        "score_semantics": (
            "match confidences are LightGlue assignment scores in (0, 1] (mutual matches over the threshold), "
            "not calibrated probabilities that a match is correct; the decision rule is the upstream match "
            "threshold; no geometric verification ships"
        ),
        "sample_kind": sample_kind,
        "n_matches": int(len(np.asarray(result.get("kpts0", [])).reshape(-1, 2))),
    }
    if not reference:
        return {**base, "metrics": [], "verdict": "not-measurable"}
    row = pair_metrics(result, np.asarray(reference["homography"]), tuple(reference["size"]))
    metrics = [
        {"id": "precision_3px", "value": row["precision_3px"], "definition": "fraction of matches under 3 px reprojection error"},
        {"id": "n_inliers", "value": row["n_inliers"], "definition": "matches under 3 px"},
        {"id": "corner_error_px", "value": row["corner_error_px"], "definition": "mean corner displacement of the RANSAC-DLT homography vs the reference"},
    ]
    return {**base, "metrics": metrics, "verdict": "sample-sanity"}


__all__ = [
    "ARTIFACT_FORMAT",
    "DEFAULT_TRAINABLE_LAYERS",
    "INPUT_SCHEMA",
    "LightGluePipeline",
    "MATCHER_LAYERS",
    "MAX_EVAL_RECORDS",
    "MIN_SCORED_RECORDS",
    "NEGATIVE_PX",
    "PARAMETER_COUNT",
    "POSITIVE_PX",
    "evaluation_report",
    "load_pipeline",
    "validate_inputs",
]

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `2`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the LightGlue GitHub release (`cvg/LightGlue`, the matcher) and the ALIKED repository at a pinned commit (`Shiaoming/ALIKED`, the extractor) **at release tag `v0.1_arxiv…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `LightGluePipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_release_snapshot",
  "formatVersion": 1,
  "modelKey": "lightglue-aliked",
  "modelId": "cvg/LightGlue",
  "revision": "v0.1_arxiv",
  "revisionKind": "github-release-tag",
  "upstreamCodeRevision": "eb42fee2d71449efb0aa5c10549752b5d75384d8",
  "license": "Apache-2.0 (LightGlue) / BSD-3-Clause (ALIKED)",
  "files": [
    {
      "path": "aliked_lightglue.pth",
      "bytes": 47632827,
      "sha256": "d975e965b105311a6143194852297dff4f02aea5cc2e10cecfed966ca0e22503",
      "sourceUrl": "https://github.com/cvg/LightGlue/releases/download/v0.1_arxiv/aliked_lightglue.pth",
      "pickleAuditSha256": "e7b998d087a5dcadd37713daf30b63cc571160c3180ebc138500ab662197e932",
      "convertsTo": {
        "path": "aliked_lightglue.safetensors",
        "bytes": 47564948,
        "sha256": "9c630a386c74c534428370ce46253e1d0968655db180f97074cb6ad797bd2bc6"
      }
    },
    {
      "path": "aliked-n16.pth",
      "bytes": 2738091,
      "sha256": "5be8704840ed662d9d8c561bf7279c222092674e7eb05fd0feab94899e9d82f2",
      "sourceUrl": "https://raw.githubusercontent.com/Shiaoming/ALIKED/683d7c65197395c0b3f01ebe76e1084a27e73a65/models/aliked-n16.pth",
      "sourceRepository": "Shiaoming/ALIKED",
      "sourceCommit": "683d7c65197395c0b3f01ebe76e1084a27e73a65",
      "pickleAuditSha256": "5b9f0ba08490293d6c17b9cef219991e1a6edda31609429679f8dca1af5a7b10",
      "convertsTo": {
        "path": "aliked-n16.safetensors",
        "bytes": 2719928,
        "sha256": "3c8ca40c0c985cd4d641e96e4b408b14d067b5b3521ac17b36590447d49d115a"
      }
    }
  ],
  "totalBytes": 50370918
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = LightGluePipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. iNaturalist photographs, homography pairs and split

`fetch_corpus` returns the 360 pinned photographs from the cache under `weights/inat-birds/` or the iNaturalist open-data bucket — every cached file is re-hashed and every fetched file refused on any byte-size or SHA-256 mismatch — and `read_corpus` decodes them into image records with their observation page, observer and species. `build_sample_dataset` draws a seeded stratified split of photographs per species (36 / 8 / 16 → 216 / 48 / 96) and turns each photograph into one pair: the photograph at 640 px on the long side (`image0`) and a copy warped by a seeded homography with seeded photometric changes (`image1`), tiers alternating `easy` / `hard`, the reference `H` recorded in the record. `validate_dataset` then checks every record against the contract, `check_split_disjoint` asserts no photograph (by decoded-pixel digest) is shared, `observer_overlap` reports how many observers contributed to more than one split (an observation about the draw, not an assertion), and the training split's pair table is written to `outputs/lightglue_matching_train.csv`.

Look for: 360 photographs, three splits with both tiers, three digests, one pair shown with its reference corners, the tier parameters (the hard tier's `rotation` of 150°), and four refusal probes — a duplicate id, an image over the side ceiling, a singular homography and a dataset too small to split — each rejected before the model does anything.

In [ ]:
import hashlib
import json
import time

import numpy as np
from PIL import Image

USE_BYOD = False  # @param {type:"boolean"}
SPLIT_SEED = 42  # @param {type:"integer"}

os.makedirs('outputs', exist_ok=True)
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    file_name, payload = next(iter(uploaded.items()))
    byod_zip = Path('work') / 'byod.zip'
    byod_zip.parent.mkdir(parents=True, exist_ok=True)
    byod_zip.write_bytes(payload)
    records = load_byod_dataset(byod_zip)
    splits = split_dataset(records, seed=SPLIT_SEED)
    data_source = 'BYOD (' + file_name + ')'
    raw_rows = {'byod_images': len(records)}
else:
    corpus_files = fetch_corpus(cache_dir='weights/inat-birds')
    corpus = read_corpus(corpus_files)
    splits = build_sample_dataset(corpus, seed=SPLIT_SEED)
    data_source = f'{CORPUS_NAME}: {CORPUS_RELEASE} ({CORPUS_LICENSE}); one seeded homography pair per photograph'
    raw_rows = {'photographs': len(corpus), 'bytes': sum(len(v) for v in corpus_files.values()), 'observers': len({r['observer'] for r in corpus})}
dataset_manifests = {name: validate_dataset(part) for name, part in splits.items()}
splits = {name: manifest['records'] for name, manifest in dataset_manifests.items()}
disjoint = check_split_disjoint(splits)
train_records, val_records, test_records = splits['train'], splits['validation'], splits['test']
write_dataset_csv(train_records, 'outputs/lightglue_matching_train.csv')
print({'data_source': data_source, 'raw_rows': raw_rows, 'splits': disjoint, 'observer_overlap': observer_overlap(splits), 'tiers': {k: v for k, v in TIER_PARAMS.items()}})
for name, manifest in dataset_manifests.items():
    print({name: {'n': manifest['n_records'], 'tiers': manifest['tiers'], 'image_side': manifest['image_side'], 'digest': manifest['digest'][:16] + '...'}})
example = train_records[0]
corners = np.array([[0.0, 0.0], [example['image0'].width - 1.0, 0.0], [example['image0'].width - 1.0, example['image0'].height - 1.0], [0.0, example['image0'].height - 1.0]])
print({'example': {'id': example['id'], 'tier': example['tier'], 'size': list(example['image0'].size), 'corners_under_H': np.round(warp_points(corners, np.asarray(example['homography'])), 1).tolist(), 'observation': example.get('inat_observation_url')}})
probes = {
    'duplicate id': [{**r, 'id': 'same'} for r in train_records[:8]],
    'image over the side ceiling': [{**train_records[0], 'image0': Image.new('RGB', (MAX_IMAGE_SIDE + 1, 8))}, *train_records[1:8]],
    'singular homography': [{**train_records[0], 'homography': [[1, 0, 0], [1, 0, 0], [0, 0, 1]]}, *train_records[1:8]],
    'too small': train_records[:3],
}
for name, probe in probes.items():
    try:
        validate_dataset(probe)
        print({'probe': name, 'verdict': 'accepted'})
    except (TypeError, ValueError) as exc:
        print({'probe': name, 'rejected': str(exc)[:110]})

## 5. Match through the inference contract

The inference contract is exercised on a drawn pair: a 256 × 192 synthetic scene — a red square, a green circle and a blue triangle on a light background with a faint grid, rendered in code exactly as the repository's `examples/sample-data/generate_samples.py` renders it and digest-asserted against `SHA256SUMS` — and its copy under a fixed reference homography, a different image family from the photographs and a pair the matcher will be asked to match again after adaptation. `validate_inputs` applies exactly the checks `match` applies and returns an input manifest; a remote URL is validated too and its rejection recorded as a finding. `match` returns `kpts0`, `kpts1` and one confidence per correspondence — ALIKED keypoints of image0 and their mutual best partners in image1 above the match threshold, in keypoint-score order; the per-pair `evaluation_report` on a drawing is `sample-sanity` — plumbing evidence, not a measurement; whether the matcher is *right* is what Section 6 measures on 96 photograph pairs. A drawing of flat shapes has few corners for ALIKED to detect, so expect tens of matches, not hundreds.

In [ ]:
SAMPLE_DIGESTS = {  # examples/sample-data/SHA256SUMS
    'shapes_scene.ppm': '3420b1d3755a5bf9bc90803fa50f3fc7bbeca5720bd6d75748ad79ac00908239',
}
SHAPES_H = np.array([[0.96, 0.05, 12.0], [-0.04, 1.03, -7.0], [2e-5, -1e-5, 1.0]])
WIDTH, HEIGHT, BACKGROUND = 256, 192, (245, 245, 245)


def scene_pixel(x, y):
    if 40 <= x < 104 and 48 <= y < 112:
        return (220, 40, 40)
    if (x - 168) ** 2 + (y - 80) ** 2 <= 34 ** 2:
        return (40, 170, 75)
    if 128 <= y < 176 and abs(x - 120) <= (y - 128) // 2:
        return (40, 90, 220)
    if x % 32 == 0 or y % 32 == 0:
        return (200, 200, 200)
    return BACKGROUND


def render_scene():
    # The repository's generate_samples.py rendering: ASCII P3, 24 values per line.
    lines = ['P3', f'{WIDTH} {HEIGHT}', '255']
    for y in range(HEIGHT):
        row = []
        for x in range(WIDTH):
            row.extend(str(v) for v in scene_pixel(x, y))
        for start in range(0, len(row), 24):
            lines.append(' '.join(row[start : start + 24]))
    return '\n'.join(lines) + '\n'


Path('outputs/sample-data').mkdir(parents=True, exist_ok=True)
scene_path = Path('outputs/sample-data/shapes_scene.ppm')
scene_path.write_bytes(render_scene().encode('ascii'))
scene_digest = hashlib.sha256(scene_path.read_bytes()).hexdigest()
if scene_digest != SAMPLE_DIGESTS['shapes_scene.ppm']:
    raise ValueError(f'Synthetic sample digest mismatch: {scene_digest} != {SAMPLE_DIGESTS["shapes_scene.ppm"]}')
scene0 = Image.open(scene_path).convert('RGB')
scene1 = warp_image(scene0, SHAPES_H)
scene1_path = Path('outputs/sample-data/shapes_scene_warped.png')
scene1.save(scene1_path)
print({'ceilings': {'MIN_SIDE': MIN_SIDE, 'MAX_SIDE': MAX_SIDE, 'DIVISIBLE_BY': DIVISIBLE_BY, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MIN_RECORDS': MIN_RECORDS, 'MAX_RECORDS': MAX_RECORDS, 'keypoints': INPUT_SCHEMA['keypoints'], 'thresholds': INPUT_SCHEMA['thresholds'], 'device': str(pipe.device)}})
input_manifest = validate_inputs(scene_path, scene1_path, names=[scene_path.name, scene1_path.name])
try:
    validate_inputs('https://example.invalid/not-allowed.png', scene1_path)
except ValueError as exc:
    input_manifest['findings'].append({'input': 'remote-url-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/lightglue_matching_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print({'scene_sha256': scene_digest[:16] + '...', 'manifest_verdict': input_manifest['verdict'], 'findings': len(input_manifest['findings'])})
t0 = time.perf_counter()
scene_match = pipe.match(scene_path, scene1_path)
scene_errors = reprojection_errors(scene_match['kpts0'], scene_match['kpts1'], SHAPES_H)
checks = {
    'same_length': len(scene_match['kpts0']) == len(scene_match['kpts1']) == len(scene_match['confidence']),
    'coordinates_inside_frames': bool(np.all(scene_match['kpts0'] >= 0) and np.all(scene_match['kpts0'][:, 0] < scene_match['size0'][0]) and np.all(scene_match['kpts0'][:, 1] < scene_match['size0'][1])),
    'confidences_in_unit_interval': bool(np.all((scene_match['confidence'] > 0) & (scene_match['confidence'] <= 1))),
    'some_matches': len(scene_match['kpts0']) > 0,
}
if not all(checks.values()):
    raise RuntimeError(f'inference output failed a sanity check: {checks}')
frozen_scene = evaluation_report(scene_match, {'homography': SHAPES_H, 'size': scene_match['size0']}, sample_kind='synthetic')
scene_rows = {'frozen': {'n_matches': int(len(scene_match['kpts0'])), 'precision_3px': float((scene_errors < 3).mean()) if len(scene_errors) else 0.0, 'median_error_px': float(np.median(scene_errors)) if len(scene_errors) else None}}
print({'checks': checks, 'seconds': round(time.perf_counter() - t0, 2), 'frozen_scene': {m['id']: round(m['value'], 3) for m in frozen_scene['metrics']}, 'verdict': frozen_scene['verdict'], 'first_matches': np.round(np.concatenate([scene_match['kpts0'][:3], scene_match['kpts1'][:3]], axis=1), 1).tolist()})

## 6. Baselines and the frozen matcher on the test pairs

Four systems frame the adaptation, each read the same way. The **identity guess** answers that every grid point of image0 is at the same coordinates in image1 — right only where the warp is tiny. The **patch nearest neighbour** answers with the best normalised-cross-correlation 15 × 15 patch within ±48 px — a matcher that knows the images through raw intensities and nothing else. The **descriptor nearest neighbour** takes the same ALIKED keypoints and descriptors the model sees and matches them by mutual nearest neighbour in descriptor space — what the pipeline is without its learned matcher, so the gap to it is what LightGlue's nine layers buy. The **frozen model** is scored by `pipe.evaluate`: every returned match against the reference `H`, **precision at 3 px** (also 1 px and 5 px), **matches and inliers per pair**, the **median error** of the inliers, and **homography accuracy at 3 px / 5 px** — the fraction of pairs whose RANSAC-DLT homography from the matches moves the image corners by less than the threshold against the reference, the HPatches-style reading. All four are scored per tier as well. Expect the frozen matcher far above the non-neural baselines on the easy tier and read where it loses: the build record measured precision at 3 px of @P:FROZEN_P3@ (@P:FROZEN_EASY_P3@ easy / @P:FROZEN_HARD_P3@ hard) with @P:FROZEN_MPP@ matches per pair and homography accuracy @P:FROZEN_HACC@, against @P:DNN_P3@ precision for the descriptor neighbour, @P:PNN_P3@ for the patch neighbour and @P:ID_P3@ for the identity guess — the rotated tier is where the matcher, and its descriptors even more, come apart.

In [ ]:
METRICS = ('precision_3px', 'precision_1px', 'matches_per_pair', 'inliers_per_pair', 'median_error_px', 'homography_acc_3px', 'homography_acc_5px')


def short(result):
    return {k: (round(result[k], 3) if isinstance(result[k], float) and np.isfinite(result[k]) else result[k]) for k in METRICS}


t0 = time.perf_counter()
baselines = pipe.evaluate_baselines(test_records)
print({'baseline_seconds': round(time.perf_counter() - t0, 1), 'per_baseline_seconds': {name: result['seconds'] for name, result in baselines.items()}})
for name, result in baselines.items():
    print({name: short(result), 'by_tier': {tier: short(v) for tier, v in result['by_tier'].items()}})
t0 = time.perf_counter()
frozen_test = pipe.evaluate(test_records)
print({'frozen_model_test': short(frozen_test), 'n': frozen_test['n'], 'verdict': frozen_test['verdict'], 'seconds': round(time.perf_counter() - t0, 1)})
print({'by_tier_frozen': {tier: short(v) for tier, v in frozen_test['by_tier'].items()}})
print({'definitions': frozen_test['definitions']})
worst = sorted(frozen_test['per_pair'], key=lambda r: r['precision_3px'])[:3]
print({'weakest_pairs_frozen': [{k: r[k] for k in ('id', 'tier', 'n_matches', 'precision_3px', 'corner_error_px')} for r in worst]})
assert frozen_test['precision_3px'] > baselines['patch_neighbour']['precision_3px'] and frozen_test['homography_acc_3px'] > baselines['identity']['homography_acc_3px']

## 7. Bounded fine-tuning of the matcher's last layers

`pipe.adapt` trains only the last `TRAINABLE_LAYERS` of LightGlue's nine transformer layers (each a self-attention and a cross-attention block) and the final assignment head (matchability and the final projection) — 2,567,169 of 11,884,625 parameters by default; ALIKED, the input projection, the positional encoding, the earlier layers and the per-layer token confidences stay as they are. The ALIKED features of the training pairs are extracted once and cached — the extractor is frozen, so nothing upstream of the matcher changes — and each pair runs the matcher to its final log-assignment matrix, scored with the **LightGlue assignment loss** (the negative log-likelihood of the ground-truth assignment: keypoint pairs under 3 px reprojection error are positives, keypoints with no partner within 5 px are unmatched, the rest ignored — the upstream training rule). AdamW at a fixed learning rate, `BATCH_SIZE` pairs accumulated per step (pairs are not stacked: keypoint counts differ), gradient clipping at 1.0, seeded order, no scheduler. Epoch 0 records the frozen matcher's validation metrics; every epoch is scored on the 48 validation pairs, and the epoch with the highest validation **precision at 3 px** is kept (ties broken by homography accuracy).

Watch the training loss (@P:LOSS_NOTE@) and the validation precision: the easy tier is already fitted, so whatever moves comes from the rotated pairs. The build record kept epoch @P:BEST_EPOCH@ of @P:EPOCHS@ (validation precision at 3 px @P:VAL_P3_0@ frozen → @P:VAL_P3_BEST@); the default is the configuration that gained on the held-out split, and a run that keeps epoch 0 is a valid outcome, not a failure.

In [ ]:
EPOCHS = 3  # @param {type:"integer"}
LEARNING_RATE = 1e-4  # @param {type:"number"}
BATCH_SIZE = 4  # @param {type:"integer"}
TRAINABLE_LAYERS = 2  # @param {type:"integer"}


def report(entry):
    row = {'epoch': entry['epoch'], 'train_loss': None if entry['train_loss'] is None else round(entry['train_loss'], 4)}
    if entry.get('val'):
        row.update({'val_' + k: (round(v, 3) if isinstance(v, float) else v) for k, v in entry['val'].items()})
    if 'note' in entry:
        row['note'] = entry['note']
    print(row)


t0 = time.perf_counter()
adapt_result = pipe.adapt(train_records, val_records, epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE, trainable_layers=TRAINABLE_LAYERS, progress=report)
adapt_seconds = round(time.perf_counter() - t0, 1)
print({'trainable_parameters': adapt_result['n_trainable'], 'total_parameters': adapt_result['n_total'], 'best_epoch': adapt_result['best_epoch'], 'selection': adapt_result['selection'], 'ground_truth': adapt_result['ground_truth'], 'feature_seconds': adapt_result['feature_seconds'], 'seconds': adapt_seconds})

## 8. Held-out evaluation

The test pairs were never used for training or epoch selection, and no photograph appears in two splits. The adapted matcher is scored exactly as the frozen one was in Section 6, the four systems are put side by side on every reading, and the per-tier breakdown is repeated. Read it in this order: **precision at 3 px** first (the metric the epoch was selected on — the build record measured @P:FROZEN_P3@ → @P:ADAPTED_P3@), then the inlier count and the homography accuracy (@P:FROZEN_HACC@ → @P:ADAPTED_HACC@), then the tiers, where @P:TIER_NOTE@. The cell asserts only that the adapted matcher is not worse than the frozen one on precision at 3 px by more than a rounding margin — a bounded adaptation of a fitted matcher may land flat, and the notebook says so rather than asserting a gain. Ninety-six pairs from one seeded split give **no dispersion estimate**; the deltas are sample-sanity evidence that the adaptation contract works, not a benchmark, and a result on rotated bird photographs says nothing about your scenes until you measure them.

In [ ]:
adapted_test = pipe.evaluate(test_records)
adapted_val = pipe.evaluate(val_records)
comparison = {metric: {'identity': round(baselines['identity'][metric], 3), 'patch_neighbour': round(baselines['patch_neighbour'][metric], 3), 'descriptor_nn': round(baselines['descriptor_nn'][metric], 3), 'frozen': round(frozen_test[metric], 3), 'adapted': round(adapted_test[metric], 3)} for metric in METRICS if np.isfinite(frozen_test[metric]) and np.isfinite(adapted_test[metric])}
comparison['delta_vs_frozen'] = {metric: round(adapted_test[metric] - frozen_test[metric], 3) for metric in METRICS if np.isfinite(frozen_test[metric]) and np.isfinite(adapted_test[metric])}
comparison['by_tier'] = {tier: {'descriptor_nn': short(baselines['descriptor_nn']['by_tier'][tier]), 'frozen': short(frozen_test['by_tier'][tier]), 'adapted': short(adapted_test['by_tier'][tier])} for tier in adapted_test['by_tier']}
for key, row in comparison.items():
    print({key: row})
evaluation_report_payload = {
    'model': {'id': MODEL_ID, 'revision': MODEL_REVISION, 'key': DEFAULT_MODEL_KEY},
    'data_source': data_source,
    'dataset_digests': {name: manifest['digest'] for name, manifest in dataset_manifests.items()},
    'splits': disjoint,
    'tiers': TIER_PARAMS,
    'baselines': {name: {k: v for k, v in result.items() if k != 'per_pair'} for name, result in baselines.items()},
    'frozen_test': {k: v for k, v in frozen_test.items() if k != 'per_pair'},
    'frozen_test_per_pair': frozen_test['per_pair'],
    'validation_metrics': {k: v for k, v in adapted_val.items() if k != 'per_pair'},
    'test_metrics': {k: v for k, v in adapted_test.items() if k != 'per_pair'},
    'test_metrics_per_pair': adapted_test['per_pair'],
    'comparison': comparison,
    'adaptation': {k: v for k, v in adapt_result.items() if k not in ('history', 'trainable_names')},
    'history': adapt_result['history'],
    'adaptation_seconds': adapt_seconds,
}
with open('outputs/lightglue_matching_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_report_payload, f, indent=2, ensure_ascii=False)
assert adapted_test['precision_3px'] >= frozen_test['precision_3px'] - 0.01
print({'report': 'outputs/lightglue_matching_evaluation_report.json'})

## 9. Re-match the drawn pair, export the adapter and reload it

The drawn pair from Section 5 is matched again by the adapted model — a drawing, a different image family from the photographs it was tuned on, so this is a small look at what the adaptation did *outside* its corpus (the build record's numbers are in `docs/release-verification.md`; a changed count or precision here is a finding to record, not a failure) — and reported with the per-pair `evaluation_report` (`sample-sanity`). Both match sets are written as JSON.

`pipe.save_artifact` writes the trained tensors — the matcher's last layers and the assignment head, about @P:ADAPTER_MB@ MB — as `adapter.safetensors`, with a `manifest.json` recording the artifact format, the base model id and revision, the digests of the base `aliked_lightglue.safetensors` and `aliked-n16.safetensors`, the tensor names, the file size and SHA-256, the training configuration and the epoch history (OUT8). `LightGluePipeline.from_artifact` re-verifies the base snapshot, checks the artifact manifest, its digest and its exact tensor set **before** deserialising, refuses any tensor outside the matcher, and overlays the tensors onto a freshly loaded base — a new object from files, not the in-memory model (VER2). The cell asserts identical matches on four test pairs (VER4).

In [ ]:
import shutil

adapted_scene_match = pipe.match(scene_path, scene1_path)
adapted_scene_errors = reprojection_errors(adapted_scene_match['kpts0'], adapted_scene_match['kpts1'], SHAPES_H)
adapted_scene = evaluation_report(adapted_scene_match, {'homography': SHAPES_H, 'size': adapted_scene_match['size0']}, sample_kind='synthetic')
scene_rows['adapted'] = {'n_matches': int(len(adapted_scene_match['kpts0'])), 'precision_3px': float((adapted_scene_errors < 3).mean()) if len(adapted_scene_errors) else 0.0, 'median_error_px': float(np.median(adapted_scene_errors)) if len(adapted_scene_errors) else None}
print({'scene': scene_rows, 'scene_after_adaptation': {m['id']: round(m['value'], 3) for m in adapted_scene['metrics']}, 'verdict': adapted_scene['verdict']})
with open('outputs/lightglue_matching_shapes.json', 'w', encoding='utf-8') as handle:
    json.dump({'reference_homography': SHAPES_H.tolist(), 'frozen': {'kpts0': scene_match['kpts0'].tolist(), 'kpts1': scene_match['kpts1'].tolist(), 'confidence': scene_match['confidence'].tolist()}, 'adapted': {'kpts0': adapted_scene_match['kpts0'].tolist(), 'kpts1': adapted_scene_match['kpts1'].tolist(), 'confidence': adapted_scene_match['confidence'].tolist()}, 'rows': scene_rows}, handle, indent=2)
artifact_dir = Path('outputs/lightglue_matching_adapter')
shutil.rmtree(artifact_dir, ignore_errors=True)
pipe.save_artifact(artifact_dir, metadata={'tutorial': 'lightglue_matching', 'data_source': data_source})
artifact_manifest = json.loads((artifact_dir / 'manifest.json').read_text(encoding='utf-8'))
print({'artifact': str(artifact_dir), 'format': artifact_manifest['format'], 'tensors': len(artifact_manifest['tensors']), 'bytes': artifact_manifest['files'][0]['bytes'], 'sha256': artifact_manifest['files'][0]['sha256'][:16] + '...', 'best_epoch': artifact_manifest['adapter']['best_epoch']})
reloaded = LightGluePipeline.from_artifact(artifact_dir, weights_dir=WEIGHTS_DIR, device=pipe.device)
identical = 0
for record in test_records[:4]:
    before, after = pipe.match(record['image0'], record['image1']), reloaded.match(record['image0'], record['image1'])
    identical += int(len(before['kpts0']) == len(after['kpts0']) and np.allclose(before['kpts1'], after['kpts1'], atol=1e-4))
parity = {'identical_pairs': identical, 'of': 4}
print({'reload_parity': parity, 'reloaded_best_epoch': reloaded.adapter['best_epoch']})
assert parity['identical_pairs'] == parity['of']

write_provenance('outputs/provenance.json', pipeline=pipe)
result_payload = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'snapshot': {'path': str(WEIGHTS_DIR), 'files': snapshot['files'], 'fetched_this_run': fetched, 'weight_file': MODEL_FILENAME, 'weight_format': 'safetensors, converted once from the audited source pickle, digest-verified', 'weight_sha256': MODEL_SHA256, 'extractor_file': EXTRACTOR_FILENAME, 'extractor_sha256': EXTRACTOR_SHA256, 'vendored_code': {'lightglue': {'repository': UPSTREAM_REPOSITORY, 'commit': UPSTREAM_COMMIT}, 'aliked': {'repository': ALIKED_REPOSITORY, 'commit': EXTRACTOR_COMMIT}}},
    'data_source': data_source,
    'corpus': {'name': CORPUS_NAME, 'release': CORPUS_RELEASE, 'license': CORPUS_LICENSE, 'base_url': CORPUS_BASE_URL, 'bytes': CORPUS_BYTES, 'pinned_photographs': len(SAMPLE_RECORDS), 'tiers': TIER_PARAMS},
    'inference_contract': {'input_manifest': input_manifest, 'sanity_checks': checks, 'scene': {'sha256': scene_digest, 'reference_homography': SHAPES_H.tolist()}, 'frozen_report': frozen_scene, 'adapted_report': adapted_scene, 'scene_rows': scene_rows},
    'comparison': comparison,
    'artifact': {'dir': str(artifact_dir), 'sha256': artifact_manifest['files'][0]['sha256'], 'bytes': artifact_manifest['files'][0]['bytes'], 'tensors': len(artifact_manifest['tensors'])},
    'reload_parity': parity,
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'numpy': numpy.__version__, 'device': str(pipe.device), 'dtype': 'float32', 'checkpoint_source': pipe.checkpoint_source},
}
with open('outputs/lightglue_matching_result.json', 'w', encoding='utf-8') as handle:
    json.dump(result_payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The frozen matcher is a strong correspondence engine on the easy tier — precision at 3 px of @P:FROZEN_EASY_P3@ with @P:FROZEN_MPP@ matches per pair over the whole split — and comes apart on the rotated tier (@P:FROZEN_HARD_P3@), where its own descriptors matched by nearest neighbour fall further (@P:DNN_HARD_P3@). A bounded fine-tuning of the matcher's last two layers and assignment head on 216 pairs @P:GAIN_SENTENCE@ That is the claim: the adaptation contract works end to end on a labelled pair set with exact references, and the numbers it produces are read on precision, inlier count and homography accuracy, per tier, against three baselines and the frozen model rather than in isolation.

The test split is 96 pairs from one seeded draw of one sample, the validation split that picks the epoch is 48, the warps are synthetic (a photograph and its own perspective-warped, re-lit copy — no viewpoint change of a real scene, no occlusion), the metrics are reference-based scores (own numpy implementations; none a human judgement), and the build record's own epoch history shows the estimate's fragility: @P:FRAGILITY@. So a gain here says the contract works; it does not say the adapted matcher is better on your images, that its confidences are calibrated, or that a match with a high confidence is right — it still returns matches for every pair, and it can be wrong confidently. Fine-tuning on a narrow set can also erode the model elsewhere; the drawn pair re-matched in Section 9 is one image of evidence about that, not a measurement.

Three things to carry to real data. **Baselines first:** the identity guess, the patch neighbour, the descriptor neighbour and the frozen matcher's score on *your* pairs are the numbers to read before any adapted one, per tier and on the homography reading. **Leakage:** keep every photograph in one split (the contract de-duplicates by decoded pixels) and split by scene, session or photographer when your images come from few sources — the sample's observer overlap is printed for exactly that reason. **References:** a synthetic warp gives an exact `H`; real pairs need depth, pose or a fitted homography before they can be scored, and a fitted homography is itself an estimate.

Successful execution proves that the recorded repository revision's package, carried in this standalone notebook, can acquire, digest-verify, audit and convert the pinned checkpoint pickles into its vendored network, fetch and digest-verify a real photograph set and build exact-reference pairs from it, validate the demonstrated dataset contract without leakage, execute the inference contract and a bounded fine-tuning, evaluate against three baselines and the frozen model on an image-disjoint split, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, matching accuracy on real viewpoint changes, rotation invariance in general, other extractors or other cameras, calibration, or production fitness.

**Optional experiments (they do not affect the default path):** set `TRAINABLE_LAYERS = 4` and compare the artifact size and the held-out precision; raise `EPOCHS` and watch the validation precision pick the epoch while the training loss keeps falling; change `LEARNING_RATE` to `1e-5` and read a smaller, steadier change; or bring your own photographs through BYOD and read the three baselines before the adapted number.

## References

- Repository README: https://github.com/kurtvalcorza/lightglue-matching-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/lightglue-matching-pipeline/blob/main/MODEL_CARD.md
- Sample dataset card (synthetic scene): https://github.com/kurtvalcorza/lightglue-matching-pipeline/blob/main/examples/sample-data/DATASET_CARD.md
- Upstream matcher weights and code: https://github.com/cvg/LightGlue (release `v0.1_arxiv`; the matcher and the ALIKED port vendored as `modeling.py` at the commit recorded in `docs/WEIGHTS.md`)
- Upstream extractor weights: https://github.com/Shiaoming/ALIKED (`models/aliked-n16.pth` at the pinned commit)
- LightGlue: Local Feature Matching at Light Speed (Lindenberger, Sarlin and Pollefeys, ICCV 2023): https://arxiv.org/abs/2306.13643
- ALIKED: A Lighter Keypoint and Descriptor Extraction Network via Deformable Transformation (Zhao et al., IEEE TIM 2023): https://arxiv.org/abs/2304.03608
- iNaturalist open data (CC0 photographs, each observer's own licence): https://www.inaturalist.org/pages/developers — bucket https://inaturalist-open-data.s3.amazonaws.com/
- DIMER Notebook Specification 2.0 and Model Card Specification 1.1 (fleet specs in the ml-worker repository)